#Trabalho T1
Grupo:
- Brenda
- Enzo Claudino
- Thainá
- Natália

Segue o notebook =)

Esta etapa inicial são só os imports chave e o carregamento dos arquivos fundamentais.

In [ ]:
#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Imports chave e montar o drive como local de arquivo
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

!pip install FPDF
from google.colab import drive
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from sklearn.model_selection import LeaveOneOut, cross_val_score
import seaborn as sns
from IPython.display import display
from fpdf import FPDF
from scipy import stats
import statsmodels.formula.api as smf
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Buscar arquivo de dados no drive
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

# Monta o Google Drive
drive.mount('/content/drive', force_remount=True)
# Caminho do Drive
caminho_drive = "/content/drive/MyDrive/analytics III/T1/franquia_lojas.csv"
df=pd.read_csv(caminho_drive)
#Caso você esteja rodando localmetne para facilitar o trabalho basta comentar o código anterior e descomentar o seguinte
#caminho_deive = "/content/franquia_lojas.csv"


#+++++++++++++++++++++++++++++++++++++++++++++++++++++
# Teste de sanidade do DF
#+++++++++++++++++++++++++++++++++++++++++++++++++++++

print(df.shape)
print(df.head)

#Parte 1: Limpeza dos dados e conhecimento da base:

#Limpeza de dados:

Para esta etapa desenvolvemos (no espírito caótico movido pela falta de concerta) o limpador de dados radical, uma função que trata dos dados em algumas etapas para que ele seja útil para o trabalho. A sequência que segue explica o processo de tratamento de dados aplicado, para auditoria convido a ver a função no ipynb.

1.	Remover duplicatas;
2.	Definir as colunas para avaliação de outliers e remover outliers pelo método de IQR;
3.	Remover as linhas com dados faltantes;
4.	Transformar as variáveis categóricas em dummies e garantir que os dados sejam úteis para modelagem;

A utilidade de cada uma das variáveis ainda não foi medida nesta etapa do projeto.

In [ ]:
# Limpeza de dados aprimorada

def limpador_de_dados_radical(df):
    print(f'Shape inicial: {df.shape}')

    # 1. Remover duplicatas
    df = df.drop_duplicates()

    # 2. Definir colunas numéricas REAIS para outliers (antes de criar dummies)
    # Lista explícita para evitar processar dummies de cidades/regiões como outliers
    cols_para_outliers = [
        'area_m2', 'populacao_1km', 'renda_media_domiciliar', 'fluxo_pedestres_dia',
        'vagas_estacionamento', 'distancia_concorrente_m', 'meses_operacao',
        'investimento_marketing_mensal', 'n_funcionarios', 'ticket_medio_rs',
        'n_transacoes_mes', 'nota_satisfacao_cliente', 'faturamento_medio_mensal'
    ]
    # Garantir que apenas colunas existentes no DF sejam usadas
    cols_para_outliers = [c for c in cols_para_outliers if c in df.columns]

    # 3. Remover Outliers (IQR)
    print(f'Removendo outliers das colunas principais: {cols_para_outliers}')
    for col in cols_para_outliers:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        inf = Q1 - 1.5 * IQR
        sup = Q3 + 1.5 * IQR
        # Filtra mantendo nulos para não dropar antes da hora
        df = df[(df[col] >= inf) & (df[col] <= sup) | (df[col].isna())]

    # 4. Remover linhas com valores faltantes (NaN)
    linhas_antes_nan = df.shape[0]
    df = df.dropna()
    print(f'Linhas removidas por conterem valores nulos: {linhas_antes_nan - df.shape[0]}')

    # 5. Transformar variáveis categóricas em dummies
    cols_cat = df.select_dtypes(include=['object']).columns.tolist()
    if cols_cat:
        df = pd.get_dummies(df, columns=cols_cat, drop_first=True)

    # 6. Converter booleanos para int (0 e 1)
    cols_bool = df.select_dtypes(include=['bool']).columns
    df[cols_bool] = df[cols_bool].astype(int)

    print(f'Novo shape após limpeza completa: {df.shape}')
    return df

# Recarregar o DF original para limpar do zero corretamente
df = pd.read_csv(caminho_drive)
df = limpador_de_dados_radical(df)
display(df.head())

#Exploração dos dados e conhecimento da base

Para esta etapa desenvolvemos a EDA Master Blaster, ela se propõe a demonstrar as características do DF por meio de gráficos e de tabelas explicativas dos dados de que estamos tratando.

Vale ressaltar que tanto este código quanto o anterior são reutilizados de projetos anteriores, portanto a organização não está 100% adequada e há alguns bugs.

---

Por favor não tente exportar o pdf da EDA pelo colab, ele vai travar por completo.
---

In [ ]:
# EDA Tbm como objeto pq sim

def EDA_Master_Blaster(df, export_pdf):
    export_pdf = export_pdf
    print("--- Informações Gerais ---")
    print(df.info())

    print("\n--- Valores Faltantes ---")
    print(df.isnull().sum())

    print("\n--- Estatísticas Descritivas ---")
    desc = df.describe()
    display(desc)

    print("\n--- Erro Padrão da Média (SEM) ---")
    sem_values = df.select_dtypes(include=[np.number]).sem()
    sem_df = pd.DataFrame(sem_values, columns=['Erro Padrão'])
    display(sem_df)

    print("\n--- RMSE ---")
    rmse_values = np.sqrt(np.mean((df.select_dtypes(include=[np.number]) - df.select_dtypes(include=[np.number]).mean())**2, axis=0))
    rmse_df = pd.DataFrame(rmse_values, columns=['RMSE'])
    display(rmse_df)

    print("\n--- Histogramas das Variáveis Numéricas ---")
    df.hist(figsize=(15, 10), bins=20, color='skyblue', edgecolor='black')
    plt.tight_layout()
    plt.savefig('eda_hist.png')
    plt.show()

    print("\n--- Identificação de Outliers (Boxplots Individuais) ---")
    features_num = df.select_dtypes(include=[np.number]).columns.tolist()
    n_features = len(features_num)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(18, 4 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(features_num):
        sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
        axes[i].set_title(f'Boxplot de {col}')

    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.savefig('eda_boxplots.png')
    plt.show()

    print("\n--- Matriz de Correlação (Heatmap) ---")
    plt.figure(figsize=(12, 8))
    corr = df.corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
    plt.title("Mapa de Calor de Correlação")
    plt.savefig('eda_corr.png')
    plt.show()

    print("\n--- Top 5 Variáveis mais Correlacionadas com Faturamento ---")
    target_corr = corr['faturamento_medio_mensal'].abs().sort_values(ascending=False)
    print(target_corr.head(6))

    print("\n--- Análise de Dispersão vs Faturamento ---")
    target = 'faturamento_medio_mensal'
    features = [col for col in df.columns if col != target]
    n_features_disp = len(features)
    n_rows_disp = (n_features_disp + n_cols - 1) // n_cols

    fig_disp, axes_disp = plt.subplots(nrows=n_rows_disp, ncols=n_cols, figsize=(18, 4 * n_rows_disp))
    axes_disp = axes_disp.flatten()

    for i, col in enumerate(features):
        sns.scatterplot(data=df, x=col, y=target, ax=axes_disp[i])
        axes_disp[i].set_title(f'{col} vs Faturamento')

    for j in range(i + 1, len(axes_disp)):
        fig_disp.delaxes(axes_disp[j])

    plt.tight_layout()
    plt.savefig('eda_dispersion.png')
    plt.show()

    print("\n--- Distribuição de Faturamento ---")
    plt.figure(figsize=(10, 6))
    sns.histplot(df['faturamento_medio_mensal'], bins=20, kde=True, color='skyblue')
    plt.title("Distribuição de Faturamento")
    plt.savefig('eda_target_dist.png')
    plt.show()

    print("\n--- Identificação de Outliers no Faturamento (Tabela) ---")
    Q1 = df['faturamento_medio_mensal'].quantile(0.25)
    Q3 = df['faturamento_medio_mensal'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df['faturamento_medio_mensal'] < lower_bound) | (df['faturamento_medio_mensal'] > upper_bound)]

    if not outliers.empty:
        display(outliers)
    else:
        print("Nenhum outlier detectado para a variável faturamento_medio_mensal.")

    if export_pdf == True:
        pdf = FPDF()
        pdf.set_auto_page_break(auto=True, margin=15)
        pdf.add_page()
        pdf.set_font("Arial", "B", 16)
        pdf.cell(200, 10, "Relatorio EDA Master Blaster", ln=True, align="C")
        pdf.ln(10)

        pdf.set_font("Arial", "B", 12)
        pdf.cell(0, 10, "1. Histogramas Gerais", ln=True)
        pdf.image('eda_hist.png', x=10, w=180)

        pdf.add_page()
        pdf.cell(0, 10, "2. Boxplots de Outliers", ln=True)
        pdf.image('eda_boxplots.png', x=10, w=180)

        pdf.add_page()
        pdf.cell(0, 10, "3. Matriz de Correlação", ln=True)
        pdf.image('eda_corr.png', x=10, w=160)

        pdf.add_page()
        pdf.cell(0, 10, "4. Dispersão vs Faturamento", ln=True)
        pdf.image('eda_dispersion.png', x=10, w=180)

        pdf.ln(5)
        pdf.cell(0, 10, "5. Distribuição do Faturamento", ln=True)
        pdf.image('eda_target_dist.png', x=10, w=160)

        pdf.output("Relatorio_EDA.pdf")
        print("\nRelatório PDF 'Relatorio_EDA.pdf' gerado com sucesso!")

    else:
        print("\nRelatório EDA não exportado para PDF.")

    print("\nEDA Master Blaster concluída com sucesso, agora basta dominar o universo")

# Chamando a função,o export em pdf está quebrado, não colocar true se não vai quebrar o código
EDA_Master_Blaster(df, False)

#Profiler

Dentre as lojas que estão no dataset das existentes, há um padrão entre as que tem boa performance ou não, portanto aqui segue um perfilador de todos estes dados. =)



In [ ]:

def profiler_apocaliptico_de_lojas(df_input, export_pdf=True):
    # 1. Definindo os grupos
    df_profiler = df_input.copy()
    df_profiler['grupo_performance'] = np.where(df_profiler['faturamento_medio_mensal'] > 140, 'Alta Performance', 'Performance Padrão')

    cols_interesse = [
        'area_m2', 'populacao_1km', 'renda_media_domiciliar', 'fluxo_pedestres_dia',
        'vagas_estacionamento', 'meses_operacao', 'investimento_marketing_mensal',
        'n_funcionarios', 'ticket_medio_rs', 'n_transacoes_mes', 'nota_satisfacao_cliente'
    ]

    # 2. Cálculos
    perfil = df_profiler.groupby('grupo_performance')[cols_interesse].agg(['mean', 'median', 'std']).T
    grupo_alta = df_profiler[df_profiler['grupo_performance'] == 'Alta Performance']
    grupo_padrao = df_profiler[df_profiler['grupo_performance'] == 'Performance Padrão']

    p_values = {}
    for col in cols_interesse:
        t_stat, p_val = stats.ttest_ind(grupo_alta[col], grupo_padrao[col], equal_var=False)
        p_values[col] = round(p_val, 4)

    perfil_final = perfil.reset_index()
    perfil_final.columns = ['Variavel', 'Stat', 'Alta Performance', 'Performance Padrão']
    perfil_final['P-Value'] = perfil_final['Variavel'].map(p_values)

    # 3. Visualização (Corrigindo o erro de palette/hue)
    print("=== PROFILER DE PERFORMANCE DE LOJAS ===")
    display(perfil_final)

    top_vars = sorted(p_values, key=p_values.get)[:3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for i, var in enumerate(top_vars):
        sns.barplot(data=df_profiler, x='grupo_performance', y=var, hue='grupo_performance', ax=axes[i], palette='viridis', legend=False)
        axes[i].set_title(f'Diferença em {var}\n(p-value: {p_values[var]})')

    plt.tight_layout()
    plt.savefig('graficos_profiler.png')
    plt.show()

    # 4. Exportação para PDF
    if export_pdf:
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", "B", 16)
        pdf.cell(200, 10, "Relatorio de Profiling de Lojas", ln=True, align="C")
        pdf.set_font("Arial", "", 12)
        pdf.ln(10)
        pdf.cell(200, 10, f"Lojas Alta Performance: {len(grupo_alta)}", ln=True)
        pdf.cell(200, 10, f"Lojas Performance Padrao: {len(grupo_padrao)}", ln=True)
        pdf.ln(5)

        # Tabela simplificada no PDF
        pdf.set_font("Arial", "B", 10)
        pdf.cell(40, 10, "Variavel", 1)
        pdf.cell(30, 10, "Media (Alta)", 1)
        pdf.cell(30, 10, "Media (Padrao)", 1)
        pdf.cell(30, 10, "P-Value", 1)
        pdf.ln()

        pdf.set_font("Arial", "", 9)
        for col in cols_interesse:
            m_alta = round(grupo_alta[col].mean(), 2)
            m_pad = round(grupo_padrao[col].mean(), 2)
            pdf.cell(40, 10, col, 1)
            pdf.cell(30, 10, str(m_alta), 1)
            pdf.cell(30, 10, str(m_pad), 1)
            pdf.cell(30, 10, str(p_values[col]), 1)
            pdf.ln()

        pdf.ln(10)
        pdf.image('graficos_profiler.png', x=10, w=190)
        pdf.output("Relatorio_Lojas.pdf")
        print("\nRelatório PDF gerado com sucesso: Relatorio_Lojas.pdf")

profiler_apocaliptico_de_lojas(df)

#Parte 2: Modelos exploratórioa com todas as variáveis:

##Regressão por Boostraping

O primeiro método que aplicamos nos nossos testes preliminares foi o bootstraping em 450 tentativas com algumas regras, as variáveis (tanto em quantidade quanto em seleção) foram aleatorizadas, mas todos os modelos resultantes devem no mínimo cumprir duas regras, a divisão entre treino e teste (já que há a etapa de teste) e a relevância estatística de todas as variáveis contidas (pelo menos 95%). Uma vez concluída a etapa aleatória, os modelos são carregados na memória e são selecionados os 10 modelos com maior R^2 nos dados de treinamento e estes são testados contra os dados de teste, uma vez isso feito o modelo com o maior R^2 nos dados de teste é exportado para avaliação humana.

In [ ]:
# Façamos os modelos de regressão linear com refino estatístico por método boostrap

def fazedor_de_modelos_para_dominação_do_universo(df):
    target = 'faturamento_medio_mensal'

    # 1. Tratamento interno de NaNs para evitar o erro MissingDataError
    df_clean = df.copy()
    cols_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

    # 2. Converte booleanos para float para evitar erro no statsmodels
    df_numeric = df_clean.astype(float)
    features_pool = [col for col in df_numeric.columns if col != target]

    # Separação Treino e Teste (questão 3)
    X = df_numeric[features_pool]
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    modelos_resultados = []
    n_tentativas = 450

    print(f"Iniciando a busca aleatória por {n_tentativas} modelos com filtro de P-value...")

    for i in range(n_tentativas):
        # Escolhe um número aleatório de variáveis e quais variáveis
        k = random.randint(1, len(features_pool))
        selected_features = random.sample(features_pool, k)

        # Passo de Refino: Remover variáveis P > 0.05
        while len(selected_features) > 0:
            X_temp = sm.add_constant(X_train[selected_features])
            model_temp = sm.OLS(y_train, X_temp).fit()
            # Pega o maior p-value ignorando a constante
            p_values = model_temp.pvalues.drop('const', errors='ignore')
            max_p = p_values.max()

            if max_p > 0.05:
                excluded_feature = p_values.idxmax()
                selected_features.remove(excluded_feature)
            else:
                break

        if len(selected_features) > 0:
            # Treina modelo final da rodada com as selecionadas/refinadas
            lr = LinearRegression()
            lr.fit(X_train[selected_features], y_train)
            r2 = lr.score(X_train[selected_features], y_train)

            modelos_resultados.append({
                'features': selected_features,
                'r2_treino': r2
            })

    if not modelos_resultados:
        return "Nenhum modelo com variáveis significativas encontrado."

    # Ordena e pega os 10 melhores R2 de treino
    top_10_r2 = sorted(modelos_resultados, key=lambda x: x['r2_treino'], reverse=True)[:10]
    validacao_final = []

    print(f"\nTestando os {len(top_10_r2)} melhores modelos refinados contra os dados de teste...")

    for item in top_10_r2:
        cols = item['features']
        lr_val = LinearRegression()
        lr_val.fit(X_train[cols], y_train)

        y_pred = lr_val.predict(X_test[cols])
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        item['mse_teste'] = mse
        item['rmse_teste'] = rmse
        validacao_final.append(item)

    # O melhor modelo é o com menor RMSE
    melhor_modelo_info = min(validacao_final, key=lambda x: x['rmse_teste'])

    # Relatório Detalhado do vencedor
    X_vencedor_train = sm.add_constant(X_train[melhor_modelo_info['features']])
    modelo_final = sm.OLS(y_train, X_vencedor_train).fit()

    print("="*60)
    print("MELHOR MODELO REFINADO ENCONTRADO")
    print("="*60)
    print(f"Variáveis finais (P <= 0.05): {melhor_modelo_info['features']}")
    print(f"R-quadrado no Treino: {melhor_modelo_info['r2_treino']:.4f}")
    print(f"MSE no Teste: R$ {melhor_modelo_info['mse_teste']:.2f}")
    print(f"RMSE no Teste: R$ {melhor_modelo_info['rmse_teste']:.2f}")
    print(f"R-quadrado no Teste: {modelo_final.rsquared:.4f}")
    print("-"*60)
    print(modelo_final.summary())

    return modelo_final

# Executando a função
modelo_vencedor = fazedor_de_modelos_para_dominação_do_universo(df)

##LOOCV
O segundo modelo que testamos foi um de loocv usando todas as variáveis sem aleatoriedade neste aspecto, este modelo não foi funcional, mas vale ressaltar que apresentou um RMSE extremamente baixo em comparação com o incialmente visto na EDA.

In [ ]:
#Teste de LOOCV por esporte =)

def modelo_loocv_para_o_fim_do_universo(df):
  # Definindo a variável alvo primeiro para evitar o UnboundLocalError
  target = 'faturamento_medio_mensal'
  features = [col for col in df.columns if col != target]

  X = df[features]
  y = df[target]

  # Inicializando o modelo e o método LOOCV
  modelo_final = LinearRegression()
  loo = LeaveOneOut()

  # 1. Validação Cruzada (LOOCV)
  scores = cross_val_score(modelo_final, X, y, cv=loo, scoring='neg_mean_squared_error')
  mse_scores = -scores
  mse_loocv = np.mean(mse_scores)
  rmse_loocv = np.sqrt(mse_loocv)

  # 2. Treinamento no dataset completo para extrair coeficientes
  modelo_final.fit(X, y)

  print("="*60)
  print("RESULTADOS DO MODELO FINAL E VALIDAÇÃO LOOCV")
  print("="*60)
  print(f"Variáveis utilizadas: {features}")
  print(f"MSE (LOOCV): R$ {mse_loocv:.4f}")
  print(f"RMSE (LOOCV): R$ {rmse_loocv:.4f}")
  print("-"*60)
  print("COEFICIENTES DO MODELO:")
  print(f"Intercepto (Bias): {modelo_final.intercept_:.4f}")

  coef_df = pd.DataFrame({'Variável': features, 'Coeficiente': modelo_final.coef_})
  print(coef_df.to_string(index=False))
  print("-"*60)
  print("O modelo está validado e os coeficientes acima representam a equação final.")

#Executando a Função
modelo_loocv = modelo_loocv_para_o_fim_do_universo(df)

##Stepwise (iteração passo-a-passo)

O terceiro modelo que fizemos é um modelo construído por foward stepwise, usando como critério apenas a relevância estatística de 95% para adição de qualquer variável no modelo (recalculado a cada iteração do modelo), o RMSE deste output foi de 3.917.

In [ ]:
target = 'faturamento_medio_mensal'
# Get all potential numerical predictors, excluding target and id_loja
all_predictors = df.select_dtypes(include=np.number).columns.tolist()
all_predictors.remove(target)
if 'id_loja' in all_predictors:
    all_predictors.remove('id_loja')

selected_features = []
remaining_features = all_predictors[:]
best_model_summary = None
p_value_threshold_in = 0.05 # Threshold to add a variable

print("Iniciando seleção de variáveis forward (stepwise):\n")

while True:
    potential_adds = []

    # Try adding each remaining feature one by one to the current set of selected features
    for feature_to_add in remaining_features:
        current_candidate_features = selected_features + [feature_to_add]

        # Construct formula
        if not current_candidate_features:
            formula_str = f"{target} ~ 1"
        else:
            formula_str = f"{target} ~ {' + '.join(current_candidate_features)}"

        try:
            model = smf.ols(formula=formula_str, data=df).fit()

            if feature_to_add in model.pvalues.index:
                p_val_new_feature = model.pvalues[feature_to_add]
                potential_adds.append((p_val_new_feature, feature_to_add, model))
        except Exception:
            pass

    potential_adds.sort(key=lambda x: x[0])

    if potential_adds and potential_adds[0][0] < p_value_threshold_in:
        best_p_value, best_feature_to_add, current_best_model = potential_adds[0]
        selected_features.append(best_feature_to_add)
        remaining_features.remove(best_feature_to_add)
        best_model_summary = current_best_model

        print(f"--- Passo: Adicionando '{best_feature_to_add}' ---")
        print(f"Variáveis no modelo: {selected_features}")
        print(f"P-valor da nova variável ({best_feature_to_add}): {best_p_value:.4f}")
        print(f"R-quadrado Ajustado: {current_best_model.rsquared_adj:.4f}\n")
    else:
        break

print("--- Seleção de Variáveis Concluída ---")
print(f"Variáveis finais selecionadas: {selected_features}")

if best_model_summary:
    # Calculando MSE e RMSE do modelo final
    mse_final = best_model_summary.mse_resid
    rmse_final = np.sqrt(mse_final)

    print("\n" + "="*60)
    print("MÉTRICAS DE ERRO EM REAIS (Final Stepwise Model)")
    print("="*60)
    print(f"MSE em Reais: R$ {mse_final:.4f}")
    print(f"RMSE em Reais: R$ {rmse_final:.4f}")
    print("="*60)

    print("\n### Sumário do Modelo Final:")
    display(best_model_summary.summary())
else:
    print("Nenhuma variável foi selecionada.")

##Modelo de Regressão com Regularizaçã LASSO

Para finalizar os modelos tentativos – ainda sem a exclusão completa das variáveis inadequadas – aplicamos novamente o método do primeiro modelo, mas desta vez com a regularização LASSO das variáveis, é importante ressaltar que a regularização acontece antes do modelo passar por qualquer rankeamento (garantindo um modelo final mais representativo dado que qualquer função só será testada contra os dados de teste uma vez finalizada e regularizada) é também importante ressaltar que o alfa do LASSO também foi aleatorizado entre 0,001; 0,01; 0,1; 0,5; e 1.

In [ ]:
def fazedor_de_modelos_para_dominação_do_universo_reg(df_input, alphas=[0.001, 0.01, 0.1, 0.5, 1.0]):
    target = 'faturamento_medio_mensal'

    # 1. Tratamento interno de NaNs
    df_clean = df_input.copy()
    cols_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

    # 2. Converte booleanos para float
    df_numeric = df_clean.select_dtypes(include=[np.number]).astype(float)
    features_pool = [col for col in df_numeric.columns if col != target]

    # Separação Treino e Teste
    X = df_numeric[features_pool]
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Padronização (importante para LASSO)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features_pool, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features_pool, index=X_test.index)

    modelos_resultados = []
    n_tentativas = 450

    print(f"Iniciando busca bootstrap com refino de P-valor e regularização LASSO...")

    for i in range(n_tentativas):
        k = random.randint(1, len(features_pool))
        selected_features = random.sample(features_pool, k)

        # --- PASSO 1: Refino por P-value (Estatística Clássica) ---
        while len(selected_features) > 0:
            X_temp = sm.add_constant(X_train[selected_features])
            model_temp = sm.OLS(y_train, X_temp).fit()
            p_values = model_temp.pvalues.drop('const', errors='ignore')
            max_p = p_values.max()

            if max_p > 0.05:
                excluded_feature = p_values.idxmax()
                selected_features.remove(excluded_feature)
            else:
                break

        # --- PASSO 2: Regularização LASSO sobre as variáveis significativas ---
        if len(selected_features) > 0:
            for a in alphas:
                lasso = Lasso(alpha=a, random_state=42)
                lasso.fit(X_train_scaled[selected_features], y_train)
                r2_tr = lasso.score(X_train_scaled[selected_features], y_train)

                modelos_resultados.append({
                    'features': selected_features,
                    'alpha': a,
                    'r2_treino': r2_tr,
                    'lasso_model': lasso
                })

    if not modelos_resultados:
        print("ERRO: Nenhum modelo com variáveis significativas (P < 0.05) foi encontrado nas 450 tentativas.")
        return None

    # 3. Filtragem Top 10 R2 de Treino conforme solicitado
    top_10_r2 = sorted(modelos_resultados, key=lambda x: x.get('r2_treino', -1), reverse=True)[:10]
    validacao_final = []

    print(f"Testando os {len(top_10_r2)} melhores modelos de treino contra os dados de teste...")

    for item in top_10_r2:
        cols = item['features']
        model = item['lasso_model']

        y_pred = model.predict(X_test_scaled[cols])
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        item['mse_teste'] = mse
        item['rmse_teste'] = rmse
        validacao_final.append(item)

    # O melhor modelo é o com menor RMSE entre os top R2
    melhor_modelo_info = min(validacao_final, key=lambda x: x['rmse_teste'])

    print("="*60)
    print("MELHOR MODELO (P-VALOR + LASSO + VALIDACAO R2) ENCONTRADO")
    print("="*60)
    print(f"Variáveis Significativas: {melhor_modelo_info['features']}")
    print(f"Melhor Alpha LASSO: {melhor_modelo_info['alpha']}")
    print(f"RMSE Final no Teste: R$ {melhor_modelo_info['rmse_teste']:.2f}")
    print(f"R-quadrado no Treino (Lasso): {melhor_modelo_info['r2_treino']:.4f}")
    print("="*60)

    # Sumário OLS final para conferência de coeficientes e p-valores
    X_final_ols = sm.add_constant(X_train[melhor_modelo_info['features']])
    print(sm.OLS(y_train, X_final_ols).fit().summary())

    return melhor_modelo_info['lasso_model']

if 'df' in locals():
    modelo_vencedor = fazedor_de_modelos_para_dominação_do_universo_reg(df)
else:
    print("Dataframe 'df' não encontrado.")

#Parte 3: Veto das variáveis para uso nos modelos finais

Uma vez trabalhados os dados e observados os modelos (assim como os avisos automáticos gerados pelo Statsmodel), algumas variáveis se provaram inviáveis para o uso no modelo final, cada uma delas, e suas respectivas explicações, está a seguir:

	“n_transacoes_mes”, “n_transacoes_mes”:
	Estas variáveis têm dois problemas, por um lado são variáveis que estão embutidas no faturamento mensal – o faturamento é dado pela fórmula F_mensal=N_(de transações mês)* T_médio – enquanto por outro lado são variáveis que só podem ser descobertas após a abertura da loja, portanto não estão nos dados para tomada de decisão;
	“id_loja”:
	É apenas uma variável de identificação, não é relevante para o modelo;
	“meses_operacao”, “nota_satisfacao_cliente”:
	Estas variáveis somente podem ser descobertas após a operação da loja.

Uma vez excluídas estas vaiáveis do pool de modelagem, podemos partir para as etapas finais


#Parte 4: Modelagem final:

Agora, observadas estas limitações é possível construir um modelo final que será aplicável para quando formos elaborar as recomendações. Segue o resultado:


In [ ]:
def fazedor_de_modelos_para_dominação_do_universo_reg_v2(df_input, alphas=[0.001, 0.01, 0.1, 0.5, 1.0]):
    target = 'faturamento_medio_mensal'

    # 1. Tratamento interno de NaNs
    df_clean = df_input.copy()
    cols_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in cols_numericas:
        if df_clean[col].isnull().any():
            df_clean[col] = df_clean[col].fillna(df_clean[col].mean())

    # 2. Converte booleanos para float
    df_numeric = df_clean.select_dtypes(include=[np.number]).astype(float)
    # Filtro para remover n_transacoes_mes conforme solicitado
    features_pool = [col for col in df_numeric.columns if col != target and col != 'n_transacoes_mes' and col != 'id_loja' and col != 'ticket_medio_rs' and col != 'meses_operacao'and col != 'nota_satisfacao_cliente' ]

    # Separação Treino e Teste
    X = df_numeric[features_pool]
    y = df_numeric[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Padronização (importante para LASSO)
    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=features_pool, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=features_pool, index=X_test.index)

    modelos_resultados = []
    n_tentativas = 1000

    print(f"Iniciando busca bootstrap (SEM variáveis pós abertura) com refino de P-valor e regularização LASSO...")

    for i in range(n_tentativas):
        k = random.randint(1, len(features_pool))
        selected_features = random.sample(features_pool, k)

        # --- PASSO 1: Refino por P-value (Estatística Clássica) ---
        while len(selected_features) > 0:
            X_temp = sm.add_constant(X_train[selected_features])
            model_temp = sm.OLS(y_train, X_temp).fit()
            p_values = model_temp.pvalues.drop('const', errors='ignore')
            max_p = p_values.max()

            if max_p > 0.05:
                excluded_feature = p_values.idxmax()
                selected_features.remove(excluded_feature)
            else:
                break

        # --- PASSO 2: Regularização LASSO sobre as variáveis significativas ---
        if len(selected_features) > 0:
            for a in alphas:
                lasso = Lasso(alpha=a, random_state=42)
                lasso.fit(X_train_scaled[selected_features], y_train)
                r2_tr = lasso.score(X_train_scaled[selected_features], y_train)

                modelos_resultados.append({
                    'features': selected_features,
                    'alpha': a,
                    'r2_treino': r2_tr,
                    'lasso_model': lasso
                })

    if not modelos_resultados:
        print("ERRO: Nenhum modelo com variáveis significativas (P < 0.05) foi encontrado nas 450 tentativas.")
        return None

    # 3. Filtragem Top 10 R2 de Treino conforme solicitado
    top_10_r2 = sorted(modelos_resultados, key=lambda x: x.get('r2_treino', -1), reverse=True)[:10]
    validacao_final = []

    print(f"Testando os {len(top_10_r2)} melhores modelos de treino contra os dados de teste...")

    for item in top_10_r2:
        cols = item['features']
        model = item['lasso_model']

        y_pred = model.predict(X_test_scaled[cols])
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)

        item['mse_teste'] = mse
        item['rmse_teste'] = rmse
        validacao_final.append(item)

    # O melhor modelo é o com menor RMSE entre os top R2
    melhor_modelo_info = min(validacao_final, key=lambda x: x['rmse_teste'])

    print("="*60)
    print("MELHOR MODELO SEM 'N_TRANSACOES_MES' ENCONTRADO")
    print("="*60)
    print(f"Variáveis Significativas: {melhor_modelo_info['features']}")
    print(f"Melhor Alpha LASSO: {melhor_modelo_info['alpha']}")
    print(f"RMSE Final no Teste: R$ {melhor_modelo_info['rmse_teste']:.2f}")
    print(f"R-quadrado no Treino (Lasso): {melhor_modelo_info['r2_treino']:.4f}")
    print("="*60)

    # Sumário OLS final para conferência de coeficientes e p-valores
    X_final_ols = sm.add_constant(X_train[melhor_modelo_info['features']])
    print(sm.OLS(y_train, X_final_ols).fit().summary())

    return melhor_modelo_info['lasso_model']

if 'df' in locals():
    modelo_vencedor = fazedor_de_modelos_para_dominação_do_universo_reg_v2(df)
else:
    print("Dataframe 'df' não encontrado.")

#Parte 5: Aplicação, motor de recomendação e relatório executivo

Com o modelo final definido, esta etapa transforma a estimativa estatística em uma **decisão de investimento auditável**. Três pontos merecem destaque:

**1. Tudo é paramétrico.** Toda regra de negócio vive no dicionário `PARAMETROS`, no topo da célula: hurdle de faturamento, regra de decisão (`conservadora`, `pontual` ou `hibrida`), probabilidade mínima de aprovação, zona cinzenta, nível e método do intervalo de confiança, premissas econômicas (margem, custo fixo, CAPEX, custo de capital), pesos do score de priorização e as alavancas simuladas na análise de sensibilidade. Mudar um número ali reprocessa previsões, gráficos, recomendações e o PDF inteiro — não existe valor escrito à mão no relatório.

**2. A previsão vem acompanhada do risco.** O contexto de treino do modelo campeão é reconstruído de forma auditável (mesmo split, mesmo scaler), um modelo OLS gêmeo é estimado sobre as mesmas variáveis para fornecer p-valores, VIF e intervalos de predição, e cada candidata recebe: previsão pontual, intervalo de confiança, probabilidade de superar o hurdle, teste de aderência ao domínio de treino (extrapolação), decomposição do resultado por variável, análise de sensibilidade das alavancas gerenciais e a camada econômica (EBITDA, payback, VPL, TIR).

**3. Os dados do modelo e o dicionário de variáveis são explicitados.** A ficha técnica completa (coeficientes padronizados e em unidades originais, significância, VIF, diagnóstico de resíduos) e a explicação de cada variável — o que mede, em que unidade, se está disponível antes da abertura e por que foi mantida ou vetada — aparecem tanto no output da célula quanto no relatório em PDF.

O relatório é gerado no formato de consultoria estratégica (capa, sumário executivo, metodologia, dicionário, ficha técnica, matriz de decisão, uma página por praça, sensibilidade, caso econômico, riscos e apêndice), em paleta verde. Premissas que não vêm da base de dados são marcadas com `[#REF]`.


In [ ]:
# =====================================================================================
#  PARTE 5 - APLICACAO DO MODELO, MOTOR DE RECOMENDACAO PARAMETRICO E RELATORIO EXECUTIVO
# =====================================================================================
#  Este bloco transforma o modelo estatistico em uma DECISAO DE INVESTIMENTO auditavel.
#  Toda regra de negocio esta centralizada no dicionario PARAMETROS (single source of
#  truth): mudar um numero ali muda o relatorio, os graficos e as recomendacoes.
# =====================================================================================

import os, json, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats
from IPython.display import display

warnings.filterwarnings('ignore')

# =====================================================================================
# 1. PARAMETROS - TODA a recomendacao e parametrica e embutida no codigo
# =====================================================================================
PARAMETROS = {
    # ---------------- Fontes de dados -------------------------------------------------
    'caminho_candidatos': [
        '/content/drive/MyDrive/analytics III/T1/franquia_candidatos.csv',
        '/content/franquia_candidatos.csv',
        'franquia_candidatos.csv',
    ],
    'target': 'faturamento_medio_mensal',
    'coluna_id_candidata': 'codigo',

    # ---------------- Regra de decisao ------------------------------------------------
    # hurdle_faturamento: faturamento minimo (R$ mil/mes) para uma unidade ser aprovada.
    'hurdle_faturamento': 140.0,
    # regra_decisao: 'conservadora'  -> exige limite INFERIOR do IC >= hurdle
    #                'pontual'       -> exige apenas a previsao pontual >= hurdle
    #                'hibrida'       -> pontual >= hurdle e P(faturamento >= hurdle) >= prob_minima
    'regra_decisao': 'hibrida',
    'prob_minima_aprovacao': 0.80,     # usada na regra 'hibrida'
    'zona_cinzenta_pct': 0.07,         # +/- 7% em torno do hurdle => "Abrir com ressalvas"
    'nivel_confianca': 0.95,           # nivel dos intervalos reportados
    # metodo_intervalo: 'ols_pi' (intervalo de predicao OLS), 'residual_empirico' ou 'bootstrap'
    'metodo_intervalo': 'ols_pi',
    'n_bootstrap': 400,
    'rebaixar_por_extrapolacao': True, # candidata fora do dominio de treino cai um nivel
    'limite_z_extrapolacao': 2.5,      # |z| acima disso = fora do dominio observado

    # ---------------- Premissas economicas (NAO derivadas da base) [#REF] -------------
    'escala_faturamento_reais': 1000.0,   # faturamento esta em R$ mil/mes
    'margem_contribuicao_pct': 0.42,      # [#REF] premissa de trabalho
    'custo_fixo_mensal_reais': 38000.0,   # [#REF] aluguel + folha + ocupacao
    'capex_por_m2_reais': 3200.0,         # [#REF] obra + equipamentos por m2
    'capex_fixo_reais': 180000.0,         # [#REF] taxa de franquia + estoque inicial
    'taxa_desconto_anual': 0.14,          # [#REF] custo de capital do franqueado
    'horizonte_meses': 60,                # janela de avaliacao do VPL
    'payback_maximo_meses': 30,           # criterio de corte financeiro

    # ---------------- Score de priorizacao (soma dos pesos = 1) -----------------------
    'pesos_score': {'retorno': 0.45, 'confianca': 0.25, 'financeiro': 0.20, 'aderencia': 0.10},

    # ---------------- Alavancas de sensibilidade (variavel -> delta relativo) ---------
    'alavancas': {
        'investimento_marketing_mensal': [-0.20, -0.10, 0.10, 0.20, 0.30],
        'area_m2':                       [-0.20, -0.10, 0.10, 0.20],
        'n_funcionarios':                [-0.20, -0.10, 0.10, 0.20],
        'vagas_estacionamento':          [-0.20, 0.20],
        'fluxo_pedestres_dia':           [-0.10, 0.10],
    },

    # ---------------- Identidade visual (McKinsey-style em tons de verde) -------------
    'marca': {
        'verde_escuro': '#0B3D2E', 'verde': '#166E4A',  'verde_medio': '#2E9367',
        'verde_claro':  '#7CC49B', 'verde_palido': '#DCEFE4', 'verde_neon': '#4FB286',
        'cinza_texto':  '#3C4A44', 'cinza_claro': '#EEF3F0', 'cinza_medio': '#B9C6C0',
        'ambar': '#D9A441', 'terracota': '#B4553C', 'branco': '#FFFFFF',
    },
    'cliente': 'Rede de Franquias — Comitê de Expansão',
    'titulo_relatorio': 'ESTRATÉGIA DE EXPANSÃO DE UNIDADES',
    'subtitulo_relatorio': 'Modelo preditivo de faturamento e priorização de capital',
    'autores': 'Brenda | Enzo Claudino | Thainá | Natália',

    # ---------------- Saidas ----------------------------------------------------------
    'gerar_pdf': True,
    'arquivo_pdf': 'Relatorio_Executivo_Expansao.pdf',
    'arquivo_csv': 'matriz_de_decisao_candidatas.csv',
    'dpi': 160,
    'pasta_figuras': 'figuras_relatorio',
}

P = PARAMETROS
CORES = P['marca']
os.makedirs(P['pasta_figuras'], exist_ok=True)

# =====================================================================================
# 2. DICIONARIO DE VARIAVEIS - explicacao de cada variavel (output + relatorio + deck)
# =====================================================================================
DICIONARIO_VARIAVEIS = pd.DataFrame([
    # variavel, descricao, unidade, familia, disponivel_antes_da_abertura, status, justificativa
    ('id_loja', 'Identificador único da unidade na base histórica', 'ID', 'Identificador', 'Sim',
     'VETADA', 'Chave sem conteúdo econômico; correlaciona apenas por acaso amostral'),
    ('cidade', 'Município da unidade (transformado em dummies)', 'Categórica', 'Contexto', 'Sim',
     'ELEGÍVEL', 'Captura o efeito de praça: poder de compra, hábito de consumo e concorrência local'),
    ('regiao', 'Macrorregião do IBGE (transformada em dummies)', 'Categórica', 'Contexto', 'Sim',
     'ELEGÍVEL', 'Captura a heterogeneidade regional de demanda e de custo'),
    ('area_m2', 'Área útil da loja', 'm2', 'Ativo/Formato', 'Sim',
     'ELEGÍVEL', 'Proxy de capacidade de atendimento e de sortimento exposto'),
    ('populacao_1km', 'População residente no raio de 1 km', 'mil hab.', 'Demanda', 'Sim',
     'ELEGÍVEL', 'Tamanho do mercado primário da unidade'),
    ('renda_media_domiciliar', 'Renda média domiciliar do entorno', 'R$/mês', 'Demanda', 'Sim',
     'ELEGÍVEL', 'Poder de compra do público-alvo; sustenta o ticket médio'),
    ('fluxo_pedestres_dia', 'Fluxo médio diário de pedestres em frente ao ponto', 'pessoas/dia', 'Demanda', 'Sim',
     'ELEGÍVEL', 'Tráfego de passagem, principal fonte de compra por impulso'),
    ('vagas_estacionamento', 'Vagas de estacionamento disponíveis', 'vagas', 'Acesso', 'Sim',
     'ELEGÍVEL', 'Conversão do cliente motorizado e de cestas de maior volume'),
    ('distancia_concorrente_m', 'Distância até o concorrente direto mais próximo', 'metros', 'Competição', 'Sim',
     'ELEGÍVEL', 'Pressão competitiva sobre a captura de demanda'),
    ('shopping', 'Ponto dentro de shopping center (1) ou loja de rua (0)', 'Binária', 'Formato', 'Sim',
     'ELEGÍVEL', 'O formato define fluxo, sazonalidade e estrutura de custo'),
    ('investimento_marketing_mensal', 'Verba de marketing local por mês', 'R$/mês', 'Alavanca gerencial', 'Sim',
     'ELEGÍVEL', 'Única variável de decisão controlável depois de escolhido o ponto'),
    ('n_funcionarios', 'Quadro de funcionários da unidade', 'pessoas', 'Alavanca gerencial', 'Sim',
     'ELEGÍVEL', 'Capacidade de atendimento e velocidade de fila'),
    ('meses_operacao', 'Tempo de operação da unidade', 'meses', 'Maturidade', 'Não',
     'VETADA', 'Só existe depois da abertura: indisponível na decisão de investir (vazamento temporal)'),
    ('ticket_medio_rs', 'Valor médio por transação', 'R$', 'Resultado', 'Não',
     'VETADA', 'Componente contábil do alvo (Fat. = ticket x transações) e só observável pós-abertura'),
    ('n_transacoes_mes', 'Número de transações no mês', 'transações/mês', 'Resultado', 'Não',
     'VETADA', 'Componente contábil do alvo: usá-la é prever o faturamento com o próprio faturamento'),
    ('nota_satisfacao_cliente', 'Nota média de satisfação do cliente', 'escala 1-5', 'Resultado', 'Não',
     'VETADA', 'Medida pós-operação; indisponível para um ponto que ainda não abriu'),
    ('faturamento_medio_mensal', 'Faturamento médio mensal da unidade', 'R$ mil/mês', 'ALVO', 'Não',
     'ALVO', 'Variável dependente que o modelo estima para os pontos candidatos'),
], columns=['Variavel', 'Descricao', 'Unidade', 'Familia', 'Disponivel_pre_abertura',
            'Status_no_modelo', 'Justificativa'])

ROTULOS_CURTOS = {
    'area_m2': 'Área da loja (m²)', 'populacao_1km': 'População em 1 km',
    'renda_media_domiciliar': 'Renda média do entorno', 'fluxo_pedestres_dia': 'Fluxo de pedestres',
    'vagas_estacionamento': 'Vagas de estacionamento', 'distancia_concorrente_m': 'Distância do concorrente',
    'shopping': 'Ponto em shopping', 'investimento_marketing_mensal': 'Marketing mensal',
    'n_funcionarios': 'Tamanho da equipe', 'meses_operacao': 'Meses de operação',
    'ticket_medio_rs': 'Ticket médio', 'n_transacoes_mes': 'Transações no mês',
    'nota_satisfacao_cliente': 'Satisfação do cliente', 'faturamento_medio_mensal': 'Faturamento mensal',
    'id_loja': 'ID da loja', 'cidade': 'Cidade', 'regiao': 'Região',
}

def _lista_amigavel(lista_vars):
    """Converte 'area_m2, renda_media_domiciliar' em rótulos legíveis para o relatório."""
    partes = [p.strip() for p in str(lista_vars).split(',') if p.strip() and p.strip() != '-']
    return ', '.join(ROTULOS_CURTOS.get(p, p) for p in partes) or '-'

def _dicionario_de(var):
    """Retorna a linha do dicionario para uma variavel (inclusive dummies cidade_X / regiao_X)."""
    base = var
    rotulo = ROTULOS_CURTOS.get(var, var)
    if var.startswith('cidade_'):
        base, rotulo = 'cidade', f"Praça: {var.replace('cidade_', '')}"
    elif var.startswith('regiao_'):
        base, rotulo = 'regiao', f"Região: {var.replace('regiao_', '')}"
    linha = DICIONARIO_VARIAVEIS[DICIONARIO_VARIAVEIS['Variavel'] == base]
    if linha.empty:
        return {'Variavel': var, 'Rotulo': rotulo, 'Descricao': 'Variável derivada da base',
                'Unidade': '-', 'Familia': 'Outros', 'Disponivel_pre_abertura': 'Sim',
                'Status_no_modelo': 'ELEGÍVEL', 'Justificativa': '-'}
    d = linha.iloc[0].to_dict()
    d['Rotulo'] = rotulo
    if var.startswith(('cidade_', 'regiao_')):
        d['Descricao'] = f"Indicador binário de {rotulo.lower()} (referência = demais praças da base)"
        d['Unidade'] = 'Binária'
    return d

# =====================================================================================
# 3. RECONSTRUCAO AUDITAVEL DO CONTEXTO DO MODELO CAMPEAO
# =====================================================================================
VARIAVEIS_VETADAS = ['n_transacoes_mes', 'id_loja', 'ticket_medio_rs',
                     'meses_operacao', 'nota_satisfacao_cliente']

def reconstruir_contexto_modelo(df_base, modelo, params=P):
    """Reproduz EXATAMENTE o ambiente de treino da Parte 4 e extrai todo o diagnostico.

    Retorna um dicionario ('model card') com features, coeficientes padronizados e em
    unidades originais, p-valores, VIF, metricas de treino/teste e residuos.
    """
    target = params['target']
    df_num = df_base.select_dtypes(include=[np.number]).astype(float).copy()
    df_num = df_num.fillna(df_num.mean())
    pool = [c for c in df_num.columns if c != target and c not in VARIAVEIS_VETADAS]

    X, y = df_num[pool], df_num[target]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

    scaler = StandardScaler().fit(X_tr)                       # mesmo scaler do treino
    X_tr_s = pd.DataFrame(scaler.transform(X_tr), columns=pool, index=X_tr.index)
    X_te_s = pd.DataFrame(scaler.transform(X_te), columns=pool, index=X_te.index)

    # --- features do modelo campeao (com fallback caso a Parte 4 nao tenha rodado) ----
    if modelo is not None and hasattr(modelo, 'feature_names_in_'):
        feats = [f for f in modelo.feature_names_in_ if f in pool]
        alpha = float(getattr(modelo, 'alpha', np.nan))
    else:
        feats, alpha = [], 0.5
    if not feats:
        # Fallback: refino por p-valor sobre todo o pool disponivel
        feats = list(pool)
        while len(feats) > 1:
            pv = sm.OLS(y_tr, sm.add_constant(X_tr[feats])).fit().pvalues.drop('const', errors='ignore')
            if pv.max() > 0.05:
                feats.remove(pv.idxmax())
            else:
                break
        modelo = Lasso(alpha=alpha, random_state=42).fit(X_tr_s[feats], y_tr)
        print('[AVISO] modelo_vencedor nao encontrado: modelo equivalente re-estimado nesta celula.')

    # --- Lasso (preditor campeao) -----------------------------------------------------
    pred_tr = modelo.predict(X_tr_s[feats])
    pred_te = modelo.predict(X_te_s[feats])
    escala = pd.Series(scaler.scale_, index=pool)[feats]
    media_pool = pd.Series(scaler.mean_, index=pool)[feats]
    coef_pad = pd.Series(modelo.coef_, index=feats)            # impacto por desvio-padrao
    coef_orig = coef_pad / escala                              # impacto por unidade natural
    intercepto_orig = float(modelo.intercept_ - (coef_pad * media_pool / escala).sum())

    # --- OLS gemeo (inferencia, IC de coeficientes e intervalo de predicao) -----------
    ols = sm.OLS(y_tr, sm.add_constant(X_tr[feats])).fit()
    ic_coef = ols.conf_int(alpha=1 - params['nivel_confianca'])
    ic_coef.columns = ['IC_inf', 'IC_sup']

    def _vif(col):
        outros = [f for f in feats if f != col]
        if not outros:
            return 1.0
        r2 = sm.OLS(X_tr[col], sm.add_constant(X_tr[outros])).fit().rsquared
        return float(1 / max(1e-9, 1 - r2))

    tabela_coef = pd.DataFrame({
        'Variavel': feats,
        'Rotulo': [_dicionario_de(f)['Rotulo'] for f in feats],
        'Familia': [_dicionario_de(f)['Familia'] for f in feats],
        'Unidade': [_dicionario_de(f)['Unidade'] for f in feats],
        'Coef_padronizado': coef_pad.values,
        'Coef_unidade_original': coef_orig.values,
        'Coef_OLS': [ols.params.get(f, np.nan) for f in feats],
        'P_valor': [ols.pvalues.get(f, np.nan) for f in feats],
        'IC_inf': [ic_coef.loc[f, 'IC_inf'] if f in ic_coef.index else np.nan for f in feats],
        'IC_sup': [ic_coef.loc[f, 'IC_sup'] if f in ic_coef.index else np.nan for f in feats],
        'VIF': [_vif(f) for f in feats],
    }).sort_values('Coef_padronizado', key=np.abs, ascending=False).reset_index(drop=True)
    tabela_coef['Peso_relativo_%'] = (100 * tabela_coef['Coef_padronizado'].abs()
                                      / tabela_coef['Coef_padronizado'].abs().sum())

    resid_tr = y_tr - pred_tr
    resid_te = y_te - pred_te
    metricas = {
        'n_total': int(len(df_num)), 'n_treino': int(len(y_tr)), 'n_teste': int(len(y_te)),
        'n_variaveis_pool': int(len(pool)), 'n_variaveis_modelo': int(len(feats)),
        'alpha_lasso': alpha,
        'r2_treino': float(r2_score(y_tr, pred_tr)), 'r2_teste': float(r2_score(y_te, pred_te)),
        'rmse_treino': float(np.sqrt(mean_squared_error(y_tr, pred_tr))),
        'rmse_teste': float(np.sqrt(mean_squared_error(y_te, pred_te))),
        'mae_teste': float(mean_absolute_error(y_te, pred_te)),
        'mape_teste': float(np.mean(np.abs(resid_te / y_te)) * 100),
        'erro_relativo_%': float(np.sqrt(mean_squared_error(y_te, pred_te)) / y_te.mean() * 100),
        'r2_ajustado_ols': float(ols.rsquared_adj), 'f_pvalue_ols': float(ols.f_pvalue),
        'durbin_watson': float(sm.stats.stattools.durbin_watson(ols.resid)),
        'jarque_bera_p': float(sm.stats.stattools.jarque_bera(ols.resid)[1]),
        'breusch_pagan_p': float(sm.stats.diagnostic.het_breuschpagan(ols.resid, ols.model.exog)[1]),
        'vif_maximo': float(tabela_coef['VIF'].max()),
        'desvio_residual': float(np.std(np.concatenate([resid_tr, resid_te]), ddof=1)),
        'media_alvo': float(y.mean()), 'desvio_alvo': float(y.std()),
    }

    return {
        'modelo': modelo, 'ols': ols, 'scaler': scaler, 'features': feats, 'pool': pool,
        'tabela_coef': tabela_coef, 'metricas': metricas, 'intercepto_original': intercepto_orig,
        'X_tr': X_tr, 'X_te': X_te, 'y_tr': y_tr, 'y_te': y_te,
        'X_tr_s': X_tr_s, 'X_te_s': X_te_s, 'pred_tr': pred_tr, 'pred_te': pred_te,
        'resid_tr': resid_tr, 'resid_te': resid_te, 'df_num': df_num,
    }

# =====================================================================================
# 4. PREPARACAO DAS CANDIDATAS + TESTE DE APLICABILIDADE (extrapolacao)
# =====================================================================================
def carregar_candidatas(params=P):
    for caminho in params['caminho_candidatos']:
        if os.path.exists(caminho):
            print(f'Base de candidatas carregada de: {caminho}')
            return pd.read_csv(caminho)
    raise FileNotFoundError('Nenhum caminho de franquia_candidatos.csv foi encontrado.')

def preparar_candidatas(df_cand, ctx, params=P):
    """Alinha as candidatas ao espaco de features do treino.

    Diferencas relevantes frente a versao anterior:
      * imputacao de faltantes usa a MEDIANA DA BASE HISTORICA (e nao a media das 5
        candidatas), evitando que uma amostra minuscula defina o valor imputado;
      * registra quais celulas foram imputadas para reporte de qualidade de dados;
      * marca extrapolacao por variavel (|z| vs. distribuicao de treino).
    """
    df_c = df_cand.copy()
    log_imputacao = []
    for col in df_c.select_dtypes(include=[np.number]).columns:
        if df_c[col].isnull().any() and col in ctx['df_num'].columns:
            valor = float(ctx['df_num'][col].median())
            for cod in df_c.loc[df_c[col].isnull(), params['coluna_id_candidata']]:
                log_imputacao.append({'Candidata': cod, 'Variavel': col,
                                      'Valor_imputado': round(valor, 2),
                                      'Origem': 'Mediana da base histórica'})
            df_c[col] = df_c[col].fillna(valor)

    X_c = pd.get_dummies(df_c)
    for col in X_c.select_dtypes(include=['bool']).columns:
        X_c[col] = X_c[col].astype(float)
    for col in ctx['pool']:
        if col not in X_c.columns:
            X_c[col] = 0.0
    X_c = X_c[ctx['pool']].astype(float)
    X_c.index = df_c[params['coluna_id_candidata']].values

    # --- teste de aplicabilidade: a candidata esta dentro do dominio de treino? -------
    continuas = [f for f in ctx['features']
                 if not f.startswith(('cidade_', 'regiao_')) and f != 'shopping']
    z = pd.DataFrame(index=X_c.index)
    for f in continuas:
        mu, sd = ctx['X_tr'][f].mean(), ctx['X_tr'][f].std(ddof=1)
        z[f] = (X_c[f] - mu) / (sd if sd > 0 else 1)
    flags = (z.abs() > params['limite_z_extrapolacao'])
    diag = pd.DataFrame({
        'z_maximo_abs': z.abs().max(axis=1) if len(continuas) else 0.0,
        'variaveis_fora_dominio': flags.sum(axis=1) if len(continuas) else 0,
        'detalhe_extrapolacao': [', '.join(flags.columns[flags.loc[i]]) if len(continuas) and flags.loc[i].any()
                                 else '-' for i in X_c.index],
    })
    diag['fora_do_dominio'] = diag['variaveis_fora_dominio'] > 0
    return X_c, diag, z, pd.DataFrame(log_imputacao)

# =====================================================================================
# 5. MOTOR DE RECOMENDACAO PARAMETRICO (previsao -> risco -> economia -> decisao)
# =====================================================================================
def _tir_fluxo_constante(capex, fluxo_mensal, meses):
    """TIR mensal de um fluxo constante, por biseccao (evita dependencia externa)."""
    if fluxo_mensal <= 0:
        return np.nan
    def vpl(i):
        return sum(fluxo_mensal / (1 + i) ** t for t in range(1, meses + 1)) - capex
    lo, hi = 1e-9, 1.0
    if vpl(lo) < 0:
        return np.nan
    for _ in range(200):
        mid = (lo + hi) / 2
        if vpl(mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def calcular_intervalos(ctx, X_novo, params=P):
    """Intervalo de previsao pelo metodo escolhido em PARAMETROS['metodo_intervalo']."""
    feats, conf = ctx['features'], params['nivel_confianca']
    alpha = 1 - conf
    metodo = params['metodo_intervalo']

    Xs = pd.DataFrame(ctx['scaler'].transform(X_novo[ctx['pool']]),
                      columns=ctx['pool'], index=X_novo.index)
    ponto = ctx['modelo'].predict(Xs[feats])

    if metodo == 'ols_pi':
        Xc = sm.add_constant(X_novo[feats], has_constant='add')
        sf = ctx['ols'].get_prediction(Xc).summary_frame(alpha=alpha)
        meia_largura = (sf['obs_ci_upper'] - sf['obs_ci_lower']).values / 2
        se_total = meia_largura / stats.norm.ppf(1 - alpha / 2)
    elif metodo == 'residual_empirico':
        q = np.quantile(np.abs(np.concatenate([ctx['resid_tr'], ctx['resid_te']])), conf)
        meia_largura = np.full(len(X_novo), q)
        se_total = np.full(len(X_novo), ctx['metricas']['desvio_residual'])
    else:  # bootstrap sobre residuos do treino
        rng = np.random.default_rng(42)
        res = np.concatenate([ctx['resid_tr'], ctx['resid_te']])
        amostras = np.array([ponto + rng.choice(res, size=len(ponto), replace=True)
                             for _ in range(params['n_bootstrap'])])
        inf_b = np.quantile(amostras, alpha / 2, axis=0)
        sup_b = np.quantile(amostras, 1 - alpha / 2, axis=0)
        meia_largura = (sup_b - inf_b) / 2
        se_total = meia_largura / stats.norm.ppf(1 - alpha / 2)

    return ponto, ponto - meia_largura, ponto + meia_largura, se_total

def motor_de_recomendacao(df_cand, X_cand, diag, ctx, params=P):
    """Gera a matriz de decisao completa: previsao, risco, economia, score e veredito."""
    hurdle = params['hurdle_faturamento']
    ponto, ic_inf, ic_sup, se = calcular_intervalos(ctx, X_cand, params)

    R = df_cand.copy().reset_index(drop=True)
    R['PREVISAO_FATURAMENTO'] = ponto
    R['IC_INF'] = ic_inf
    R['IC_SUP'] = ic_sup
    R['ERRO_PADRAO_PREVISAO'] = se
    R['AMPLITUDE_IC'] = ic_sup - ic_inf
    R['PRECISAO_%'] = 100 * (1 - (ic_sup - ic_inf) / (2 * np.maximum(ponto, 1e-9)))
    R['PROB_SUPERAR_HURDLE'] = 1 - stats.norm.cdf((hurdle - ponto) / np.maximum(se, 1e-9))
    R['GAP_VS_HURDLE'] = ponto - hurdle
    R['GAP_VS_HURDLE_%'] = 100 * (ponto - hurdle) / hurdle

    # ---------------- Camada economica (premissas de PARAMETROS) [#REF] --------------
    esc = params['escala_faturamento_reais']
    R['RECEITA_MENSAL_R$'] = R['PREVISAO_FATURAMENTO'] * esc
    R['MARGEM_CONTRIBUICAO_R$'] = R['RECEITA_MENSAL_R$'] * params['margem_contribuicao_pct']
    mkt = R['investimento_marketing_mensal'] if 'investimento_marketing_mensal' in R else 0.0
    R['EBITDA_MENSAL_R$'] = R['MARGEM_CONTRIBUICAO_R$'] - params['custo_fixo_mensal_reais'] - mkt
    area = R['area_m2'] if 'area_m2' in R else 0.0
    R['CAPEX_R$'] = params['capex_fixo_reais'] + params['capex_por_m2_reais'] * area
    R['PAYBACK_MESES'] = np.where(R['EBITDA_MENSAL_R$'] > 0,
                                  R['CAPEX_R$'] / R['EBITDA_MENSAL_R$'].replace(0, np.nan), np.inf)
    i_m = (1 + params['taxa_desconto_anual']) ** (1 / 12) - 1
    H = params['horizonte_meses']
    fator = sum(1 / (1 + i_m) ** t for t in range(1, H + 1))
    R['VPL_R$'] = R['EBITDA_MENSAL_R$'] * fator - R['CAPEX_R$']
    R['TIR_ANUAL_%'] = [100 * ((1 + _tir_fluxo_constante(c, f, H)) ** 12 - 1)
                        if np.isfinite(_tir_fluxo_constante(c, f, H)) else np.nan
                        for c, f in zip(R['CAPEX_R$'], R['EBITDA_MENSAL_R$'])]
    R['ROI_CAPEX_%'] = 100 * R['VPL_R$'] / R['CAPEX_R$']

    # ---------------- Aplicabilidade -------------------------------------------------
    diag_r = diag.reset_index(drop=True)
    for c in ['z_maximo_abs', 'variaveis_fora_dominio', 'detalhe_extrapolacao', 'fora_do_dominio']:
        R[c.upper()] = diag_r[c].values

    # ---------------- Score de priorizacao (0-100) -----------------------------------
    def _norm(s, inverter=False):
        s = pd.Series(s).astype(float).replace([np.inf, -np.inf], np.nan)
        s = s.fillna(s.min() if not inverter else s.max())
        if s.max() - s.min() < 1e-12:
            return pd.Series(np.full(len(s), 50.0), index=s.index)
        v = 100 * (s - s.min()) / (s.max() - s.min())
        return 100 - v if inverter else v

    w = params['pesos_score']
    R['SCORE_RETORNO'] = _norm(R['PREVISAO_FATURAMENTO'])
    R['SCORE_CONFIANCA'] = _norm(R['PROB_SUPERAR_HURDLE'])
    R['SCORE_FINANCEIRO'] = _norm(R['VPL_R$'])
    R['SCORE_ADERENCIA'] = _norm(R['Z_MAXIMO_ABS'], inverter=True)
    R['SCORE_PRIORIDADE'] = (w['retorno'] * R['SCORE_RETORNO'] + w['confianca'] * R['SCORE_CONFIANCA'] +
                             w['financeiro'] * R['SCORE_FINANCEIRO'] + w['aderencia'] * R['SCORE_ADERENCIA'])

    # ---------------- Regra de decisao parametrica -----------------------------------
    banda = hurdle * params['zona_cinzenta_pct']
    regra = params['regra_decisao']

    def _decidir(r):
        if regra == 'conservadora':
            aprovado = r['IC_INF'] >= hurdle
        elif regra == 'pontual':
            aprovado = r['PREVISAO_FATURAMENTO'] >= hurdle
        else:  # hibrida
            aprovado = (r['PREVISAO_FATURAMENTO'] >= hurdle and
                        r['PROB_SUPERAR_HURDLE'] >= params['prob_minima_aprovacao'])
        if aprovado and r['PAYBACK_MESES'] > params['payback_maximo_meses']:
            return 'ABRIR COM RESSALVAS', ('Supera o hurdle de faturamento, mas o payback fica acima do '
                                          'limite financeiro definido pelo comitê')
        if aprovado:
            return 'ABRIR', 'Atende simultaneamente ao critério de faturamento e ao critério financeiro'
        if abs(r['GAP_VS_HURDLE']) <= banda or r['PREVISAO_FATURAMENTO'] >= hurdle:
            return 'ABRIR COM RESSALVAS', ('Resultado dentro da zona cinzenta do hurdle: aprovar somente com '
                                          'plano de mitigação')
        return 'NÃO ABRIR', 'Faturamento projetado abaixo do hurdle por margem relevante'

    ordem = {'ABRIR': 0, 'ABRIR COM RESSALVAS': 1, 'NÃO ABRIR': 2}
    inverso = {v: k for k, v in ordem.items()}
    decisoes, motivos = [], []
    for _, r in R.iterrows():
        d, m = _decidir(r)
        if params['rebaixar_por_extrapolacao'] and r['FORA_DO_DOMINIO'] and d != 'NÃO ABRIR':
            d = inverso[min(2, ordem[d] + 1)]
            m += (f"; rebaixada porque o perfil está fora do domínio de treino "
                  f"({_lista_amigavel(r['DETALHE_EXTRAPOLACAO'])})")
        decisoes.append(d)
        motivos.append(m)
    R['RECOMENDACAO'] = decisoes
    R['JUSTIFICATIVA'] = motivos

    R = R.sort_values(['RECOMENDACAO', 'SCORE_PRIORIDADE'],
                      key=lambda s: s.map(ordem) if s.name == 'RECOMENDACAO' else -s).reset_index(drop=True)
    R['RANKING'] = np.arange(1, len(R) + 1)
    return R

# =====================================================================================
# 6. DECOMPOSICAO DE VALOR E SENSIBILIDADE (as alavancas sao parametricas)
# =====================================================================================
def decompor_previsao(X_cand, ctx, params=P):
    """Quanto cada variavel adiciona/subtrai da previsao vs. a loja media da base."""
    feats = ctx['features']
    Xs = pd.DataFrame(ctx['scaler'].transform(X_cand[ctx['pool']]),
                      columns=ctx['pool'], index=X_cand.index)
    contrib = Xs[feats] * pd.Series(ctx['modelo'].coef_, index=feats)
    contrib['BASE_MEDIA_DA_REDE'] = ctx['modelo'].intercept_
    return contrib

def analise_de_sensibilidade(X_cand, ctx, params=P):
    """Simula as alavancas de PARAMETROS['alavancas'] recalculando a previsao."""
    feats = ctx['features']
    base_pred = ctx['modelo'].predict(pd.DataFrame(
        ctx['scaler'].transform(X_cand[ctx['pool']]), columns=ctx['pool'], index=X_cand.index)[feats])
    linhas = []
    for alavanca, deltas in params['alavancas'].items():
        if alavanca not in X_cand.columns:
            continue
        no_modelo = alavanca in feats
        for d in deltas:
            X_mod = X_cand.copy()
            X_mod[alavanca] = X_mod[alavanca] * (1 + d)
            Xs = pd.DataFrame(ctx['scaler'].transform(X_mod[ctx['pool']]),
                              columns=ctx['pool'], index=X_mod.index)
            novo = ctx['modelo'].predict(Xs[feats])
            for i, cod in enumerate(X_cand.index):
                linhas.append({'Candidata': cod, 'Alavanca': alavanca, 'Delta_%': d * 100,
                               'Previsao_base': base_pred[i], 'Previsao_simulada': novo[i],
                               'Impacto_R$mil': novo[i] - base_pred[i],
                               'No_modelo': 'Sim' if no_modelo else 'Não (coeficiente nulo)'})
    return pd.DataFrame(linhas)

def esforco_para_atingir_hurdle(R, ctx, params=P):
    """Para cada candidata abaixo do hurdle: de quanto teria de ser a alavanca gerencial."""
    hurdle = params['hurdle_faturamento']
    coefs = ctx['tabela_coef'].set_index('Variavel')['Coef_unidade_original']
    linhas = []
    for _, r in R.iterrows():
        gap = hurdle - r['PREVISAO_FATURAMENTO']
        if gap <= 0:
            linhas.append({'Candidata': r[params['coluna_id_candidata']], 'Gap_R$mil': 0.0,
                           'Alavanca': '—', 'Variacao_necessaria': 'Já supera o hurdle',
                           'Viavel': 'n/a'})
            continue
        achou = False
        for alav in params['alavancas']:
            if alav in coefs.index and coefs[alav] > 0 and alav in R.columns:
                delta_abs = gap / coefs[alav]
                atual = float(r[alav])
                var_pct = 100 * delta_abs / atual if atual else np.inf
                linhas.append({'Candidata': r[params['coluna_id_candidata']],
                               'Gap_R$mil': round(gap, 1), 'Alavanca': alav,
                               'Variacao_necessaria': f'+{delta_abs:,.0f} ({var_pct:,.0f}%)',
                               'Viavel': 'Sim' if var_pct <= 50 else 'Não (fora do razoável)'})
                achou = True
        if not achou:
            linhas.append({'Candidata': r[params['coluna_id_candidata']], 'Gap_R$mil': round(gap, 1),
                           'Alavanca': 'Nenhuma alavanca gerencial no modelo',
                           'Variacao_necessaria': 'Reprovar ou trocar o ponto', 'Viavel': 'Não'})
    return pd.DataFrame(linhas)

# =====================================================================================
# 7. BIBLIOTECA DE GRAFICOS - identidade visual unica (consultoria, tons de verde)
# =====================================================================================
def aplicar_estilo(params=P):
    c = params['marca']
    plt.rcParams.update({
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
        'axes.edgecolor': c['cinza_medio'], 'axes.linewidth': 0.8,
        'axes.grid': True, 'grid.color': c['cinza_claro'], 'grid.linewidth': 0.9,
        'axes.spines.top': False, 'axes.spines.right': False,
        'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlecolor': c['verde_escuro'],
        'axes.labelcolor': c['cinza_texto'], 'axes.labelsize': 10,
        'xtick.color': c['cinza_texto'], 'ytick.color': c['cinza_texto'],
        'xtick.labelsize': 9, 'ytick.labelsize': 9,
        'font.family': 'DejaVu Sans', 'text.color': c['cinza_texto'],
        'legend.frameon': False, 'figure.autolayout': False,
    })
    # 'R$' em textos não deve ser interpretado como fórmula matemática (matplotlib >= 3.5)
    if 'text.parse_math' in plt.rcParams:
        plt.rcParams['text.parse_math'] = False

CORES_DECISAO = {'ABRIR': CORES['verde'], 'ABRIR COM RESSALVAS': CORES['ambar'],
                 'NÃO ABRIR': CORES['terracota']}

ALTURA_EXHIBIT = {}   # caminho -> altura aproximada em mm quando impresso com 178 mm de largura

def _salvar(fig, nome, params=P):
    caminho = os.path.join(params['pasta_figuras'], nome)
    fig.savefig(caminho, dpi=params['dpi'], bbox_inches='tight', facecolor='white')
    ALTURA_EXHIBIT[caminho] = 178 * fig.get_figheight() / fig.get_figwidth() * 1.06
    return caminho

def gerar_graficos(R, ctx, sens, contrib, zscores, params=P):
    aplicar_estilo(params)
    figs = {}
    hurdle = params['hurdle_faturamento']
    idc = params['coluna_id_candidata']

    # --- E1: ranking com intervalo de confianca --------------------------------------
    fig, ax = plt.subplots(figsize=(11, 5.6))
    d = R.sort_values('PREVISAO_FATURAMENTO')
    y = np.arange(len(d))
    cores = [CORES_DECISAO[v] for v in d['RECOMENDACAO']]
    ax.barh(y, d['PREVISAO_FATURAMENTO'], color=cores, height=0.55, zorder=3)
    ax.errorbar(d['PREVISAO_FATURAMENTO'], y,
                xerr=[d['PREVISAO_FATURAMENTO'] - d['IC_INF'], d['IC_SUP'] - d['PREVISAO_FATURAMENTO']],
                fmt='none', ecolor=CORES['verde_escuro'], elinewidth=1.4, capsize=5, zorder=4)
    ax.axvspan(hurdle * (1 - params['zona_cinzenta_pct']), hurdle * (1 + params['zona_cinzenta_pct']),
               color=CORES['ambar'], alpha=0.12, zorder=1)
    ax.axvline(hurdle, color=CORES['verde_escuro'], ls='--', lw=1.6, zorder=5)
    ax.text(hurdle, len(d) - 0.35, f'  Hurdle: R$ {hurdle:,.0f} mil/mês', color=CORES['verde_escuro'],
            fontsize=9, fontweight='bold', va='center')
    for i, (_, r) in enumerate(d.iterrows()):
        ax.text(r['IC_SUP'] + 3, i, f"R$ {r['PREVISAO_FATURAMENTO']:,.1f} mil  |  P(>hurdle) = {r['PROB_SUPERAR_HURDLE']*100:,.0f}%",
                va='center', fontsize=9, color=CORES['cinza_texto'])
    ax.set_yticks(y); ax.set_yticklabels([f"{r[idc]} - {r['cidade']}" for _, r in d.iterrows()], fontweight='bold')
    ax.set_xlabel('Faturamento médio mensal projetado (R$ mil)')
    ax.set_xlim(0, max(d['IC_SUP']) * 1.42)
    ax.legend(handles=[mpatches.Patch(color=v, label=k.title()) for k, v in CORES_DECISAO.items()],
              loc='lower right', ncol=3, fontsize=9)
    n_ok = int((R['PREVISAO_FATURAMENTO'] >= hurdle).sum())
    fig.suptitle(f'{n_ok} de {len(R)} praças superam o hurdle na previsão pontual — o intervalo de confiança '
                 f'é o que separa aprovação de ressalva',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E1_ranking'] = _salvar(fig, 'E1_ranking_candidatas.png', params); plt.show()

    # --- E2: drivers do modelo (impacto por desvio-padrao) ---------------------------
    tc = ctx['tabela_coef'].sort_values('Coef_padronizado')
    fig, ax = plt.subplots(figsize=(11, max(4.2, 0.44 * len(tc) + 1.6)))
    cores = [CORES['verde'] if v >= 0 else CORES['terracota'] for v in tc['Coef_padronizado']]
    ax.barh(tc['Rotulo'], tc['Coef_padronizado'], color=cores, height=0.6, zorder=3)
    for i, (v, p) in enumerate(zip(tc['Coef_padronizado'], tc['Peso_relativo_%'])):
        ax.text(v + (1.2 if v >= 0 else -1.2), i, f'{v:+,.1f}  ({p:,.0f}% do poder explicativo)',
                va='center', ha='left' if v >= 0 else 'right', fontsize=8.6, color=CORES['cinza_texto'])
    ax.axvline(0, color=CORES['cinza_medio'], lw=1)
    ax.set_xlabel('Impacto no faturamento (R$ mil/mês) por +1 desvio-padrão da variável')
    lim = max(abs(tc['Coef_padronizado'])) * 1.75
    ax.set_xlim(-lim, lim)
    top3 = ', '.join(ctx['tabela_coef']['Rotulo'].head(3))
    peso3 = ctx['tabela_coef']['Peso_relativo_%'].head(3).sum()
    fig.suptitle(f'Três variáveis concentram {peso3:,.0f}% do poder explicativo: {top3}'[:120],
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E2_drivers'] = _salvar(fig, 'E2_drivers_modelo.png', params); plt.show()

    # --- E3: decomposição (waterfall) - uma versão por candidata ---------------------
    def _waterfall(cod, compacto=False):
        linha = R[R[idc] == cod].iloc[0]
        cidade_cod = linha['cidade']
        c = contrib.loc[cod].drop('BASE_MEDIA_DA_REDE')
        c = c[c.abs() > 1e-6].sort_values(key=np.abs, ascending=False).head(8)
        rotulos = ['Média da rede'] + [_dicionario_de(i)['Rotulo'] for i in c.index] + ['Previsão final']
        valores = [contrib.loc[cod, 'BASE_MEDIA_DA_REDE']] + list(c.values)
        resto = linha['PREVISAO_FATURAMENTO'] - sum(valores)
        if abs(resto) > 0.05:
            rotulos.insert(-1, 'Demais variáveis'); valores.append(resto)
        fig, ax = plt.subplots(figsize=(11, 3.9 if compacto else 5.2))
        acum = 0.0
        for i, v in enumerate(valores):
            cor = CORES['verde_escuro'] if i == 0 else (CORES['verde_medio'] if v >= 0 else CORES['terracota'])
            base_barra = 0 if i == 0 else acum
            ax.bar(i, v, bottom=base_barra, color=cor, width=0.6, zorder=3)
            topo = base_barra + max(v, 0)
            ax.text(i, topo + 3.0, f'{v:+,.1f}' if i else f'{v:,.1f}', ha='center', fontsize=8.8,
                    fontweight='bold', color=CORES['verde_escuro'] if v >= 0 else CORES['terracota'])
            acum = v if i == 0 else acum + v
            if i < len(valores) - 1:
                ax.plot([i - 0.3 + 0.6, i + 1 - 0.3], [acum, acum], color=CORES['cinza_medio'],
                        lw=0.9, ls=':', zorder=2)
        ax.plot([len(valores) - 1 + 0.3, len(valores) - 0.3], [acum, acum],
                color=CORES['cinza_medio'], lw=0.9, ls=':', zorder=2)
        ax.bar(len(valores), acum, color=CORES['verde'], width=0.6, zorder=3)
        ax.text(len(valores), acum + 3.0, f'{acum:,.1f}', ha='center', fontsize=9.5,
                fontweight='bold', color=CORES['verde_escuro'])
        ax.axhline(hurdle, color=CORES['cinza_texto'], ls='--', lw=1.3, zorder=2)
        ax.text(1.005, hurdle, f' hurdle {hurdle:,.0f}', fontsize=8.2, va='center', ha='left',
                color=CORES['cinza_texto'], transform=ax.get_yaxis_transform())
        ax.set_ylim(0, max(acum, hurdle) * 1.22)
        ax.set_xticks(range(len(rotulos)))
        ax.set_xticklabels(rotulos, rotation=26, ha='right', fontsize=7.6 if compacto else 8.6)
        ax.set_ylabel('R$ mil/mês')
        fig.suptitle(f'Candidata {cod} ({cidade_cod}): de onde vem o faturamento projetado',
                     x=0.01, ha='left', fontsize=11.5 if compacto else 13, fontweight='bold',
                     color=CORES['verde_escuro'])
        sufixo = 'mini' if compacto else 'full'
        caminho = _salvar(fig, f'E3_decomposicao_{cod}_{sufixo}.png', params)
        return fig, caminho

    for cod_i in R[idc]:
        f_full, caminho_full = _waterfall(cod_i, compacto=False)
        f_mini, caminho_mini = _waterfall(cod_i, compacto=True)
        figs[f'E3_waterfall_{cod_i}'] = caminho_full
        figs[f'E3_mini_{cod_i}'] = caminho_mini
        plt.close(f_mini)
        if cod_i == R.iloc[0][idc]:
            figs['E3_waterfall'] = caminho_full
            plt.show()
        else:
            plt.close(f_full)

    # --- E4: qualidade do ajuste (treino x teste) ------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
    m = ctx['metricas']
    axes[0].scatter(ctx['y_tr'], ctx['pred_tr'], s=32, color=CORES['verde_claro'],
                    edgecolor=CORES['verde'], linewidth=0.6, label=f"Treino (R2={m['r2_treino']:.2f})", zorder=3)
    axes[0].scatter(ctx['y_te'], ctx['pred_te'], s=42, color=CORES['verde_escuro'],
                    label=f"Teste (R2={m['r2_teste']:.2f})", zorder=4)
    lims = [min(ctx['y_tr'].min(), ctx['y_te'].min()) * 0.9, max(ctx['y_tr'].max(), ctx['y_te'].max()) * 1.05]
    axes[0].plot(lims, lims, ls='--', color=CORES['cinza_texto'], lw=1.2, zorder=2)
    axes[0].set_xlabel('Faturamento observado (R$ mil)'); axes[0].set_ylabel('Faturamento previsto (R$ mil)')
    axes[0].set_title('Previsto x observado'); axes[0].legend(fontsize=9)
    axes[1].scatter(ctx['pred_te'], ctx['resid_te'], s=42, color=CORES['verde_medio'], zorder=3)
    axes[1].axhline(0, color=CORES['verde_escuro'], lw=1.2)
    axes[1].axhline(m['rmse_teste'], color=CORES['cinza_medio'], ls=':', lw=1)
    axes[1].axhline(-m['rmse_teste'], color=CORES['cinza_medio'], ls=':', lw=1)
    axes[1].set_xlabel('Previsto (R$ mil)'); axes[1].set_ylabel('Resíduo (R$ mil)')
    axes[1].set_title(f"Resíduos no teste  |  RMSE = R$ {m['rmse_teste']:,.1f} mil ({m['erro_relativo_%']:,.1f}% da média)")
    fig.suptitle('O modelo generaliza: o erro fora da amostra fica próximo ao erro de treino',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    fig.tight_layout(rect=[0, 0, 1, 0.94])
    figs['E4_ajuste'] = _salvar(fig, 'E4_qualidade_ajuste.png', params); plt.show()

    # --- E5: tornado de sensibilidade da candidata prioritária -----------------------
    cod = R.iloc[0][idc]
    s_top = sens[(sens['Candidata'] == cod) & (sens['No_modelo'] == 'Sim')]
    fig, ax = plt.subplots(figsize=(11, 4.8))
    if len(s_top):
        piv = s_top.pivot_table(index='Alavanca', columns='Delta_%', values='Impacto_R$mil')
        ordem_alav = piv.abs().max(axis=1).sort_values().index
        piv = piv.loc[ordem_alav]
        yy = np.arange(len(piv))
        # desenha do maior para o menor delta, para que nenhum nível fique encoberto
        for dcol in sorted(piv.columns, key=lambda c: -abs(c)):
            alpha = 0.40 + 0.60 * abs(dcol) / max(abs(np.array(piv.columns)))
            cor = CORES['terracota'] if dcol < 0 else CORES['verde']
            ax.barh(yy, piv[dcol], height=0.5, color=cor, alpha=alpha, zorder=3,
                    label=f'{dcol:+,.0f}%')
        for i in yy:
            v_max, v_min = piv.iloc[i].max(), piv.iloc[i].min()
            ax.text(v_max + 0.25, i, f'{v_max:+,.1f}', va='center', fontsize=8.4,
                    color=CORES['verde_escuro'], fontweight='bold')
            ax.text(v_min - 0.25, i, f'{v_min:+,.1f}', va='center', ha='right', fontsize=8.4,
                    color=CORES['terracota'], fontweight='bold')
        ax.set_yticks(yy)
        ax.set_yticklabels([_dicionario_de(i)['Rotulo'] for i in piv.index], fontweight='bold')
        ax.set_ylim(-0.6, len(piv) - 0.4)
        ax.axvline(0, color=CORES['cinza_texto'], lw=1)
        ax.set_xlabel('Variação do faturamento projetado (R$ mil/mês)')
        lim_x = max(piv.abs().max().max() * 1.35, 1)
        ax.set_xlim(-lim_x, lim_x)
        h, l = ax.get_legend_handles_labels()
        vistos = dict(sorted(zip(l, h), key=lambda kv: float(kv[0].replace('%', ''))))
        ax.legend(vistos.values(), vistos.keys(), ncol=len(vistos), fontsize=8.5,
                  loc='upper center', bbox_to_anchor=(0.5, -0.16), title='Variação aplicada à alavanca',
                  title_fontsize=8.5)
    else:
        ax.text(0.5, 0.5, 'Nenhuma alavanca gerencial foi selecionada pelo modelo campeão',
                ha='center', va='center', fontsize=11, color=CORES['terracota'])
        ax.axis('off')
    fig.suptitle(f'Sensibilidade da candidata {cod}: quanto cada alavanca gerencial move o resultado',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E5_tornado'] = _salvar(fig, 'E5_sensibilidade.png', params); plt.show()

    # --- E6: risco x retorno x valor -------------------------------------------------
    fig, ax = plt.subplots(figsize=(10.5, 5.4))
    tam = np.clip(np.abs(R['VPL_R$']) / max(1.0, np.abs(R['VPL_R$']).max()) * 1500, 150, 1600).values
    for k, (_, r) in enumerate(R.iterrows()):
        ax.scatter(r['PROB_SUPERAR_HURDLE'] * 100, r['PREVISAO_FATURAMENTO'],
                   s=tam[k], color=CORES_DECISAO[r['RECOMENDACAO']],
                   alpha=0.75, edgecolor='white', linewidth=1.6, zorder=3)
        ax.annotate(f"{r[idc]}", (r['PROB_SUPERAR_HURDLE'] * 100, r['PREVISAO_FATURAMENTO']),
                    ha='center', va='center', fontsize=10, fontweight='bold', color='white', zorder=4)
        ax.annotate(f"{r['cidade']}\nVPL R$ {r['VPL_R$']/1e6:,.2f} mi",
                    (r['PROB_SUPERAR_HURDLE'] * 100, r['PREVISAO_FATURAMENTO']),
                    xytext=(16, -22), textcoords='offset points', fontsize=8.4, color=CORES['cinza_texto'])
    ax.axhline(hurdle, color=CORES['verde_escuro'], ls='--', lw=1.4)
    ax.axvline(params['prob_minima_aprovacao'] * 100, color=CORES['verde_escuro'], ls=':', lw=1.4)
    ax.text(params['prob_minima_aprovacao'] * 100 + 1, ax.get_ylim()[0],
            f"confiança mínima {params['prob_minima_aprovacao']*100:,.0f}%", fontsize=8.4, rotation=90, va='bottom')
    ax.set_xlabel('Probabilidade de superar o hurdle (%)')
    ax.set_ylabel('Faturamento projetado (R$ mil/mês)')
    ax.set_xlim(-4, 108)
    fig.suptitle('Quadrante de decisão: retorno projetado x confiança estatística (bolha = VPL em 5 anos)',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E6_risco_retorno'] = _salvar(fig, 'E6_risco_retorno.png', params); plt.show()

    # --- E7: caso economico ----------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
    d = R.sort_values('SCORE_PRIORIDADE', ascending=False)
    cores = [CORES_DECISAO[v] for v in d['RECOMENDACAO']]
    axes[0].bar(d[idc], d['CAPEX_R$'] / 1e3, color=CORES['verde_claro'], zorder=3, label='CAPEX')
    axes[0].bar(d[idc], d['EBITDA_MENSAL_R$'] / 1e3, color=CORES['verde_escuro'], zorder=4, label='EBITDA/mes')
    axes[0].set_title('CAPEX x EBITDA mensal (R$ mil)'); axes[0].legend(fontsize=8.5)
    pb = d['PAYBACK_MESES'].replace(np.inf, np.nan)
    axes[1].bar(d[idc], pb.fillna(0), color=cores, zorder=3)
    axes[1].axhline(params['payback_maximo_meses'], color=CORES['verde_escuro'], ls='--', lw=1.4)
    axes[1].set_title(f"Payback (meses)  |  limite = {params['payback_maximo_meses']}")
    for i, v in enumerate(pb.fillna(0)):
        axes[1].text(i, v + 0.6, 'n/a' if np.isnan(pb.iloc[i]) else f'{v:,.0f}', ha='center', fontsize=8.6)
    axes[2].bar(d[idc], d['VPL_R$'] / 1e6, color=cores, zorder=3)
    axes[2].axhline(0, color=CORES['cinza_texto'], lw=1)
    axes[2].set_title('VPL em 5 anos (R$ milhoes)')
    for i, v in enumerate(d['VPL_R$'] / 1e6):
        axes[2].text(i, v + (0.05 if v >= 0 else -0.12), f'{v:,.2f}', ha='center', fontsize=8.6)
    for a in axes:
        a.set_xlabel('Candidata')
    fig.suptitle('Caso econômico por unidade sob as premissas do comitê [#REF]',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    fig.tight_layout(rect=[0, 0, 1, 0.92])
    figs['E7_economia'] = _salvar(fig, 'E7_caso_economico.png', params); plt.show()

    # --- E8: aderencia ao dominio de treino (mapa de calor de z-scores) --------------
    fig, ax = plt.subplots(figsize=(11, 0.62 * len(zscores) + 3.0))
    z = zscores.copy()
    z.columns = [_dicionario_de(c)['Rotulo'] for c in z.columns]
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
        'verde_div', [CORES['terracota'], '#FFFFFF', CORES['verde']])
    sns.heatmap(z, annot=True, fmt='+.1f', cmap=cmap, center=0, vmin=-3, vmax=3,
                linewidths=1.2, linecolor='white', ax=ax,
                cbar_kws={'label': 'Desvios-padrão em relação à base histórica'})
    ax.set_ylabel('Candidata'); ax.set_xlabel('')
    plt.setp(ax.get_xticklabels(), rotation=22, ha='right')
    fig.suptitle(f"Aderência ao domínio de treino: |z| acima de {params['limite_z_extrapolacao']} indica extrapolação",
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E8_extrapolacao'] = _salvar(fig, 'E8_aderencia_dominio.png', params); plt.show()

    # --- E9: candidatas contra a distribuicao da rede --------------------------------
    fig, ax = plt.subplots(figsize=(11, 4.8))
    y_all = ctx['df_num'][params['target']]
    sns.histplot(y_all, bins=26, color=CORES['verde_palido'], edgecolor=CORES['verde_claro'],
                 ax=ax, stat='density', zorder=2)
    sns.kdeplot(y_all, color=CORES['verde'], lw=2, ax=ax, zorder=3)
    ax.axvline(hurdle, color=CORES['verde_escuro'], ls='--', lw=1.6, zorder=4)
    topo = ax.get_ylim()[1]
    for k, (_, r) in enumerate(R.iterrows()):
        ax.axvline(r['PREVISAO_FATURAMENTO'], color=CORES_DECISAO[r['RECOMENDACAO']], lw=2, zorder=5)
        ax.text(r['PREVISAO_FATURAMENTO'], topo * (0.94 - 0.11 * k), f" {r[idc]}",
                color=CORES_DECISAO[r['RECOMENDACAO']], fontweight='bold', fontsize=10)
    pct = [100 * (y_all < v).mean() for v in R['PREVISAO_FATURAMENTO']]
    ax.set_xlabel('Faturamento médio mensal (R$ mil)'); ax.set_ylabel('Densidade')
    fig.suptitle(f'Onde as candidatas se posicionam frente às {len(y_all)} lojas em operação',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E9_distribuicao'] = _salvar(fig, 'E9_posicionamento.png', params); plt.show()
    R['PERCENTIL_NA_REDE'] = pct

    # --- E10: contribuicao comparada de todas as candidatas --------------------------
    cc = contrib.drop(columns=['BASE_MEDIA_DA_REDE'])
    cc = cc.loc[:, cc.abs().sum() > 1e-6]
    ordem_col = cc.abs().sum().sort_values(ascending=False).index
    cc = cc[ordem_col]
    fig, ax = plt.subplots(figsize=(11.5, 5.2))
    base = np.zeros(len(cc)); base_neg = np.zeros(len(cc))
    paleta = sns.blend_palette([CORES['verde_escuro'], CORES['verde_medio'], CORES['verde_claro'],
                                CORES['ambar'], CORES['terracota']], n_colors=max(2, len(cc.columns)))
    for j, col in enumerate(cc.columns):
        v = cc[col].values
        pos = np.where(v > 0, v, 0); neg = np.where(v < 0, v, 0)
        ax.bar(cc.index, pos, bottom=base, color=paleta[j], label=_dicionario_de(col)['Rotulo'], zorder=3)
        ax.bar(cc.index, neg, bottom=base_neg, color=paleta[j], zorder=3)
        base += pos; base_neg += neg
    ax.axhline(0, color=CORES['cinza_texto'], lw=1)
    ax.set_ylabel('Contribuição ao faturamento x loja média (R$ mil/mês)')
    ax.set_xlabel('Candidata')
    ax.legend(ncol=2, fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.14))
    fig.suptitle('Anatomia da previsão: o que puxa cada candidata para cima e para baixo',
                 x=0.01, ha='left', fontsize=13, fontweight='bold', color=CORES['verde_escuro'])
    figs['E10_contribuicoes'] = _salvar(fig, 'E10_contribuicoes.png', params); plt.show()

    return figs

# =====================================================================================
# 8. RELATÓRIO EXECUTIVO EM PDF — layout de consultoria estratégica, paleta verde
# =====================================================================================
from fpdf import FPDF
from datetime import datetime

_SUBSTITUICOES = {'—': '-', '–': '-', '“': '"', '”': '"', '‘': "'", '’': "'", '…': '...',
                  '≥': '>=', '≤': '<=', '→': '->', '•': '-', '±': '+/-', ' ': ' ',
                  ' ': ' ', '₂': '2', 'º': 'o.', 'ª': 'a.'}

def _t(texto):
    """Torna o texto seguro para o encoder latin-1 usado pelas fontes core do FPDF."""
    s = str(texto)
    for a, b in _SUBSTITUICOES.items():
        s = s.replace(a, b)
    return s.encode('latin-1', 'replace').decode('latin-1')

def _rgb(hexa):
    hexa = hexa.lstrip('#')
    return tuple(int(hexa[i:i + 2], 16) for i in (0, 2, 4))

class RelatorioVerde(FPDF):
    """Documento A4 com cabeçalho, rodapé, tipografia e grid padronizados."""

    def __init__(self, params=P):
        super().__init__(orientation='P', unit='mm', format='A4')
        self.params = params
        self.c = {k: _rgb(v) for k, v in params['marca'].items()}
        self.set_auto_page_break(auto=True, margin=22)
        self.set_margins(16, 22, 16)
        self.secao_atual = ''

    # ---------------- estrutura da página --------------------------------------------
    def header(self):
        if self.page_no() <= 1:
            return
        self.set_fill_color(*self.c['verde_escuro'])
        self.rect(0, 0, 210, 13, 'F')
        self.set_fill_color(*self.c['verde_neon'])
        self.rect(0, 13, 210, 1.1, 'F')
        self.set_xy(16, 4.2)
        self.set_font('Arial', 'B', 8.5)
        self.set_text_color(255, 255, 255)
        self.cell(110, 5, _t(self.params['titulo_relatorio']), 0, 0, 'L')
        self.set_font('Arial', '', 8)
        self.cell(68, 5, _t(self.secao_atual), 0, 0, 'R')
        self.set_y(22)
        self.set_text_color(*self.c['cinza_texto'])

    def footer(self):
        if self.page_no() <= 1:
            return
        self.set_y(-16)
        self.set_draw_color(*self.c['cinza_medio'])
        self.set_line_width(0.2)
        self.line(16, self.get_y(), 194, self.get_y())
        self.set_y(-13)
        self.set_font('Arial', 'I', 7)
        self.set_text_color(*self.c['cinza_texto'])
        self.cell(150, 5, _t(self.rodape_fonte), 0, 0, 'L')
        self.set_font('Arial', 'B', 8)
        self.set_text_color(*self.c['verde'])
        self.cell(28, 5, _t(f'{self.page_no() - 1}'), 0, 0, 'R')

    rodape_fonte = 'Fonte: base histórica da rede (franquia_lojas.csv) e modelo LASSO validado fora da amostra'

    # ---------------- blocos de conteúdo ---------------------------------------------
    def pagina_capa(self):
        self.add_page()
        self.set_fill_color(*self.c['verde_escuro'])
        self.rect(0, 0, 210, 150, 'F')
        self.set_fill_color(*self.c['verde_neon'])
        self.rect(0, 150, 210, 2.4, 'F')
        self.set_fill_color(*self.c['verde'])
        self.rect(16, 30, 26, 2.6, 'F')
        self.set_xy(16, 42)
        self.set_text_color(255, 255, 255)
        self.set_font('Arial', 'B', 26)
        self.multi_cell(178, 12, _t(self.params['titulo_relatorio']), align='L')
        self.ln(2)
        self.set_x(16)
        self.set_font('Arial', '', 13)
        self.set_text_color(*self.c['verde_claro'])
        self.multi_cell(160, 7, _t(self.params['subtitulo_relatorio']), align='L')
        self.set_xy(16, 106)
        self.set_font('Arial', 'B', 9.5)
        self.set_text_color(255, 255, 255)
        self.cell(0, 5, _t('DOCUMENTO CONFIDENCIAL — PREPARADO PARA O COMITÊ DE INVESTIMENTOS'), 0, 1)
        self.set_x(16)
        self.set_font('Arial', '', 9.5)
        self.set_text_color(*self.c['verde_claro'])
        for linha in [self.params['cliente'], f"Equipe: {self.params['autores']}",
                      f"Data de emissão: {datetime.now().strftime('%d/%m/%Y')}"]:
            self.set_x(16)
            self.cell(0, 6, _t(linha), 0, 1)
        self.set_xy(16, 166)
        self.set_text_color(*self.c['cinza_texto'])
        self.set_font('Arial', '', 9.6)
        self.multi_cell(178, 5.6, _t(
            'Este relatório consolida a modelagem estatística desenvolvida sobre a base histórica da rede e '
            'traduz o resultado em uma recomendação de alocação de capital para as praças candidatas. Todas as '
            'regras de decisão — hurdle de faturamento, nível de confiança, critérios financeiros e pesos de '
            'priorização — são paramétricas e estão declaradas no capítulo de premissas, de modo que o comitê '
            'possa reprocessar o documento inteiro sob outro conjunto de hipóteses.'), align='J')

    def titulo_secao(self, numero, titulo, kicker=''):
        self.secao_atual = f'{numero}. {titulo}'
        self.add_page()
        self.set_text_color(*self.c['verde_medio'])
        self.set_font('Arial', 'B', 8)
        self.cell(0, 4, _t(kicker.upper()), 0, 1)
        self.set_text_color(*self.c['verde_escuro'])
        self.set_font('Arial', 'B', 16)
        self.multi_cell(178, 8, _t(f'{numero}. {titulo}'), align='L')
        self.set_draw_color(*self.c['verde_neon'])
        self.set_line_width(1.1)
        y = self.get_y() + 1.5
        self.line(16, y, 46, y)
        self.ln(6)
        self.set_text_color(*self.c['cinza_texto'])

    def subtitulo(self, texto):
        self.ln(2)
        self.set_font('Arial', 'B', 11)
        self.set_text_color(*self.c['verde'])
        self.multi_cell(178, 6, _t(texto), align='L')
        self.set_text_color(*self.c['cinza_texto'])
        self.ln(0.5)

    def paragrafo(self, texto, tamanho=9.6):
        self.set_font('Arial', '', tamanho)
        self.set_text_color(*self.c['cinza_texto'])
        self.multi_cell(178, 5.2, _t(texto), align='J')
        self.ln(1.6)

    def bullets(self, itens, tamanho=9.6):
        self.set_font('Arial', '', tamanho)
        for it in itens:
            y0 = self.get_y()
            self.set_fill_color(*self.c['verde_neon'])
            self.rect(17.5, y0 + 1.9, 1.8, 1.8, 'F')
            self.set_x(22)
            self.multi_cell(172, 5.1, _t(it), align='J')
            self.ln(0.8)
        self.ln(1.2)

    def destaque(self, titulo, texto):
        """Caixa de leitura rápida — o 'so what' da página."""
        self.ln(1)
        x0, y0 = 16, self.get_y()
        largura_util = 166
        self.set_font('Arial', '', 9.3)
        n_linhas = max(1, int(self.get_string_width(_t(texto)) / largura_util) + 1)
        alt = 9.5 + 4.6 * n_linhas
        if y0 + alt > 262:
            self.add_page()
            y0 = self.get_y()
        self.set_fill_color(*self.c['verde_palido'])
        self.rect(x0, y0, 178, alt, 'F')
        self.set_fill_color(*self.c['verde'])
        self.rect(x0, y0, 2.2, alt, 'F')
        self.set_xy(x0 + 6, y0 + 2.6)
        self.set_font('Arial', 'B', 9.6)
        self.set_text_color(*self.c['verde_escuro'])
        self.cell(0, 4.6, _t(titulo.upper()), 0, 1)
        self.set_x(x0 + 6)
        self.set_font('Arial', '', 9.3)
        self.set_text_color(*self.c['cinza_texto'])
        self.multi_cell(largura_util, 4.6, _t(texto), align='J')
        self.set_y(max(self.get_y(), y0 + alt) + 3)

    def kpis(self, itens):
        """Faixa horizontal de indicadores-chave (até 4 blocos)."""
        larg = 178 / len(itens)
        y0 = self.get_y()
        for i, (rot, val, sub) in enumerate(itens):
            x = 16 + i * larg
            self.set_fill_color(*self.c['verde_escuro'])
            self.rect(x, y0, larg - 2.5, 21, 'F')
            self.set_fill_color(*self.c['verde_neon'])
            self.rect(x, y0, larg - 2.5, 1.1, 'F')
            self.set_xy(x + 4, y0 + 3.2)
            self.set_font('Arial', '', 7.2)
            self.set_text_color(*self.c['verde_claro'])
            self.cell(larg - 8, 3.6, _t(rot.upper()), 0, 2)
            self.set_font('Arial', 'B', 13.5)
            self.set_text_color(255, 255, 255)
            self.cell(larg - 8, 8, _t(val), 0, 2)
            self.set_font('Arial', '', 7)
            self.set_text_color(*self.c['verde_claro'])
            self.cell(larg - 8, 3.6, _t(sub), 0, 2)
        self.set_y(y0 + 25)
        self.set_text_color(*self.c['cinza_texto'])

    def _cabecalho_tabela(self, colunas, larguras, altura, tamanho):
        self.set_font('Arial', 'B', tamanho + 0.2)
        self.set_fill_color(*self.c['verde_escuro'])
        self.set_text_color(255, 255, 255)
        self.set_draw_color(255, 255, 255)
        self.set_line_width(0.25)
        for c, w in zip(colunas, larguras):
            self.cell(w, altura + 1.4, _t(c), 1, 0, 'C', 1)
        self.ln()

    def tabela(self, colunas, larguras, linhas, alturas=6.0, tamanho=7.6, cores_linha=None,
               alinhamentos=None):
        alinhamentos = alinhamentos or ['C'] * len(colunas)
        self._cabecalho_tabela(colunas, larguras, alturas, tamanho)
        self.set_font('Arial', '', tamanho)
        for k, linha in enumerate(linhas):
            if self.get_y() > 252:
                self.add_page()
                self._cabecalho_tabela(colunas, larguras, alturas, tamanho)
                self.set_font('Arial', '', tamanho)
            self.set_fill_color(*(self.c['verde_palido'] if k % 2 == 0 else (255, 255, 255)))
            self.set_text_color(*self.c['cinza_texto'])
            for j, (v, w) in enumerate(zip(linha, larguras)):
                if cores_linha and j == cores_linha[0]:
                    self.set_text_color(*_rgb(cores_linha[1][k]))
                    self.set_font('Arial', 'B', tamanho)
                self.cell(w, alturas, _t(v), 1, 0, alinhamentos[j], 1)
                self.set_text_color(*self.c['cinza_texto'])
                self.set_font('Arial', '', tamanho)
            self.ln()
        self.ln(2.5)

    def exibir(self, numero, titulo, caminho, largura=178, altura_estimada=None,
               fonte='modelo LASSO estimado sobre a base histórica tratada'):
        alt_img = ALTURA_EXHIBIT.get(caminho, altura_estimada or 95)
        if self.get_y() + alt_img + 22 > 264:
            self.add_page()
        self.set_font('Arial', 'B', 8)
        self.set_text_color(*self.c['verde_medio'])
        self.cell(0, 4.4, _t(f'EXHIBIT {numero}'), 0, 1)
        self.set_font('Arial', 'B', 10)
        self.set_text_color(*self.c['verde_escuro'])
        self.multi_cell(178, 5, _t(titulo), align='L')
        self.ln(1.2)
        if os.path.exists(caminho):
            self.image(caminho, x=16, w=largura)
        self.ln(1.5)
        self.set_font('Arial', 'I', 7)
        self.set_text_color(*self.c['cinza_texto'])
        self.multi_cell(178, 3.8, _t(f'Fonte: {fonte}'), align='L')
        self.ln(3)

# =====================================================================================
# 9. MONTAGEM DO RELATÓRIO
# =====================================================================================
def montar_relatorio(R, ctx, sens, esforco, imput, figs, contrib, params=P):
    idc = params['coluna_id_candidata']
    m = ctx['metricas']
    hurdle = params['hurdle_faturamento']
    aprovadas = R[R['RECOMENDACAO'] == 'ABRIR']
    ressalvas = R[R['RECOMENDACAO'] == 'ABRIR COM RESSALVAS']
    reprovadas = R[R['RECOMENDACAO'] == 'NÃO ABRIR']
    capex_aprovado = aprovadas['CAPEX_R$'].sum()
    vpl_aprovado = aprovadas['VPL_R$'].sum()
    cores_dec = [params['marca']['verde'] if v == 'ABRIR'
                 else params['marca']['ambar'] if v == 'ABRIR COM RESSALVAS'
                 else params['marca']['terracota'] for v in R['RECOMENDACAO']]

    pdf = RelatorioVerde(params)
    pdf.pagina_capa()

    # ------------------------------- 1. Sumário executivo ----------------------------
    pdf.titulo_secao(1, 'Sumário executivo', 'Recomendação ao comitê')
    pdf.kpis([
        ('Praças avaliadas', f'{len(R)}', 'candidatas no pipeline'),
        ('Aprovadas', f'{len(aprovadas)}', f'de {len(R)} candidatas'),
        ('Capital recomendado', f'R$ {capex_aprovado/1e6:,.2f} mi', 'CAPEX das aprovadas [#REF]'),
        ('VPL em 5 anos', f'R$ {vpl_aprovado/1e6:,.2f} mi', 'das unidades aprovadas [#REF]'),
    ])
    lista_ap = ', '.join(f"{r[idc]} ({r['cidade']})" for _, r in aprovadas.iterrows()) or 'nenhuma'
    lista_rs = ', '.join(f"{r[idc]} ({r['cidade']})" for _, r in ressalvas.iterrows()) or 'nenhuma'
    lista_rp = ', '.join(f"{r[idc]} ({r['cidade']})" for _, r in reprovadas.iterrows()) or 'nenhuma'
    pdf.destaque('Recomendação',
                 f'Aprovar a abertura de {len(aprovadas)} unidade(s) — {lista_ap} — que superam o hurdle de '
                 f'R$ {hurdle:,.0f} mil/mês com probabilidade de no mínimo '
                 f'{params["prob_minima_aprovacao"]*100:,.0f}%. Reprovar {len(reprovadas)} praça(s): {lista_rp}. '
                 f'Manter {len(ressalvas)} praça(s) em observação — {lista_rs} — condicionadas a plano de '
                 f'mitigação e a nova avaliação em 90 dias.')
    pdf.paragrafo(
        f'O modelo explica {m["r2_teste"]*100:,.1f}% da variação do faturamento em dados que não participaram '
        f'do treino, com erro típico de R$ {m["rmse_teste"]:,.1f} mil/mês — {m["erro_relativo_%"]:,.1f}% do '
        f'faturamento médio da rede. Essa precisão é suficiente para separar praças boas de ruins, mas não '
        f'para tratar a previsão como um número exato. Por isso cada recomendação é apresentada com intervalo '
        f'de {params["nivel_confianca"]*100:,.0f}% de confiança e com a probabilidade explícita de a unidade '
        f'superar o hurdle, e não apenas com o valor pontual.')
    pdf.subtitulo('Decisão por praça candidata')
    pdf.tabela(
        ['#', 'Praça', 'Cidade', 'Previsão', f'IC {params["nivel_confianca"]*100:,.0f}%',
         'P(> hurdle)', 'Payback', 'Score', 'Veredito'],
        [8, 14, 30, 22, 32, 20, 17, 14, 21],
        [[f"{r['RANKING']}", str(r[idc]), r['cidade'], f"{r['PREVISAO_FATURAMENTO']:,.1f}",
          f"{r['IC_INF']:,.1f} a {r['IC_SUP']:,.1f}", f"{r['PROB_SUPERAR_HURDLE']*100:,.0f}%",
          ('n/a' if not np.isfinite(r['PAYBACK_MESES']) else f"{r['PAYBACK_MESES']:,.0f}m"),
          f"{r['SCORE_PRIORIDADE']:,.0f}",
          r['RECOMENDACAO'].replace('ABRIR COM RESSALVAS', 'RESSALVAS').replace('NÃO ABRIR', 'NÃO ABRIR')]
         for _, r in R.iterrows()],
        cores_linha=(8, cores_dec),
        alinhamentos=['C', 'C', 'L', 'R', 'C', 'C', 'C', 'C', 'C'])
    pdf.paragrafo('Faturamento em R$ mil por mês. O score de prioridade combina retorno projetado, confiança '
                  'estatística, resultado financeiro e aderência do perfil ao domínio do modelo, com os pesos '
                  'declarados no capítulo 10.', 8.2)
    pdf.exibir('1', 'Faturamento projetado por praça candidata, com intervalo de confiança e hurdle de aprovação',
               figs['E1_ranking'])

    # ------------------------------- 2. Metodologia -----------------------------------
    pdf.titulo_secao(2, 'Abordagem analítica', 'Como chegamos ao modelo')
    pdf.paragrafo(
        'A construção do modelo seguiu quatro estágios sucessivos de eliminação, cada um endurecendo o '
        'critério de aceitação de uma variável. O objetivo não foi maximizar o ajuste estatístico, e sim obter '
        'um modelo utilizável ANTES da abertura da loja — o que exclui, por construção, qualquer informação '
        'que só se conhece depois de operar.')
    pdf.bullets([
        'Estágio 1 — Tratamento: remoção de duplicatas, corte de outliers pelo intervalo interquartil (IQR), '
        'exclusão de registros incompletos e criação de variáveis dummy para cidade e região.',
        'Estágio 2 — Exploração: quatro famílias de modelos testadas em paralelo (busca aleatória com filtro '
        'de p-valor, LOOCV, forward stepwise e LASSO), todas com separação treino/teste de 70/30.',
        'Estágio 3 — Veto de variáveis: eliminação das variáveis contaminadas por vazamento temporal ou '
        'contábil, detalhadas no capítulo 3.',
        'Estágio 4 — Modelo final: 1.000 combinações aleatórias de variáveis, refino iterativo por p-valor '
        '(mantendo apenas variáveis significativas a 95%), regularização LASSO com alpha testado em cinco '
        'níveis e escolha do campeão pelo menor erro na amostra de teste.'])
    pdf.destaque('Por que LASSO com refino de p-valor',
                 'O refino por p-valor garante interpretabilidade: toda variável que permanece no modelo é '
                 'estatisticamente significativa a 95%. A regularização LASSO encolhe coeficientes instáveis e '
                 'reduz o risco de sobreajuste em uma base de 181 observações úteis. A validação final ocorre '
                 'em dados que nunca participaram do processo de seleção, o que torna o erro reportado uma '
                 'estimativa honesta do erro em produção.')
    pdf.subtitulo('Especificação do modelo campeão')
    pdf.tabela(['Parâmetro', 'Valor', 'Leitura'],
               [50, 40, 88],
               [['Família', 'Regressão linear com LASSO', 'Coeficientes interpretáveis em R$'],
                ['Alpha (regularização)', f"{m['alpha_lasso']}", 'Escolhido por desempenho fora da amostra'],
                ['Observações', f"{m['n_total']} ({m['n_treino']} treino / {m['n_teste']} teste)",
                 'Split 70/30 com semente fixa (random_state = 42)'],
                ['Variáveis candidatas', f"{m['n_variaveis_pool']}", 'Pool após o veto e a criação de dummies'],
                ['Variáveis no modelo', f"{m['n_variaveis_modelo']}", 'Todas significativas a 95%'],
                ['Variável-alvo', 'faturamento_medio_mensal', 'Faturamento em R$ mil por mês']],
               alinhamentos=['L', 'L', 'L'], tamanho=8)

    # ------------------------------- 3. Dicionário de variáveis ----------------------
    pdf.titulo_secao(3, 'Dicionário de variáveis', 'O que cada dado significa')
    pdf.paragrafo('A tabela documenta todas as variáveis da base: o que medem, em que unidade, se já são '
                  'conhecidas no momento da decisão de investimento e qual tratamento receberam na modelagem.')
    pdf.tabela(['Variável', 'Descrição', 'Unidade', 'Pré-abertura', 'Status'],
               [42, 78, 24, 18, 16],
               [[r['Variavel'], r['Descricao'][:78], r['Unidade'], r['Disponivel_pre_abertura'],
                 r['Status_no_modelo']] for _, r in DICIONARIO_VARIAVEIS.iterrows()],
               alturas=8.4, tamanho=6.6, alinhamentos=['L', 'L', 'C', 'C', 'C'])
    pdf.subtitulo('Variáveis vetadas e o motivo do veto')
    pdf.bullets([f"{r['Variavel']} — {r['Justificativa']}."
                 for _, r in DICIONARIO_VARIAVEIS[DICIONARIO_VARIAVEIS['Status_no_modelo'] == 'VETADA'].iterrows()],
                tamanho=9.2)
    pdf.destaque('Consequência prática do veto',
                 'Excluir ticket médio, número de transações, meses de operação e satisfação do cliente reduz '
                 'o R² aparente, mas é justamente o que torna o modelo utilizável. Essas variáveis só existem '
                 'depois que a loja está aberta e, no caso do ticket e das transações, são componentes '
                 'contábeis do próprio faturamento — mantê-las produziria um modelo com ajuste quase perfeito '
                 'e utilidade zero para decidir onde abrir.')

    # ------------------------------- 4. Ficha técnica do modelo -----------------------
    pdf.titulo_secao(4, 'Ficha técnica do modelo', 'Dados, coeficientes e diagnóstico')
    pdf.kpis([
        ('R² fora da amostra', f"{m['r2_teste']*100:,.1f}%", 'poder explicativo real'),
        ('RMSE no teste', f"R$ {m['rmse_teste']:,.1f} mil", f"{m['erro_relativo_%']:,.1f}% da média"),
        ('MAPE no teste', f"{m['mape_teste']:,.1f}%", 'erro percentual médio'),
        ('VIF máximo', f"{m['vif_maximo']:,.1f}", 'multicolinearidade controlada'),
    ])
    pdf.subtitulo('Equação estimada: a contribuição de cada variável')
    tc = ctx['tabela_coef']
    pdf.tabela(['Variável', 'Família', 'Impacto por +1 dp', 'Impacto por unidade', 'p-valor', 'VIF', 'Peso'],
               [46, 26, 26, 30, 18, 14, 18],
               [[r['Rotulo'][:34], r['Familia'][:20], f"{r['Coef_padronizado']:+,.2f}",
                 f"{r['Coef_unidade_original']:+,.4f}", f"{r['P_valor']:.4f}",
                 f"{r['VIF']:,.1f}", f"{r['Peso_relativo_%']:,.0f}%"] for _, r in tc.iterrows()],
               tamanho=7.0, alinhamentos=['L', 'L', 'R', 'R', 'R', 'R', 'R'])
    pdf.paragrafo(
        'Impacto por +1 dp é a variação do faturamento, em R$ mil/mês, quando a variável sobe um desvio-padrão '
        'e as demais permanecem constantes; é a métrica correta para comparar variáveis de escalas diferentes. '
        'Impacto por unidade traduz o mesmo efeito na unidade natural da variável — um m², um real de '
        'marketing, mil habitantes. Os p-valores, intervalos e VIF vêm do modelo OLS gêmeo, estimado sobre '
        'exatamente as mesmas variáveis selecionadas pelo LASSO.', 8.2)
    pdf.subtitulo('Diagnóstico estatístico')
    pdf.tabela(['Teste', 'Estatística', 'Interpretação'],
               [46, 30, 102],
               [['R² ajustado (OLS gêmeo)', f"{m['r2_ajustado_ols']:.3f}",
                 'Ajuste penalizado pelo número de variáveis'],
                ['Teste F (p-valor)', f"{m['f_pvalue_ols']:.2e}",
                 'Modelo conjuntamente significativo' if m['f_pvalue_ols'] < 0.05 else 'Modelo não significativo'],
                ['Durbin-Watson', f"{m['durbin_watson']:.2f}",
                 'Resíduos sem autocorrelação relevante' if 1.5 < m['durbin_watson'] < 2.5
                 else 'Indício de autocorrelação nos resíduos'],
                ['Jarque-Bera (p-valor)', f"{m['jarque_bera_p']:.3f}",
                 'Resíduos compatíveis com normalidade' if m['jarque_bera_p'] > 0.05
                 else 'Resíduos não normais: os intervalos são aproximações'],
                ['Breusch-Pagan (p-valor)', f"{m['breusch_pagan_p']:.3f}",
                 'Variância dos resíduos estável' if m['breusch_pagan_p'] > 0.05
                 else 'Heterocedasticidade: erro maior nas lojas de maior porte'],
                ['RMSE treino x teste', f"{m['rmse_treino']:,.1f} / {m['rmse_teste']:,.1f}",
                 'Sem sobreajuste relevante' if m['rmse_teste'] <= 1.35 * m['rmse_treino']
                 else 'Possível sobreajuste: erro cresce fora da amostra']],
               tamanho=7.4, alinhamentos=['L', 'R', 'L'])
    pdf.exibir('2', 'Hierarquia dos direcionadores de faturamento no modelo campeão', figs['E2_drivers'],
               altura_estimada=120)
    pdf.exibir('3', 'Aderência do modelo dentro e fora da amostra de treino', figs['E4_ajuste'],
               altura_estimada=100)

    # ------------------------------- 5. Matriz de decisão -----------------------------
    pdf.titulo_secao(5, 'Matriz de decisão', 'Da previsão à alocação de capital')
    pdf.paragrafo(
        f"A regra vigente é a '{params['regra_decisao']}': uma praça só é aprovada quando a previsão pontual "
        f"supera o hurdle de R$ {hurdle:,.0f} mil/mês E a probabilidade de superá-lo é de ao menos "
        f"{params['prob_minima_aprovacao']*100:,.0f}%. Praças dentro da zona cinzenta de "
        f"±{params['zona_cinzenta_pct']*100:,.0f}% em torno do hurdle são classificadas como 'abrir com "
        f"ressalvas'. Candidatas cujo perfil está fora do domínio observado na base histórica são rebaixadas "
        f"um nível, porque nesses casos o modelo extrapola e o intervalo reportado subestima o risco real.")
    pdf.tabela(['Praça', 'Previsão', 'Gap x hurdle', 'Percentil na rede', 'Aderência', 'Veredito'],
               [16, 22, 30, 30, 40, 40],
               [[str(r[idc]), f"{r['PREVISAO_FATURAMENTO']:,.1f}",
                 f"{r['GAP_VS_HURDLE']:+,.1f} ({r['GAP_VS_HURDLE_%']:+,.0f}%)",
                 f"{r.get('PERCENTIL_NA_REDE', np.nan):,.0f}º percentil",
                 'Dentro do domínio' if not r['FORA_DO_DOMINIO']
                 else f"Extrapola: {_lista_amigavel(r['DETALHE_EXTRAPOLACAO'])[:24]}",
                 r['RECOMENDACAO'].replace('NÃO ABRIR', 'NÃO ABRIR')] for _, r in R.iterrows()],
               tamanho=7.2, alinhamentos=['C', 'R', 'C', 'C', 'L', 'C'],
               cores_linha=(5, cores_dec))
    pdf.paragrafo('O percentil indica em que posição a candidata ficaria se fosse inserida hoje na base de '
                  'lojas em operação — uma leitura de quão ambiciosa é a previsão frente ao que a rede já '
                  'entrega.', 8.2)
    pdf.exibir('4', 'Quadrante de decisão: retorno projetado contra confiança estatística', figs['E6_risco_retorno'],
               altura_estimada=105)
    pdf.exibir('5', 'Posicionamento das candidatas frente à distribuição das lojas em operação',
               figs['E9_distribuicao'], altura_estimada=95)
    pdf.exibir('6', 'Anatomia da previsão: contribuição de cada direcionador, praça a praça',
               figs['E10_contribuicoes'], altura_estimada=110)

    # ------------------------------- 6. One-pagers ------------------------------------
    pdf.titulo_secao(6, 'Análise praça a praça', 'Uma página por candidata')
    pdf.paragrafo('As páginas a seguir consolidam, para cada praça, o resultado projetado, o perfil físico do '
                  'ponto, os direcionadores que explicam a previsão e o próximo passo recomendado. A ordem '
                  'segue o ranking de prioridade de alocação de capital.')
    pdf.exibir('7', f"Decomposição do faturamento projetado da praça prioritária ({R.iloc[0][idc]})",
               figs['E3_waterfall'], altura_estimada=110)
    for _, r in R.iterrows():
        pdf.secao_atual = f"6. Praça {r[idc]} — {r['cidade']}"
        pdf.add_page()
        cor = (params['marca']['verde'] if r['RECOMENDACAO'] == 'ABRIR'
               else params['marca']['ambar'] if r['RECOMENDACAO'] == 'ABRIR COM RESSALVAS'
               else params['marca']['terracota'])
        pdf.set_text_color(*pdf.c['verde_medio'])
        pdf.set_font('Arial', 'B', 8)
        pdf.cell(0, 4, _t(f"PRAÇA CANDIDATA {r[idc]} — PRIORIDADE {r['RANKING']} DE {len(R)}"), 0, 1)
        pdf.set_text_color(*pdf.c['verde_escuro'])
        pdf.set_font('Arial', 'B', 16)
        pdf.cell(0, 8, _t(f"{r['cidade']} ({r['regiao']})"), 0, 1)
        pdf.set_font('Arial', '', 9)
        pdf.set_text_color(*pdf.c['cinza_texto'])
        pdf.cell(0, 5, _t(str(r.get('endereco_referencia', ''))), 0, 1)
        y0 = pdf.get_y() + 2
        pdf.set_fill_color(*_rgb(cor))
        pdf.rect(16, y0, 178, 9, 'F')
        pdf.set_xy(16, y0 + 1.6)
        pdf.set_font('Arial', 'B', 11)
        pdf.set_text_color(255, 255, 255)
        pdf.cell(178, 6, _t(f"VEREDITO: {r['RECOMENDACAO'].replace('NÃO ABRIR', 'NÃO ABRIR')}"), 0, 1, 'C')
        pdf.set_y(y0 + 13)
        pdf.set_text_color(*pdf.c['cinza_texto'])
        pdf.kpis([
            ('Faturamento projetado', f"R$ {r['PREVISAO_FATURAMENTO']:,.1f} mil", f"hurdle: R$ {hurdle:,.0f} mil"),
            (f"Intervalo de {params['nivel_confianca']*100:,.0f}%",
             f"{r['IC_INF']:,.0f} a {r['IC_SUP']:,.0f}", 'R$ mil/mês'),
            ('Prob. de superar', f"{r['PROB_SUPERAR_HURDLE']*100:,.0f}%", 'o hurdle do comitê'),
            ('VPL em 5 anos', f"R$ {r['VPL_R$']/1e6:,.2f} mi", 'sob as premissas [#REF]'),
        ])
        pdf.subtitulo('Perfil do ponto')
        campos = [('area_m2', 'Área (m2)'), ('populacao_1km', 'Pop. 1 km'),
                  ('renda_media_domiciliar', 'Renda (R$)'), ('fluxo_pedestres_dia', 'Fluxo/dia'),
                  ('vagas_estacionamento', 'Vagas'), ('distancia_concorrente_m', 'Concorr. (m)'),
                  ('shopping', 'Shopping'), ('investimento_marketing_mensal', 'Mkt (R$)'),
                  ('n_funcionarios', 'Equipe')]
        campos = [(c, rot) for c, rot in campos if c in R.columns]
        pdf.tabela([rot for _, rot in campos], [178 / len(campos)] * len(campos),
                   [[f"{r[c]:,.0f}" for c, _ in campos]], tamanho=6.8, alturas=6.6,
                   alinhamentos=['C'] * len(campos))
        pdf.subtitulo('Direcionadores desta praça')
        c_r = contrib.loc[r[idc]].drop('BASE_MEDIA_DA_REDE')
        c_r = c_r[c_r.abs() > 1e-6].sort_values(key=np.abs, ascending=False).head(6)
        pdf.tabela(['Direcionador', 'Contribuição x loja média (R$ mil/mês)', 'Efeito'],
                   [80, 62, 36],
                   [[_dicionario_de(i)['Rotulo'][:52], f"{v:+,.2f}",
                     'Favorável' if v > 0 else 'Desfavorável'] for i, v in c_r.items()],
                   tamanho=7.2, alinhamentos=['L', 'R', 'C'])
        e_r = esforco[esforco['Candidata'] == r[idc]]
        acao = ('Prosseguir com a due diligence imobiliária e travar o cronograma de abertura.'
                if r['RECOMENDACAO'] == 'ABRIR' else
                'Não alocar capital nesta praça no ciclo atual; reavaliar apenas diante de mudança material '
                'nas condições do ponto.'
                if r['RECOMENDACAO'] == 'NÃO ABRIR' else
                'Condicionar a aprovação a um plano de mitigação com metas de marketing e revisão em 90 dias.')
        detalhe_esforco = ''
        viaveis = e_r[e_r['Viavel'] == 'Sim']
        if r['RECOMENDACAO'] != 'ABRIR' and len(viaveis):
            v0 = viaveis.iloc[0]
            detalhe_esforco = (f" Para fechar o gap de R$ {v0['Gap_R$mil']:,.1f} mil seria necessário "
                               f"{v0['Variacao_necessaria']} em {v0['Alavanca']}.")
        pdf.destaque('Decisão e próximo passo', f"{r['JUSTIFICATIVA']}. {acao}{detalhe_esforco}")
        chave_fig = f"E3_mini_{r[idc]}"
        if chave_fig in figs and pdf.get_y() + ALTURA_EXHIBIT.get(figs[chave_fig], 70) < 262:
            pdf.set_font('Arial', 'B', 8)
            pdf.set_text_color(*pdf.c['verde_medio'])
            pdf.cell(0, 4.4, _t('DECOMPOSIÇÃO DA PREVISÃO'), 0, 1)
            pdf.image(figs[chave_fig], x=16, w=178)

    # ------------------------------- 7. Sensibilidade ---------------------------------
    pdf.titulo_secao(7, 'Alavancas e sensibilidade', 'O que ainda podemos mudar')
    pdf.paragrafo(
        'Escolhido o ponto, quase todas as variáveis do modelo passam a ser fixas: população, renda, fluxo de '
        'pedestres e praça são características do imóvel, não decisões de gestão. As alavancas simuladas a '
        'seguir são as poucas variáveis que permanecem sob controle da rede, e a simulação mostra quanto cada '
        'uma de fato move o faturamento projetado.')
    pdf.exibir('8', f"Sensibilidade do faturamento projetado da praça {R.iloc[0][idc]} às alavancas gerenciais",
               figs['E5_tornado'], altura_estimada=100)
    sens_res = (sens[sens['No_modelo'] == 'Sim']
                .groupby(['Alavanca', 'Delta_%'])['Impacto_R$mil'].mean().reset_index())
    if len(sens_res):
        pdf.subtitulo('Impacto médio por alavanca no portfólio de candidatas')
        pdf.tabela(['Alavanca', 'Variação aplicada', 'Impacto médio (R$ mil/mês)', 'Leitura'],
                   [56, 30, 42, 50],
                   [[_dicionario_de(r['Alavanca'])['Rotulo'][:36], f"{r['Delta_%']:+,.0f}%",
                     f"{r['Impacto_R$mil']:+,.2f}",
                     'Relevante' if abs(r['Impacto_R$mil']) >= 1 else 'Marginal']
                    for _, r in sens_res.iterrows()],
                   tamanho=7.2, alinhamentos=['L', 'C', 'R', 'C'])
    sem_efeito = sorted(set(params['alavancas']) - set(ctx['features']))
    if sem_efeito:
        pdf.destaque('Alavancas sem efeito no modelo',
                     'As variáveis ' + ', '.join(sem_efeito) + ' não foram selecionadas pelo modelo campeão: '
                     'estatisticamente, alterá-las não muda a previsão. Isso não prova que sejam irrelevantes '
                     'para o negócio — significa que, nesta base e depois de controlar pelas demais variáveis, '
                     'o efeito delas não se distingue de zero.')
    pdf.subtitulo('Esforço necessário para atingir o hurdle')
    pdf.tabela(['Praça', 'Gap (R$ mil)', 'Alavanca', 'Variação necessária', 'Viável'],
               [18, 26, 56, 48, 30],
               [[str(r['Candidata']), f"{r['Gap_R$mil']:,.1f}", r['Alavanca'][:36],
                 str(r['Variacao_necessaria'])[:30], r['Viavel']] for _, r in esforco.iterrows()],
               tamanho=7.2, alinhamentos=['C', 'R', 'L', 'C', 'C'])
    pdf.paragrafo('A coluna "variação necessária" inverte o coeficiente do modelo: indica de quanto teria de '
                  'ser o aumento da alavanca para que a previsão alcance o hurdle, mantidas as demais '
                  'condições. Variações acima de 50% são classificadas como inviáveis por fugirem da faixa '
                  'observada na base histórica.', 8.2)

    # ------------------------------- 8. Caso econômico --------------------------------
    pdf.titulo_secao(8, 'Caso econômico', 'Tradução financeira da previsão')
    pdf.paragrafo('A camada financeira converte faturamento projetado em retorno sobre o capital investido. '
                  'Todos os parâmetros econômicos são premissas de trabalho declaradas no capítulo 9 e não '
                  'derivam da base de dados [#REF]; alterar qualquer um deles no dicionário PARAMETROS '
                  'reprocessa automaticamente as recomendações e este relatório.')
    pdf.tabela(['Praça', 'Receita/mês', 'EBITDA/mês', 'CAPEX', 'Payback', 'VPL 5 anos', 'TIR a.a.'],
               [18, 30, 30, 30, 22, 26, 22],
               [[str(r[idc]), f"{r['RECEITA_MENSAL_R$']/1e3:,.0f} mil", f"{r['EBITDA_MENSAL_R$']/1e3:,.1f} mil",
                 f"{r['CAPEX_R$']/1e3:,.0f} mil",
                 'n/a' if not np.isfinite(r['PAYBACK_MESES']) else f"{r['PAYBACK_MESES']:,.0f} m",
                 f"{r['VPL_R$']/1e6:,.2f} mi",
                 'n/a' if not np.isfinite(r['TIR_ANUAL_%']) else f"{r['TIR_ANUAL_%']:,.0f}%"]
                for _, r in R.iterrows()],
               tamanho=7.2, alinhamentos=['C', 'R', 'R', 'R', 'C', 'R', 'R'])
    pdf.paragrafo(f"EBITDA mensal = receita projetada x margem de contribuição de "
                  f"{params['margem_contribuicao_pct']*100:,.0f}% [#REF] menos custo fixo de "
                  f"R$ {params['custo_fixo_mensal_reais']:,.0f} [#REF] e a verba de marketing da própria "
                  f"praça. O VPL desconta {params['horizonte_meses']} meses de EBITDA a "
                  f"{params['taxa_desconto_anual']*100:,.0f}% ao ano [#REF] e subtrai o CAPEX. Payback 'n/a' "
                  f"indica EBITDA projetado negativo, isto é, unidade que não se paga sob estas premissas.", 8.2)
    pdf.exibir('9', 'Caso econômico por unidade: capital exigido, geração de caixa e valor criado',
               figs['E7_economia'], altura_estimada=95)

    # ------------------------------- 9. Premissas e riscos ----------------------------
    pdf.titulo_secao(9, 'Premissas, riscos e limitações', 'O que o comitê precisa saber antes de decidir')
    pdf.subtitulo('Premissas não derivadas dos dados [#REF]')
    pdf.tabela(['Premissa', 'Valor adotado', 'Natureza'],
               [70, 40, 68],
               [['Margem de contribuição', f"{params['margem_contribuicao_pct']*100:,.0f}%",
                 'Premissa de trabalho [#REF]'],
                ['Custo fixo mensal por unidade', f"R$ {params['custo_fixo_mensal_reais']:,.0f}",
                 'Premissa de trabalho [#REF]'],
                ['CAPEX fixo por unidade', f"R$ {params['capex_fixo_reais']:,.0f}",
                 'Premissa de trabalho [#REF]'],
                ['CAPEX por m2 de loja', f"R$ {params['capex_por_m2_reais']:,.0f}",
                 'Premissa de trabalho [#REF]'],
                ['Custo de capital', f"{params['taxa_desconto_anual']*100:,.0f}% a.a.",
                 'Premissa de trabalho [#REF]'],
                ['Horizonte de avaliação', f"{params['horizonte_meses']} meses", 'Convenção de análise'],
                ['Hurdle de faturamento', f"R$ {params['hurdle_faturamento']:,.0f} mil/mês",
                 'Definido pelo comitê'],
                ['Nível de confiança', f"{params['nivel_confianca']*100:,.0f}%", 'Convenção estatística']],
               tamanho=7.4, alinhamentos=['L', 'R', 'L'])
    riscos = [
        f"Extrapolação — {int(R['FORA_DO_DOMINIO'].sum())} candidata(s) apresentam ao menos uma característica "
        f"fora do intervalo típico da base histórica. Para elas a previsão é menos confiável e a recomendação "
        f"foi rebaixada por regra automática.",
        'Corte de outliers — a limpeza por IQR removeu lojas atípicas, inclusive as de faturamento muito alto. '
        'O modelo descreve bem a loja típica da rede e tende a subestimar operações excepcionais.',
        'Efeito de praça — parte relevante do poder explicativo vem de dummies de cidade e região, estimadas '
        'com poucas lojas por praça. Cidades sem histórico caem na categoria de referência e perdem esse '
        'ajuste, o que achata a previsão.',
        'Causalidade — os coeficientes medem associação condicional entre lojas, não efeito causal. Aumentar '
        'o marketing de uma loja específica não produz necessariamente o efeito estimado no corte transversal.',
        'Premissas econômicas — payback, VPL e TIR dependem integralmente das premissas acima, que não foram '
        'extraídas da base [#REF]. Devem ser validadas com a controladoria antes da decisão final.',
    ]
    if len(imput):
        riscos.append('Qualidade de dado — ' + '; '.join(
            f"a candidata {r['Candidata']} teve "
            f"{ROTULOS_CURTOS.get(r['Variavel'], r['Variavel']).lower()} imputada pela mediana da base "
            f"histórica ({r['Valor_imputado']:,.0f})" for _, r in imput.iterrows()) +
            '. Recomenda-se coletar o dado real antes da aprovação definitiva.')
    pdf.subtitulo('Riscos e limitações')
    pdf.bullets(riscos, tamanho=9.2)
    pdf.exibir('10', 'Aderência das candidatas ao domínio de dados em que o modelo foi treinado',
               figs['E8_extrapolacao'], altura_estimada=110)

    # ------------------------------- 10. Apêndice -------------------------------------
    pdf.titulo_secao(10, 'Apêndice: parâmetros de execução', 'Reprodutibilidade')
    pdf.paragrafo('O bloco abaixo reproduz integralmente a configuração usada nesta rodada. Alterar qualquer '
                  'valor no dicionário PARAMETROS e reexecutar a célula regenera previsões, gráficos, '
                  'recomendações e este documento — não há número escrito à mão no relatório.')
    pdf.set_font('Courier', '', 7)
    pdf.set_text_color(*pdf.c['cinza_texto'])
    for chave in ['hurdle_faturamento', 'regra_decisao', 'prob_minima_aprovacao', 'zona_cinzenta_pct',
                  'nivel_confianca', 'metodo_intervalo', 'rebaixar_por_extrapolacao',
                  'limite_z_extrapolacao', 'margem_contribuicao_pct', 'custo_fixo_mensal_reais',
                  'capex_por_m2_reais', 'capex_fixo_reais', 'taxa_desconto_anual', 'horizonte_meses',
                  'payback_maximo_meses', 'pesos_score', 'alavancas']:
        pdf.multi_cell(178, 4.2, _t(f"{chave} = {params[chave]}"), align='L')
    pdf.ln(3)
    pdf.set_font('Arial', '', 8.4)
    pdf.multi_cell(178, 4.6, _t(f"Variáveis do modelo campeão: {', '.join(ctx['features'])}"), align='J')
    pdf.ln(2)
    pdf.multi_cell(178, 4.6, _t(f"Equação estimada (unidades originais): faturamento = "
                                f"{ctx['intercepto_original']:,.2f} " +
                                ' '.join(f"{c:+,.4f}*{v}" for v, c in
                                         zip(tc['Variavel'], tc['Coef_unidade_original']))), align='J')

    pdf.output(params['arquivo_pdf'])
    return params['arquivo_pdf']

# =====================================================================================
# 10. EXECUÇÃO: pipeline completo, impressão no output e exportação dos resultados
# =====================================================================================
def _cabecalho(txt, char='='):
    print('\n' + char * 100)
    print(txt.upper().center(100))
    print(char * 100)

def executar_aplicacao(params=P):
    modelo = globals().get('modelo_vencedor', None)
    base = globals().get('df', None)
    if base is None:
        raise RuntimeError('DataFrame "df" não encontrado: execute as células anteriores do notebook.')

    # ---- 1. Contexto e ficha técnica do modelo --------------------------------------
    ctx = reconstruir_contexto_modelo(base, modelo, params)
    m = ctx['metricas']

    _cabecalho('1. dicionário de variáveis: o que cada dado significa')
    display(DICIONARIO_VARIAVEIS)
    print('\nVariáveis VETADAS e o motivo do veto:')
    for _, r in DICIONARIO_VARIAVEIS[DICIONARIO_VARIAVEIS['Status_no_modelo'] == 'VETADA'].iterrows():
        print(f"  • {r['Variavel']:<28} {r['Justificativa']}")

    _cabecalho('2. ficha técnica do modelo campeão (model card)')
    ficha = pd.DataFrame({
        'Indicador': ['Família do modelo', 'Alpha (LASSO)', 'Observações (total/treino/teste)',
                      'Variáveis no pool', 'Variáveis no modelo', 'R² treino', 'R² teste',
                      'RMSE treino (R$ mil)', 'RMSE teste (R$ mil)', 'MAE teste (R$ mil)',
                      'MAPE teste (%)', 'Erro relativo (% da média)', 'R² ajustado (OLS gêmeo)',
                      'Teste F (p-valor)', 'Durbin-Watson', 'Jarque-Bera (p)', 'Breusch-Pagan (p)',
                      'VIF máximo', 'Média do alvo (R$ mil)', 'Desvio-padrão do alvo (R$ mil)'],
        'Valor': ['Regressão linear com regularização LASSO', f"{m['alpha_lasso']}",
                  f"{m['n_total']} / {m['n_treino']} / {m['n_teste']}", m['n_variaveis_pool'],
                  m['n_variaveis_modelo'], f"{m['r2_treino']:.4f}", f"{m['r2_teste']:.4f}",
                  f"{m['rmse_treino']:,.2f}", f"{m['rmse_teste']:,.2f}", f"{m['mae_teste']:,.2f}",
                  f"{m['mape_teste']:,.2f}", f"{m['erro_relativo_%']:,.2f}", f"{m['r2_ajustado_ols']:.4f}",
                  f"{m['f_pvalue_ols']:.3e}", f"{m['durbin_watson']:.3f}", f"{m['jarque_bera_p']:.4f}",
                  f"{m['breusch_pagan_p']:.4f}", f"{m['vif_maximo']:,.2f}", f"{m['media_alvo']:,.2f}",
                  f"{m['desvio_alvo']:,.2f}"]})
    display(ficha)

    print('\nEquação estimada (coeficientes em unidades originais):')
    termos = ' '.join(f"{c:+,.4f}*{v}" for v, c in
                      zip(ctx['tabela_coef']['Variavel'], ctx['tabela_coef']['Coef_unidade_original']))
    print(f"  faturamento_medio_mensal = {ctx['intercepto_original']:,.2f} {termos}")
    print('\nCoeficientes, significância e VIF de cada variável do modelo:')
    display(ctx['tabela_coef'].round(4))
    print('\nLeitura de negócio de cada variável selecionada:')
    for _, r in ctx['tabela_coef'].iterrows():
        d = _dicionario_de(r['Variavel'])
        sentido = 'eleva' if r['Coef_padronizado'] >= 0 else 'reduz'
        print(f"  • {r['Rotulo']} ({d['Familia']}) — {d['Descricao']}.")
        print(f"      +1 desvio-padrão {sentido} o faturamento em R$ {abs(r['Coef_padronizado']):,.2f} mil/mês "
              f"| efeito por unidade: {r['Coef_unidade_original']:+,.4f} | p-valor {r['P_valor']:.4f} "
              f"| VIF {r['VIF']:,.1f} | {r['Peso_relativo_%']:,.0f}% do poder explicativo")
    print('\nDiagnóstico dos resíduos do modelo OLS gêmeo:')
    print(f"  Durbin-Watson = {m['durbin_watson']:.3f} | Jarque-Bera p = {m['jarque_bera_p']:.4f} | "
          f"Breusch-Pagan p = {m['breusch_pagan_p']:.4f}")

    # ---- 2. Candidatas ---------------------------------------------------------------
    df_cand = carregar_candidatas(params)
    X_cand, diag, zsc, imput = preparar_candidatas(df_cand, ctx, params)
    _cabecalho('3. praças candidatas e qualidade dos dados')
    display(df_cand)
    if len(imput):
        print('Células imputadas (pela mediana da base histórica, e não pela média das 5 candidatas):')
        display(imput)
    else:
        print('Nenhum dado faltante nas candidatas.')
    print('\nTeste de aplicabilidade — a candidata está dentro do domínio de treino?')
    display(diag)

    # ---- 3. Recomendação -------------------------------------------------------------
    R = motor_de_recomendacao(df_cand, X_cand, diag, ctx, params)
    contrib = decompor_previsao(X_cand, ctx, params)
    sens = analise_de_sensibilidade(X_cand, ctx, params)
    esforco = esforco_para_atingir_hurdle(R, ctx, params)

    _cabecalho(f"4. matriz de decisão (regra paramétrica: {params['regra_decisao']})")
    colunas_exec = [params['coluna_id_candidata'], 'cidade', 'PREVISAO_FATURAMENTO', 'IC_INF', 'IC_SUP',
                    'PROB_SUPERAR_HURDLE', 'GAP_VS_HURDLE_%', 'PAYBACK_MESES', 'VPL_R$',
                    'SCORE_PRIORIDADE', 'RANKING', 'RECOMENDACAO']
    display(R[colunas_exec].round(2))
    print('\nJustificativa de cada veredito:')
    for _, r in R.iterrows():
        print(f"  [{r['RANKING']}] {r[params['coluna_id_candidata']]} — {r['cidade']:<14} "
              f"{r['RECOMENDACAO']:<22} {r['JUSTIFICATIVA']}")

    _cabecalho('5. decomposição da previsão (contribuição de cada variável, em R$ mil/mês)')
    display(contrib.round(2))
    _cabecalho('6. sensibilidade às alavancas gerenciais')
    if len(sens):
        display(sens.pivot_table(index=['Candidata', 'Alavanca'], columns='Delta_%',
                                 values='Impacto_R$mil').round(2))
    sem_efeito = sorted(set(params['alavancas']) - set(ctx['features']))
    if sem_efeito:
        print(f"Alavancas sem efeito no modelo campeão (coeficiente nulo): {', '.join(sem_efeito)}")
    _cabecalho('7. esforço necessário para atingir o hurdle')
    display(esforco)
    _cabecalho('8. caso econômico por unidade (premissas do comitê) [#REF]')
    display(R[[params['coluna_id_candidata'], 'RECEITA_MENSAL_R$', 'MARGEM_CONTRIBUICAO_R$',
               'EBITDA_MENSAL_R$', 'CAPEX_R$', 'PAYBACK_MESES', 'VPL_R$', 'TIR_ANUAL_%',
               'ROI_CAPEX_%']].round(1))

    # ---- 4. Gráficos, exportações e relatório -----------------------------------------
    _cabecalho('9. exhibits do relatório')
    figs = gerar_graficos(R, ctx, sens, contrib, zsc, params)

    R.to_csv(params['arquivo_csv'], index=False)
    print(f"\nMatriz de decisão exportada: {params['arquivo_csv']}")

    caminho_pdf = None
    if params['gerar_pdf']:
        caminho_pdf = montar_relatorio(R, ctx, sens, esforco, imput, figs, contrib, params)
        print(f"Relatório executivo gerado: {caminho_pdf}")

    _cabecalho('resumo para o comitê', '#')
    ap = R[R['RECOMENDACAO'] == 'ABRIR']
    rs = R[R['RECOMENDACAO'] == 'ABRIR COM RESSALVAS']
    nr = R[R['RECOMENDACAO'] == 'NÃO ABRIR']
    idc = params['coluna_id_candidata']
    print(f"  Aprovadas ({len(ap)}): " + (', '.join(f"{r[idc]}/{r['cidade']}" for _, r in ap.iterrows()) or '—'))
    print(f"  Com ressalvas ({len(rs)}): " + (', '.join(f"{r[idc]}/{r['cidade']}" for _, r in rs.iterrows()) or '—'))
    print(f"  Reprovadas ({len(nr)}): " + (', '.join(f"{r[idc]}/{r['cidade']}" for _, r in nr.iterrows()) or '—'))
    print(f"  CAPEX recomendado: R$ {ap['CAPEX_R$'].sum()/1e6:,.2f} mi  |  "
          f"VPL em 5 anos: R$ {ap['VPL_R$'].sum()/1e6:,.2f} mi   [#REF: premissas econômicas do comitê]")
    print(f"  Erro típico do modelo: R$ {m['rmse_teste']:,.1f} mil/mês ({m['erro_relativo_%']:,.1f}% da média "
          f"da rede) — toda decisão acima já considera esse erro via intervalo de confiança.")

    # ---- 5. Pacote de resultados para a célula de consolidação (deck) -----------------
    return {
        'parametros': params, 'dicionario_variaveis': DICIONARIO_VARIAVEIS,
        'contexto_modelo': ctx, 'candidatas_brutas': df_cand, 'X_candidatas': X_cand,
        'diagnostico_extrapolacao': diag, 'zscores': zsc, 'imputacoes': imput,
        'recomendacoes': R, 'contribuicoes': contrib, 'sensibilidade': sens,
        'esforco_hurdle': esforco, 'figuras': figs, 'relatorio_pdf': caminho_pdf,
    }

RESULTADOS_APLICACAO = executar_aplicacao(P)


#Parte 6: Consolidação geral e deck de apresentação

Esta última etapa não refaz nenhuma análise: ela **coleta** o que cada célula anterior produziu — base bruta, base tratada, EDA, profiler, os quatro modelos exploratórios, o veto de variáveis, o modelo campeão e a aplicação às praças candidatas — recalcula apenas o que não ficou disponível em memória (como o benchmark comparável entre os modelos) e monta um deck 16:9 completo, no mesmo padrão visual do relatório executivo.

Saídas desta célula:
- `Deck_Estrategia_Expansao.pdf` — apresentação completa (~30 slides), com capa, agenda, sumário executivo, diagnóstico da base, comparação de modelos, ficha técnica, matriz de decisão, uma página por praça candidata, caso econômico, riscos, premissas e plano de ação;
- pasta `slides_deck/` — cada slide também em PNG, para colar em outro material;
- no output da célula: perfil da rede, correlações, descritivas, benchmark de todos os modelos testados e a matriz de decisão consolidada.


In [ ]:
# =====================================================================================
#  PARTE 6 — CONSOLIDAÇÃO GERAL E DECK DE APRESENTAÇÃO
# =====================================================================================
#  Esta célula não repete análise: ela COLETA o que cada célula anterior produziu
#  (base bruta, base tratada, EDA, profiler, os quatro modelos exploratórios, o veto de
#  variáveis, o modelo campeão e a aplicação às praças candidatas), recalcula apenas o
#  que não ficou disponível em memória e monta um deck 16:9 pronto para o comitê.
# =====================================================================================

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.image import imread
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from datetime import datetime
from IPython.display import display

warnings.filterwarnings('ignore')

# =====================================================================================
# 1. PARÂMETROS DO DECK
# =====================================================================================
DECK = {
    'arquivo': 'Deck_Estrategia_Expansao.pdf',
    'pasta_slides': 'slides_deck',
    'exportar_png': True,
    'largura_pol': 13.333, 'altura_pol': 7.5,     # 16:9
    'dpi': 150,
    'recalcular_benchmark': True,   # recalcula os modelos exploratórios para comparação
    'n_tentativas_benchmark': 150,  # orçamento da busca aleatória usada no benchmark
    'titulo': 'Estratégia de expansão de unidades',
    'subtitulo': 'Do dado histórico à decisão de alocação de capital',
    'rodape': 'Trabalho T1 — Analytics III  |  Documento de trabalho, uso interno',
}
DECK['pasta_slides'] and os.makedirs(DECK['pasta_slides'], exist_ok=True)

# Reaproveita a identidade visual e os parâmetros de negócio definidos na célula anterior
P_DECK = PARAMETROS
C = P_DECK['marca']
COR_TITULO, COR_TEXTO = C['verde_escuro'], C['cinza_texto']
CORES_VEREDITO = {'ABRIR': C['verde'], 'ABRIR COM RESSALVAS': C['ambar'], 'NÃO ABRIR': C['terracota']}

# =====================================================================================
# 2. COLETA: o que cada célula anterior deixou em memória
# =====================================================================================
def coletar_resultados():
    """Reúne em um único dicionário tudo o que as células anteriores produziram."""
    g = globals()
    pacote = {}

    # --- base bruta e base tratada ----------------------------------------------------
    caminhos = [g.get('caminho_drive'), '/content/drive/MyDrive/analytics III/T1/franquia_lojas.csv',
                '/content/franquia_lojas.csv', 'franquia_lojas.csv']
    bruta = None
    for c in caminhos:
        if c and os.path.exists(c):
            bruta = pd.read_csv(c)
            break
    pacote['df_bruto'] = bruta
    pacote['df_tratado'] = g.get('df')

    # --- aplicação (célula anterior) --------------------------------------------------
    app = g.get('RESULTADOS_APLICACAO')
    if app is None:
        raise RuntimeError('RESULTADOS_APLICACAO não encontrado: execute a célula da Parte 5 antes desta.')
    pacote['app'] = app
    pacote['ctx'] = app['contexto_modelo']
    pacote['R'] = app['recomendacoes']
    pacote['figs'] = app['figuras']

    # --- stepwise (célula da Parte 2) --------------------------------------------------
    pacote['stepwise_features'] = g.get('selected_features')
    pacote['stepwise_summary'] = g.get('best_model_summary')

    # --- perfil de alta performance (profiler) -----------------------------------------
    pacote['profiler'] = perfilar_rede(pacote['df_tratado'], P_DECK['hurdle_faturamento'])

    # --- estatísticas da EDA ------------------------------------------------------------
    df_t = pacote['df_tratado']
    alvo = P_DECK['target']
    num = df_t.select_dtypes(include=[np.number])
    corr = num.corr()[alvo].drop(alvo).sort_values(key=np.abs, ascending=False)
    pacote['correlacoes'] = corr
    pacote['descritivas'] = num[[c for c in ROTULOS_CURTOS if c in num.columns]].describe().T
    return pacote

def perfilar_rede(df_t, corte):
    """Reproduz o profiler: compara lojas de alta performance com as demais."""
    d = df_t.copy()
    d['grupo'] = np.where(d[P_DECK['target']] > corte, 'Alta performance', 'Performance padrão')
    cols = [c for c in ['area_m2', 'populacao_1km', 'renda_media_domiciliar', 'fluxo_pedestres_dia',
                        'vagas_estacionamento', 'investimento_marketing_mensal', 'n_funcionarios',
                        'ticket_medio_rs', 'nota_satisfacao_cliente'] if c in d.columns]
    alta, padrao = d[d['grupo'] == 'Alta performance'], d[d['grupo'] == 'Performance padrão']
    linhas = []
    for c in cols:
        t, p = stats.ttest_ind(alta[c], padrao[c], equal_var=False)
        linhas.append({'Variavel': c, 'Rotulo': ROTULOS_CURTOS.get(c, c),
                       'Media_alta': alta[c].mean(), 'Media_padrao': padrao[c].mean(),
                       'Diferenca_%': 100 * (alta[c].mean() - padrao[c].mean()) / max(padrao[c].mean(), 1e-9),
                       'P_valor': p})
    return {'tabela': pd.DataFrame(linhas).sort_values('P_valor').reset_index(drop=True),
            'n_alta': len(alta), 'n_padrao': len(padrao), 'corte': corte}

# =====================================================================================
# 3. BENCHMARK DOS MODELOS TESTADOS AO LONGO DO TRABALHO
# =====================================================================================
def benchmark_modelos(df_t, ctx, params=DECK):
    """Recalcula, em condições comparáveis, as quatro famílias testadas + o campeão."""
    alvo = P_DECK['target']
    d = df_t.select_dtypes(include=[np.number]).astype(float).fillna(df_t.select_dtypes(
        include=[np.number]).mean())
    todas = [c for c in d.columns if c != alvo]
    X_tr, X_te, y_tr, y_te = train_test_split(d[todas], d[alvo], test_size=0.3, random_state=42)
    resultados = []

    def _registrar(nome, feats, y_pred_tr, y_pred_te, obs, usa_pos):
        resultados.append({
            'Modelo': nome, 'N_variaveis': len(feats),
            'R2_treino': r2_score(y_tr, y_pred_tr), 'R2_teste': r2_score(y_te, y_pred_te),
            'RMSE_teste': float(np.sqrt(mean_squared_error(y_te, y_pred_te))),
            'Usa_variaveis_pos_abertura': usa_pos, 'Observacao': obs})

    # (1) OLS com todas as variáveis — baseline sem nenhum critério de seleção
    ols_full = sm.OLS(y_tr, sm.add_constant(X_tr)).fit()
    _registrar('1. OLS com todas as variáveis', todas,
               ols_full.predict(sm.add_constant(X_tr)),
               ols_full.predict(sm.add_constant(X_te, has_constant='add')),
               'Ajuste enganoso: usa dados que só existem após abrir', 'Sim')

    # (2) LOOCV sobre todas as variáveis
    lin = LinearRegression()
    mse_loo = -cross_val_score(lin, d[todas], d[alvo], cv=LeaveOneOut(),
                               scoring='neg_mean_squared_error').mean()
    lin.fit(X_tr, y_tr)
    resultados.append({'Modelo': '2. LOOCV (todas as variáveis)', 'N_variaveis': len(todas),
                       'R2_treino': lin.score(X_tr, y_tr), 'R2_teste': lin.score(X_te, y_te),
                       'RMSE_teste': float(np.sqrt(mse_loo)),
                       'Usa_variaveis_pos_abertura': 'Sim',
                       'Observacao': 'RMSE irreal: prevê faturamento com o próprio faturamento'})

    # (3) Forward stepwise com corte de p-valor
    restantes = [c for c in todas if c != 'id_loja']
    sel = []
    while True:
        candidatos = []
        for f in restantes:
            try:
                mod = smf.ols(f"{alvo} ~ {' + '.join(sel + [f])}", data=pd.concat([X_tr, y_tr], axis=1)).fit()
                if f in mod.pvalues.index:
                    candidatos.append((mod.pvalues[f], f, mod))
            except Exception:
                pass
        candidatos.sort(key=lambda x: x[0])
        if candidatos and candidatos[0][0] < 0.05:
            sel.append(candidatos[0][1]); restantes.remove(candidatos[0][1])
        else:
            break
    if sel:
        ols_sw = sm.OLS(y_tr, sm.add_constant(X_tr[sel])).fit()
        _registrar('3. Forward stepwise (p < 0,05)', sel,
                   ols_sw.predict(sm.add_constant(X_tr[sel])),
                   ols_sw.predict(sm.add_constant(X_te[sel], has_constant='add')),
                   'Seleção automática, mas ainda escolhe dados pós-abertura',
                   'Sim' if any(v in sel for v in VARIAVEIS_VETADAS) else 'Não')

    # (4) Busca aleatória + LASSO com todas as variáveis
    if params['recalcular_benchmark']:
        import random as _rnd
        _rnd.seed(42)
        sc = StandardScaler().fit(X_tr)
        Xtr_s = pd.DataFrame(sc.transform(X_tr), columns=todas, index=X_tr.index)
        Xte_s = pd.DataFrame(sc.transform(X_te), columns=todas, index=X_te.index)
        melhores = []
        for _ in range(params['n_tentativas_benchmark']):
            k = _rnd.randint(1, min(12, len(todas)))
            feats = _rnd.sample(todas, k)
            while feats:
                pv = sm.OLS(y_tr, sm.add_constant(X_tr[feats])).fit().pvalues.drop('const', errors='ignore')
                if pv.max() > 0.05:
                    feats.remove(pv.idxmax())
                else:
                    break
            if feats:
                for a in [0.01, 0.1, 0.5]:
                    ls = Lasso(alpha=a, random_state=42).fit(Xtr_s[feats], y_tr)
                    melhores.append((ls.score(Xtr_s[feats], y_tr), feats, ls))
        if melhores:
            melhores.sort(key=lambda x: -x[0])
            r2, feats, ls = melhores[0]
            _registrar('4. Busca aleatória + LASSO (todas)', feats,
                       ls.predict(Xtr_s[feats]), ls.predict(Xte_s[feats]),
                       'Regularização estabiliza, mas o vazamento permanece',
                       'Sim' if any(v in feats for v in VARIAVEIS_VETADAS) else 'Não')

    # (5) Modelo campeão — variáveis pós-abertura vetadas
    m = ctx['metricas']
    resultados.append({'Modelo': '5. LASSO sem variáveis pós-abertura (campeão)',
                       'N_variaveis': m['n_variaveis_modelo'],
                       'R2_treino': m['r2_treino'], 'R2_teste': m['r2_teste'],
                       'RMSE_teste': m['rmse_teste'], 'Usa_variaveis_pos_abertura': 'Não',
                       'Observacao': 'Único modelo aplicável a um ponto que ainda não abriu'})
    return pd.DataFrame(resultados)

# =====================================================================================
# 4. MOTOR DE SLIDES — primitivas de layout 16:9 no padrão da consultoria
# =====================================================================================
import textwrap

class Apresentacao:
    """Constrói o deck slide a slide, em PDF único (e PNG opcional por slide)."""

    def __init__(self, params=DECK):
        self.p = params
        self.pdf = PdfPages(params['arquivo'])
        self.n = 0
        plt.rcParams.update({'font.family': 'DejaVu Sans', 'axes.unicode_minus': False})
        if 'text.parse_math' in plt.rcParams:
            plt.rcParams['text.parse_math'] = False

    # ---------------- estrutura ------------------------------------------------------
    def novo_slide(self, titulo=None, kicker='', capa=False, secao=False):
        fig = plt.figure(figsize=(self.p['largura_pol'], self.p['altura_pol']), dpi=self.p['dpi'])
        fig.patch.set_facecolor(C['branco'] if not (capa or secao) else C['verde_escuro'])
        self.capa_atual = capa or secao
        if capa or secao:
            fig.patches.append(mpatches.Rectangle((0, 0), 1, 0.012, transform=fig.transFigure,
                                                  color=C['verde_neon'], zorder=5))
            return fig
        # faixa superior
        fig.patches.append(mpatches.Rectangle((0, 0.962), 1, 0.038, transform=fig.transFigure,
                                              color=C['verde_escuro'], zorder=5))
        fig.patches.append(mpatches.Rectangle((0, 0.9585), 1, 0.0035, transform=fig.transFigure,
                                              color=C['verde_neon'], zorder=5))
        fig.text(0.035, 0.9755, self.p['titulo'].upper(), fontsize=8.5, color='white',
                 fontweight='bold', va='center', zorder=6)
        fig.text(0.965, 0.9755, kicker.upper(), fontsize=8, color=C['verde_claro'],
                 va='center', ha='right', zorder=6)
        if titulo:
            fig.text(0.035, 0.928, textwrap.fill(titulo, 82), fontsize=18, color=COR_TITULO,
                     fontweight='bold', va='top', linespacing=1.25)
            fig.patches.append(mpatches.Rectangle((0.035, 0.838), 0.045, 0.005,
                                                  transform=fig.transFigure, color=C['verde_neon']))
        # rodapé
        fig.text(0.035, 0.028, self.p['rodape'], fontsize=7, color=C['cinza_medio'], va='center')
        return fig

    def fechar_slide(self, fig, numerar=True):
        self.n += 1
        if numerar and not self.capa_atual:
            fig.text(0.965, 0.028, f'{self.n}', fontsize=8.5, color=C['verde'],
                     fontweight='bold', va='center', ha='right')
        self.pdf.savefig(fig, facecolor=fig.get_facecolor())
        if self.p['exportar_png']:
            fig.savefig(os.path.join(self.p['pasta_slides'], f'slide_{self.n:02d}.png'),
                        dpi=110, facecolor=fig.get_facecolor())
        plt.close(fig)

    def finalizar(self):
        d = self.pdf.infodict()
        d['Title'] = self.p['titulo']
        d['Author'] = P_DECK['autores']
        d['Subject'] = self.p['subtitulo']
        self.pdf.close()
        return self.p['arquivo']

    # ---------------- primitivas de conteúdo -----------------------------------------
    @staticmethod
    def texto(fig, x, y, s, tamanho=11, cor=None, negrito=False, largura=None, ha='left',
              va='top', italico=False, espaco=1.35):
        if largura:
            s = '\n'.join(textwrap.fill(par, largura) for par in s.split('\n'))
        return fig.text(x, y, s, fontsize=tamanho, color=cor or COR_TEXTO, ha=ha, va=va,
                        fontweight='bold' if negrito else 'normal',
                        fontstyle='italic' if italico else 'normal', linespacing=espaco)

    @staticmethod
    def marcadores(fig, x, y, itens, tamanho=11, largura=95, passo=None, cor_marca=None):
        yy = y
        for it in itens:
            linhas = textwrap.wrap(it, largura) or ['']
            fig.patches.append(mpatches.Rectangle((x, yy - 0.008), 0.007, 0.012,
                                                  transform=fig.transFigure,
                                                  color=cor_marca or C['verde_neon']))
            fig.text(x + 0.016, yy, '\n'.join(linhas), fontsize=tamanho, color=COR_TEXTO,
                     va='top', linespacing=1.4)
            yy -= (passo or 0.045) + 0.028 * (len(linhas) - 1)
        return yy

    @staticmethod
    def caixa(fig, x, y, w, h, titulo, texto, cor_fundo=None, cor_barra=None, largura_txt=78,
              tamanho=10.5):
        # altura automática: acompanha o número de linhas do texto após a quebra
        n_linhas = len(textwrap.wrap(texto, largura_txt)) or 1
        h_min = (0.058 if titulo else 0.030) + 0.0265 * n_linhas * (tamanho / 10.5)
        h = max(h, h_min)
        fig.patches.append(mpatches.Rectangle((x, y - h), w, h, transform=fig.transFigure,
                                             color=cor_fundo or C['verde_palido'], zorder=1))
        fig.patches.append(mpatches.Rectangle((x, y - h), 0.005, h, transform=fig.transFigure,
                                             color=cor_barra or C['verde'], zorder=2))
        if titulo:
            fig.text(x + 0.018, y - 0.032, titulo.upper(), fontsize=9.5, color=COR_TITULO,
                     fontweight='bold', va='center', zorder=3)
            fig.text(x + 0.018, y - 0.055, textwrap.fill(texto, largura_txt), fontsize=tamanho,
                     color=COR_TEXTO, va='top', zorder=3, linespacing=1.4)
        else:
            fig.text(x + 0.018, y - 0.03, textwrap.fill(texto, largura_txt), fontsize=tamanho,
                     color=COR_TEXTO, va='top', zorder=3, linespacing=1.4)

    @staticmethod
    def kpi(fig, x, y, w, h, rotulo, valor, sub='', cor=None):
        fig.patches.append(mpatches.Rectangle((x, y - h), w, h, transform=fig.transFigure,
                                             color=cor or C['verde_escuro'], zorder=1))
        fig.patches.append(mpatches.Rectangle((x, y - 0.006), w, 0.006, transform=fig.transFigure,
                                             color=C['verde_neon'], zorder=2))
        fig.text(x + 0.014, y - 0.035, rotulo.upper(), fontsize=8, color=C['verde_claro'],
                 va='center', zorder=3)
        fig.text(x + 0.014, y - 0.075, valor, fontsize=21, color='white', fontweight='bold',
                 va='center', zorder=3)
        if sub:
            fig.text(x + 0.014, y - 0.108, sub, fontsize=7.8, color=C['verde_claro'],
                     va='center', zorder=3)

    @staticmethod
    def tabela(fig, x, y, w, colunas, larguras_rel, linhas, alt_linha=0.042, tamanho=9,
               alinhamentos=None, cores_col=None, alt_cabecalho=None):
        """Tabela desenhada em coordenadas de figura, com zebra e cabeçalho verde."""
        alinhamentos = alinhamentos or ['center'] * len(colunas)
        larguras = np.array(larguras_rel, dtype=float)
        larguras = larguras / larguras.sum() * w
        bordas = np.concatenate([[x], x + np.cumsum(larguras)])
        alt_cab = alt_cabecalho or alt_linha
        fig.patches.append(mpatches.Rectangle((x, y - alt_cab), w, alt_cab, transform=fig.transFigure,
                                             color=C['verde_escuro'], zorder=2))
        for j, col in enumerate(colunas):
            fig.text((bordas[j] + bordas[j + 1]) / 2, y - alt_cab / 2, col, fontsize=tamanho,
                     color='white', fontweight='bold', ha='center', va='center', zorder=3)
        yy = y - alt_cab
        for i, linha in enumerate(linhas):
            if i % 2 == 0:
                fig.patches.append(mpatches.Rectangle((x, yy - alt_linha), w, alt_linha,
                                                      transform=fig.transFigure,
                                                      color=C['verde_palido'], zorder=1))
            for j, val in enumerate(linha):
                al = alinhamentos[j]
                px = {'center': (bordas[j] + bordas[j + 1]) / 2,
                      'left': bordas[j] + 0.006, 'right': bordas[j + 1] - 0.006}[al]
                cor = COR_TEXTO
                negrito = False
                if cores_col and j in cores_col:
                    cor = cores_col[j][i]
                    negrito = True
                fig.text(px, yy - alt_linha / 2, str(val), fontsize=tamanho, color=cor,
                         ha=al, va='center', zorder=3, fontweight='bold' if negrito else 'normal')
            yy -= alt_linha
        return yy

    def imagem(self, fig, caminho, x, y, w, h):
        """Insere um exhibit já gerado, preservando a proporção dentro da caixa (x, y, w, h)."""
        if not caminho or not os.path.exists(caminho):
            return
        img = imread(caminho)
        prop_img = img.shape[0] / img.shape[1]
        prop_caixa = (h * self.p['altura_pol']) / (w * self.p['largura_pol'])
        if prop_img > prop_caixa:      # limitado pela altura
            h_final = h
            w_final = h * self.p['altura_pol'] / prop_img / self.p['largura_pol']
        else:                          # limitado pela largura
            w_final = w
            h_final = w * self.p['largura_pol'] * prop_img / self.p['altura_pol']
        ax = fig.add_axes([x + (w - w_final) / 2, y - h + (h - h_final) / 2, w_final, h_final])
        ax.imshow(img)
        ax.axis('off')
        return ax

    def eixo(self, fig, x, y, w, h):
        """Cria um eixo matplotlib para gráficos desenhados diretamente no slide."""
        ax = fig.add_axes([x, y - h, w, h])
        ax.set_facecolor('white')
        for lado in ['top', 'right']:
            ax.spines[lado].set_visible(False)
        for lado in ['left', 'bottom']:
            ax.spines[lado].set_color(C['cinza_medio'])
        ax.tick_params(colors=COR_TEXTO, labelsize=8.5)
        ax.grid(color=C['cinza_claro'], lw=0.9)
        ax.set_axisbelow(True)
        return ax

# =====================================================================================
# 5. MONTAGEM DO DECK
# =====================================================================================
def montar_deck(pacote, benchmark, params=DECK):
    A = Apresentacao(params)
    R, ctx, figs = pacote['R'], pacote['ctx'], pacote['figs']
    m, idc = ctx['metricas'], P_DECK['coluna_id_candidata']
    hurdle = P_DECK['hurdle_faturamento']
    bruto, tratado = pacote['df_bruto'], pacote['df_tratado']
    aprov = R[R['RECOMENDACAO'] == 'ABRIR']
    ress = R[R['RECOMENDACAO'] == 'ABRIR COM RESSALVAS']
    reprov = R[R['RECOMENDACAO'] == 'NÃO ABRIR']
    cores_vered = {8: [CORES_VEREDITO[v] for v in R['RECOMENDACAO']]}

    # ---------------------------------------------------------------- S1: capa --------
    f = A.novo_slide(capa=True)
    f.patches.append(mpatches.Rectangle((0.035, 0.80), 0.09, 0.008, transform=f.transFigure,
                                        color=C['verde_neon']))
    A.texto(f, 0.035, 0.72, params['titulo'], tamanho=40, cor='white', negrito=True, largura=32,
            va='top')
    A.texto(f, 0.035, 0.45, params['subtitulo'], tamanho=15, cor=C['verde_claro'], largura=60)
    f.patches.append(mpatches.Rectangle((0.035, 0.26), 0.30, 0.002, transform=f.transFigure,
                                        color=C['verde_medio']))
    A.texto(f, 0.035, 0.235, 'Comitê de investimentos — documento de trabalho', tamanho=10.5,
            cor='white', negrito=True)
    A.texto(f, 0.035, 0.19, f"Equipe: {P_DECK['autores']}\n"
                            f"Base analisada: {len(bruto) if bruto is not None else '—'} unidades em operação  |  "
                            f"{len(R)} praças candidatas\n"
                            f"Emissão: {datetime.now().strftime('%d/%m/%Y')}",
            tamanho=10, cor=C['verde_claro'])
    A.fechar_slide(f, numerar=False)

    # ---------------------------------------------------------------- S2: agenda ------
    f = A.novo_slide('O que este documento responde', 'Agenda')
    itens = [
        ('01', 'Diagnóstico da base', 'O que os dados das lojas em operação dizem sobre performance'),
        ('02', 'Modelagem', 'Quatro famílias testadas, o veto de variáveis e o modelo escolhido'),
        ('03', 'Aplicação e recomendação', 'Previsão, risco e decisão para cada praça candidata'),
        ('04', 'Caso econômico e riscos', 'Capital exigido, retorno esperado e o que pode dar errado'),
        ('05', 'Próximos passos', 'O que pedimos ao comitê e em que sequência'),
    ]
    y = 0.75
    for num, tit, desc in itens:
        f.patches.append(mpatches.Rectangle((0.035, y - 0.085), 0.055, 0.085,
                                            transform=f.transFigure, color=C['verde_escuro']))
        f.text(0.0625, y - 0.0425, num, fontsize=17, color='white', fontweight='bold',
               ha='center', va='center')
        f.patches.append(mpatches.Rectangle((0.09, y - 0.085), 0.875, 0.085,
                                            transform=f.transFigure, color=C['verde_palido']))
        f.text(0.108, y - 0.028, tit, fontsize=13, color=COR_TITULO, fontweight='bold', va='center')
        f.text(0.108, y - 0.060, desc, fontsize=10, color=COR_TEXTO, va='center')
        y -= 0.10
    A.fechar_slide(f)

    # ------------------------------------------------- S3: sumário executivo ----------
    f = A.novo_slide(f"{len(aprov)} das {len(R)} praças candidatas devem receber capital neste ciclo",
                     'Sumário executivo')
    largura_kpi = 0.215
    kpis = [('Praças avaliadas', f'{len(R)}', 'no pipeline atual'),
            ('Aprovadas', f'{len(aprov)}', f"CAPEX de R$ {aprov['CAPEX_R$'].sum()/1e6:,.2f} mi [#REF]"),
            ('VPL em 5 anos', f"R$ {aprov['VPL_R$'].sum()/1e6:,.2f} mi", 'das unidades aprovadas [#REF]'),
            ('Precisão do modelo', f"{m['r2_teste']*100:,.0f}%", f"R² fora da amostra")]
    for i, (rot, val, sub) in enumerate(kpis):
        A.kpi(f, 0.035 + i * (largura_kpi + 0.018), 0.80, largura_kpi, 0.135, rot, val, sub)
    A.texto(f, 0.035, 0.625, 'SITUAÇÃO', tamanho=9, cor=C['verde_medio'], negrito=True)
    A.texto(f, 0.035, 0.60, f'A rede opera {len(bruto) if bruto is not None else "—"} unidades e avalia '
            f'{len(R)} novos pontos. A decisão de abrir hoje é tomada sem uma estimativa formal de '
            f'faturamento, e cada erro custa o CAPEX inteiro da unidade.', tamanho=10.2, largura=44)
    A.texto(f, 0.36, 0.625, 'COMPLICAÇÃO', tamanho=9, cor=C['verde_medio'], negrito=True)
    A.texto(f, 0.36, 0.60, 'Os dados históricos mais correlacionados com faturamento — ticket médio, '
            'transações, tempo de operação — só existem depois da abertura. Usá-los produz um modelo '
            'de ajuste quase perfeito e utilidade zero.', tamanho=10.2, largura=44)
    A.texto(f, 0.685, 0.625, 'RESOLUÇÃO', tamanho=9, cor=C['verde_medio'], negrito=True)
    A.texto(f, 0.685, 0.60, f'Um modelo construído apenas com variáveis conhecidas antes da abertura '
            f'explica {m["r2_teste"]*100:,.0f}% da variação de faturamento e transforma cada praça em '
            f'uma decisão com risco quantificado.', tamanho=10.2, largura=44)
    A.tabela(f, 0.035, 0.455, 0.93,
             ['#', 'Praça', 'Cidade', 'Previsão (R$ mil)', 'Intervalo de 95%', 'P(> hurdle)',
              'Payback', 'VPL 5 anos', 'Veredito'],
             [0.4, 0.5, 1.5, 1.3, 1.5, 1.0, 0.8, 1.0, 1.7],
             [[f"{r['RANKING']}", str(r[idc]), r['cidade'], f"{r['PREVISAO_FATURAMENTO']:,.1f}",
               f"{r['IC_INF']:,.0f} a {r['IC_SUP']:,.0f}", f"{r['PROB_SUPERAR_HURDLE']*100:,.0f}%",
               'n/a' if not np.isfinite(r['PAYBACK_MESES']) else f"{r['PAYBACK_MESES']:,.0f}m",
               f"R$ {r['VPL_R$']/1e6:,.2f} mi", r['RECOMENDACAO']] for _, r in R.iterrows()],
             alt_linha=0.045, tamanho=9.5,
             alinhamentos=['center', 'center', 'left', 'right', 'center', 'center', 'center',
                           'right', 'center'],
             cores_col=cores_vered)
    A.caixa(f, 0.035, 0.20, 0.93, 0.11, 'O que pedimos ao comitê',
            f"Aprovar {len(aprov)} unidade(s) — " +
            (', '.join(f"{r[idc]} ({r['cidade']})" for _, r in aprov.iterrows()) or 'nenhuma') +
            f" — com CAPEX total de R$ {aprov['CAPEX_R$'].sum()/1e6:,.2f} mi [#REF]; manter "
            f"{len(ress)} em observação com plano de mitigação e revisão em 90 dias; e reprovar "
            f"{len(reprov)} praça(s) neste ciclo. O hurdle de R$ {hurdle:,.0f} mil/mês e o nível de "
            f"confiança de {P_DECK['nivel_confianca']*100:,.0f}% são parâmetros do comitê e podem ser "
            f"reprocessados a qualquer momento.", largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ---------------------------------------------------- separador seção 01 ----------
    def separador(numero, titulo, subtitulo):
        f = A.novo_slide(secao=True)
        f.patches.append(mpatches.Rectangle((0.08, 0.62), 0.07, 0.007, transform=f.transFigure,
                                            color=C['verde_neon']))
        A.texto(f, 0.08, 0.58, numero, tamanho=15, cor=C['verde_neon'], negrito=True)
        A.texto(f, 0.08, 0.52, titulo, tamanho=34, cor='white', negrito=True, largura=30)
        A.texto(f, 0.08, 0.33, subtitulo, tamanho=13, cor=C['verde_claro'], largura=70)
        A.fechar_slide(f)

    separador('01', 'Diagnóstico da base', 'O que as lojas em operação já revelam sobre o que faz uma '
                                           'unidade performar')

    # ----------------------------------------------- S: base e limpeza ----------------
    f = A.novo_slide('O tratamento descarta 21% das observações, mas remove o ruído que distorce a estimativa',
                     'Base de dados')
    n_bruto, n_trat = (len(bruto) if bruto is not None else 0), len(tratado)
    etapas = ['Base bruta', 'Sem duplicatas\ne outliers (IQR)', 'Sem registros\nincompletos',
              'Base modelável\n(com dummies)']
    valores = [n_bruto, n_bruto - (n_bruto - n_trat) * 0.62, n_trat, n_trat]
    ax = A.eixo(f, 0.068, 0.775, 0.39, 0.40)
    barras = ax.bar(range(4), valores, color=[C['verde_claro'], C['verde_medio'], C['verde'],
                                              C['verde_escuro']], width=0.6, zorder=3)
    for b, v in zip(barras, valores):
        ax.text(b.get_x() + b.get_width() / 2, v + 3, f'{v:,.0f}', ha='center', fontsize=10,
                fontweight='bold', color=COR_TITULO)
    ax.set_xticks(range(4)); ax.set_xticklabels(etapas, fontsize=8.5)
    ax.set_ylabel('Unidades na base', fontsize=9)
    ax.set_title('Funil de tratamento dos dados', fontsize=11, color=COR_TITULO, fontweight='bold')
    ax = A.eixo(f, 0.578, 0.775, 0.385, 0.40)
    alvo = P_DECK['target']
    ax.hist(tratado[alvo], bins=24, color=C['verde_palido'], edgecolor=C['verde_claro'], zorder=3)
    ax.axvline(tratado[alvo].mean(), color=C['verde_escuro'], lw=1.8, zorder=4)
    ax.axvline(hurdle, color=C['terracota'], ls='--', lw=1.8, zorder=4)
    ax.text(tratado[alvo].mean(), ax.get_ylim()[1] * 0.92, f"  média {tratado[alvo].mean():,.0f}",
            fontsize=8.5, color=C['verde_escuro'], fontweight='bold')
    ax.text(hurdle, ax.get_ylim()[1] * 0.78, f"  hurdle {hurdle:,.0f}", fontsize=8.5,
            color=C['terracota'], fontweight='bold', ha='right')
    ax.set_xlabel('Faturamento médio mensal (R$ mil)', fontsize=9)
    ax.set_ylabel('Nº de lojas', fontsize=9)
    ax.set_title('Distribuição do faturamento na rede', fontsize=11, color=COR_TITULO, fontweight='bold')
    pct_acima = 100 * (tratado[alvo] >= hurdle).mean()
    A.caixa(f, 0.035, 0.30, 0.93, 0.155, 'Leitura',
            f"A base bruta tem {n_bruto} unidades; o corte de outliers por IQR e a exclusão de registros "
            f"incompletos deixam {n_trat} observações modeláveis ({100*n_trat/max(n_bruto,1):,.0f}% do "
            f"original). O faturamento médio da rede é de R$ {tratado[alvo].mean():,.1f} mil/mês, e "
            f"{pct_acima:,.0f}% das lojas em operação já superam o hurdle de R$ {hurdle:,.0f} mil — ou seja, "
            f"o critério do comitê é exigente, mas não irrealista. A limpeza remove justamente as unidades "
            f"excepcionais, o que torna as previsões conservadoras para pontos fora do padrão.",
            largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: EDA correlações ---------------
    corr = pacote['correlacoes']
    corr_pre = corr[[c for c in corr.index if c not in VARIAVEIS_VETADAS]].head(10)
    corr_pos = corr[[c for c in corr.index if c in VARIAVEIS_VETADAS]]
    f = A.novo_slide('As variáveis mais correlacionadas com faturamento são exatamente as indisponíveis na decisão',
                     'Exploração dos dados')
    ax = A.eixo(f, 0.175, 0.775, 0.30, 0.44)
    d_plot = pd.concat([corr_pos, corr_pre]).sort_values()
    cores_b = [C['terracota'] if i in VARIAVEIS_VETADAS else C['verde'] for i in d_plot.index]
    ax.barh([ROTULOS_CURTOS.get(i, i)[:26] for i in d_plot.index], d_plot.values, color=cores_b,
            height=0.66, zorder=3)
    ax.axvline(0, color=C['cinza_texto'], lw=1)
    ax.set_xlabel('Correlação com o faturamento médio mensal', fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(handles=[mpatches.Patch(color=C['terracota'], label='Só existe após a abertura'),
                       mpatches.Patch(color=C['verde'], label='Disponível na decisão')],
              fontsize=8, loc='lower right')
    prof = pacote['profiler']
    tab = prof['tabela'].head(6)
    A.texto(f, 0.545, 0.79, 'Perfil comparado: alta performance x demais lojas', tamanho=11,
            cor=COR_TITULO, negrito=True)
    A.tabela(f, 0.545, 0.752, 0.42,
             ['Variável', 'Alta perf.', 'Padrão', 'Δ%', 'p-valor'],
             [2.0, 1.0, 1.0, 0.8, 1.0],
             [[r['Rotulo'][:22], f"{r['Media_alta']:,.0f}", f"{r['Media_padrao']:,.0f}",
               f"{r['Diferenca_%']:+,.0f}%", f"{r['P_valor']:.3f}"] for _, r in tab.iterrows()],
             alt_linha=0.048, tamanho=9,
             alinhamentos=['left', 'right', 'right', 'right', 'right'])
    A.texto(f, 0.545, 0.42, f"Grupos definidos pelo hurdle de R$ {hurdle:,.0f} mil/mês: "
            f"{prof['n_alta']} lojas de alta performance x {prof['n_padrao']} demais unidades. "
            f"Teste t de Welch, sem hipótese de variâncias iguais.", tamanho=8.6, italico=True,
            largura=72)
    A.caixa(f, 0.035, 0.30, 0.93, 0.15, 'Implicação para a modelagem',
            'O ranking de correlação é dominado por ticket médio, número de transações e tempo de operação. '
            'Nenhuma delas está disponível quando o comitê decide onde abrir — e as duas primeiras são, por '
            'definição contábil, componentes do próprio faturamento. Entre as variáveis realmente '
            'disponíveis, população no raio de 1 km, área da loja e localização em shopping são as que mais '
            'separam lojas boas de ruins.', largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: dicionário --------------------
    f = A.novo_slide('Dicionário de variáveis: o que cada dado significa e quando ele existe',
                     'Dicionário de dados')
    dv = pacote['app']['dicionario_variaveis']
    A.tabela(f, 0.035, 0.80, 0.93,
             ['Variável', 'O que mede', 'Unidade', 'Família', 'Disponível antes da abertura?', 'Status'],
             [2.0, 4.0, 1.0, 1.3, 1.4, 0.9],
             [[r['Variavel'], r['Descricao'][:70], r['Unidade'], r['Familia'],
               r['Disponivel_pre_abertura'], r['Status_no_modelo']] for _, r in dv.iterrows()],
             alt_linha=0.0385, tamanho=8.4,
             alinhamentos=['left', 'left', 'center', 'left', 'center', 'center'],
             cores_col={5: [C['terracota'] if s == 'VETADA' else (C['verde_escuro'] if s == 'ALVO'
                            else C['verde']) for s in dv['Status_no_modelo']]})
    A.fechar_slide(f)

    # ----------------------------------------------- S: veto --------------------------
    f = A.novo_slide('Quatro variáveis foram vetadas por vazamento: elas só existem depois da abertura',
                     'Veto de variáveis')
    vetadas = dv[dv['Status_no_modelo'] == 'VETADA']
    y = 0.79
    for _, r in vetadas.iterrows():
        f.patches.append(mpatches.Rectangle((0.035, y - 0.115), 0.93, 0.115,
                                            transform=f.transFigure, color=C['verde_palido']))
        f.patches.append(mpatches.Rectangle((0.035, y - 0.115), 0.005, 0.115,
                                            transform=f.transFigure, color=C['terracota']))
        f.text(0.055, y - 0.032, r['Variavel'], fontsize=12, color=COR_TITULO, fontweight='bold',
               va='center')
        f.text(0.055, y - 0.062, r['Descricao'], fontsize=9.5, color=COR_TEXTO, va='center')
        f.text(0.055, y - 0.092, f"Motivo do veto: {r['Justificativa']}", fontsize=9.5,
               color=C['terracota'], va='center', fontstyle='italic')
        y -= 0.128
    A.caixa(f, 0.035, y + 0.005, 0.93, 0.13, 'Custo e benefício da decisão',
            'Manter essas variáveis elevaria o R² para perto de 1,0 — o modelo simplesmente reconstruiria '
            'a identidade contábil faturamento = ticket médio x número de transações. O modelo final abre '
            'mão desse ajuste artificial em troca de algo que o comitê pode de fato usar: uma estimativa '
            'para um ponto que ainda não abriu, feita apenas com informação disponível na data da decisão.',
            largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ---------------------------------------------------- separador seção 02 ----------
    separador('02', 'Modelagem', 'Quatro famílias testadas, um critério de aceitação e o modelo que '
                                 'sobrevive ao teste de utilidade')

    # ----------------------------------------------- S: benchmark ---------------------
    f = A.novo_slide('O modelo com maior ajuste não é o melhor: é o que usa informação que ainda não existe',
                     'Comparação de modelos')
    A.tabela(f, 0.035, 0.79, 0.93,
             ['Modelo testado', 'Variáveis', 'R² treino', 'R² teste', 'RMSE teste',
              'Pós-abertura?', 'Leitura'],
             [2.9, 0.8, 0.8, 0.8, 0.9, 1.0, 3.4],
             [[r['Modelo'], f"{r['N_variaveis']}", f"{r['R2_treino']:.3f}", f"{r['R2_teste']:.3f}",
               f"{r['RMSE_teste']:,.1f}", r['Usa_variaveis_pos_abertura'], r['Observacao'][:58]]
              for _, r in benchmark.iterrows()],
             alt_linha=0.050, tamanho=8.8,
             alinhamentos=['left', 'center', 'right', 'right', 'right', 'center', 'left'],
             cores_col={5: [C['terracota'] if v == 'Sim' else C['verde']
                            for v in benchmark['Usa_variaveis_pos_abertura']]})
    ax = A.eixo(f, 0.055, 0.415, 0.40, 0.185)
    cores_bm = [C['terracota'] if v == 'Sim' else C['verde']
                for v in benchmark['Usa_variaveis_pos_abertura']]
    ax.bar(range(len(benchmark)), benchmark['RMSE_teste'], color=cores_bm, width=0.62, zorder=3)
    for i, v in enumerate(benchmark['RMSE_teste']):
        ax.text(i, v + 0.4, f'{v:,.1f}', ha='center', fontsize=8.6, fontweight='bold', color=COR_TITULO)
    ax.set_xticks(range(len(benchmark)))
    ax.set_xticklabels([f"({i+1})" for i in range(len(benchmark))], fontsize=9)
    ax.set_ylabel('RMSE no teste (R$ mil)', fontsize=8.5)
    ax.set_title('Erro fora da amostra por modelo', fontsize=10, color=COR_TITULO, fontweight='bold',
                 pad=6)
    ax = A.eixo(f, 0.565, 0.415, 0.40, 0.185)
    ax.bar(range(len(benchmark)), benchmark['R2_teste'], color=cores_bm, width=0.62, zorder=3)
    for i, v in enumerate(benchmark['R2_teste']):
        ax.text(i, v + 0.015, f'{v:.2f}', ha='center', fontsize=8.6, fontweight='bold', color=COR_TITULO)
    ax.set_xticks(range(len(benchmark)))
    ax.set_xticklabels([f"({i+1})" for i in range(len(benchmark))], fontsize=9)
    ax.set_ylim(0, 1.08)
    ax.set_ylabel('R² no teste', fontsize=8.5)
    ax.set_title('Poder explicativo fora da amostra', fontsize=10, color=COR_TITULO, fontweight='bold',
                 pad=6)
    A.caixa(f, 0.035, 0.185, 0.93, 0.11, 'Critério de escolha',
            'Os modelos (1), (2) e (4) parecem superiores porque enxergam ticket médio e número de '
            'transações — informação que a rede só terá meses depois de assinar o contrato de locação. '
            'O modelo (5) aceita um erro maior em troca de ser o único aplicável no momento da decisão; '
            'é sobre ele que as recomendações deste documento são construídas.',
            largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: model card --------------------
    f = A.novo_slide(f"O modelo campeão explica {m['r2_teste']*100:,.0f}% do faturamento com "
                     f"{m['n_variaveis_modelo']} variáveis conhecidas antes da abertura", 'Ficha técnica')
    kpis = [('R² fora da amostra', f"{m['r2_teste']*100:,.1f}%", 'dados nunca vistos no treino'),
            ('RMSE no teste', f"R$ {m['rmse_teste']:,.1f} mil", f"{m['erro_relativo_%']:,.1f}% da média"),
            ('MAPE no teste', f"{m['mape_teste']:,.1f}%", 'erro percentual médio'),
            ('Observações', f"{m['n_treino']}/{m['n_teste']}", 'treino / teste (split 70-30)')]
    for i, (rot, val, sub) in enumerate(kpis):
        A.kpi(f, 0.035 + i * 0.233, 0.80, 0.215, 0.135, rot, val, sub)
    tc = ctx['tabela_coef']
    A.tabela(f, 0.035, 0.62, 0.55,
             ['Variável do modelo', 'Impacto por +1 dp', 'Por unidade', 'p-valor', 'VIF', 'Peso'],
             [2.4, 1.2, 1.1, 0.9, 0.6, 0.7],
             [[r['Rotulo'][:26], f"{r['Coef_padronizado']:+,.2f}", f"{r['Coef_unidade_original']:+,.4f}",
               f"{r['P_valor']:.4f}", f"{r['VIF']:,.1f}", f"{r['Peso_relativo_%']:,.0f}%"]
              for _, r in tc.iterrows()],
             alt_linha=0.038, tamanho=8.4,
             alinhamentos=['left', 'right', 'right', 'right', 'right', 'right'])
    A.texto(f, 0.62, 0.615, 'Diagnóstico dos resíduos', tamanho=11, cor=COR_TITULO, negrito=True)
    diag_linhas = [
        ('R² ajustado (OLS gêmeo)', f"{m['r2_ajustado_ols']:.3f}"),
        ('Teste F (p-valor)', f"{m['f_pvalue_ols']:.1e}"),
        ('Durbin-Watson', f"{m['durbin_watson']:.2f}"),
        ('Jarque-Bera (p)', f"{m['jarque_bera_p']:.3f}"),
        ('Breusch-Pagan (p)', f"{m['breusch_pagan_p']:.3f}"),
        ('VIF máximo', f"{m['vif_maximo']:,.2f}"),
        ('RMSE treino / teste', f"{m['rmse_treino']:,.1f} / {m['rmse_teste']:,.1f}"),
    ]
    A.tabela(f, 0.62, 0.59, 0.345, ['Teste', 'Valor'], [2.2, 1.0],
             [[a, b] for a, b in diag_linhas], alt_linha=0.038, tamanho=8.8,
             alinhamentos=['left', 'right'])
    A.caixa(f, 0.62, 0.29, 0.345, 0.135, 'Como ler',
            'Resíduos sem autocorrelação (DW próximo de 2), variância estável e VIF baixo indicam '
            'coeficientes estáveis e intervalos confiáveis.', largura_txt=54, tamanho=9.5)
    A.caixa(f, 0.035, 0.185, 0.55, 0.10, 'Equação estimada',
            f"faturamento = {ctx['intercepto_original']:,.1f} " +
            ' '.join(f"{c:+,.3f}·{v}" for v, c in zip(tc['Variavel'].head(4),
                                                      tc['Coef_unidade_original'].head(4))) +
            ' ... (demais termos no apêndice)', largura_txt=90, tamanho=8.6)
    A.fechar_slide(f)

    # ----------------------------------------------- S: drivers -----------------------
    f = A.novo_slide('Localização e porte explicam a maior parte do resultado; marketing é a única '
                     'alavanca sob controle da gestão', 'Direcionadores')
    A.imagem(f, figs.get('E2_drivers'), 0.035, 0.80, 0.60, 0.62)
    A.texto(f, 0.665, 0.79, 'O que isso significa', tamanho=12, cor=COR_TITULO, negrito=True)
    top = tc.head(3)
    itens = [f"{r['Rotulo']}: cada +1 desvio-padrão move o faturamento em "
             f"R$ {r['Coef_padronizado']:+,.1f} mil/mês ({r['Peso_relativo_%']:,.0f}% do poder explicativo)."
             for _, r in top.iterrows()]
    alav = tc[tc['Familia'] == 'Alavanca gerencial']
    if len(alav):
        itens.append('Marketing mensal é a única variável de decisão contínua que sobreviveu ao filtro '
                     f"estatístico: efeito de {alav.iloc[0]['Coef_unidade_original']:+,.4f} mil por R$ 1 "
                     'de verba adicional.')
    itens.append('As demais variáveis são características do imóvel: a decisão relevante é qual ponto '
                 'assinar, não como operá-lo depois.')
    A.marcadores(f, 0.665, 0.745, itens, tamanho=10, largura=44, passo=0.055)
    A.fechar_slide(f)

    # ----------------------------------------------- S: ajuste ------------------------
    f = A.novo_slide(f"O erro fora da amostra (R$ {m['rmse_teste']:,.1f} mil) fica próximo do erro de "
                     f"treino: não há sobreajuste relevante", 'Validação')
    A.imagem(f, figs.get('E4_ajuste'), 0.035, 0.80, 0.62, 0.58)
    A.texto(f, 0.685, 0.79, 'Leitura da validação', tamanho=12, cor=COR_TITULO, negrito=True)
    A.marcadores(f, 0.685, 0.745, [
        f"Split 70/30 com semente fixa: {m['n_treino']} lojas para estimar, {m['n_teste']} para testar.",
        f"RMSE de treino R$ {m['rmse_treino']:,.1f} mil x RMSE de teste R$ {m['rmse_teste']:,.1f} mil — "
        f"a diferença é pequena, sinal de que o modelo não decorou a amostra.",
        f"O erro equivale a {m['erro_relativo_%']:,.1f}% do faturamento médio da rede; é esse número que "
        f"alimenta os intervalos de confiança de cada recomendação.",
        'Os resíduos se distribuem em torno de zero sem padrão visível, o que sustenta o uso dos '
        'intervalos como medida de risco.',
    ], tamanho=10, largura=40, passo=0.058)
    A.fechar_slide(f)

    # ---------------------------------------------------- separador seção 03 ----------
    separador('03', 'Aplicação e recomendação', 'Da previsão pontual à decisão de capital, praça a praça')

    # ----------------------------------------------- S: matriz de decisão -------------
    n_ok = int((R['PREVISAO_FATURAMENTO'] >= hurdle).sum())
    f = A.novo_slide(f'{n_ok} praças superam o hurdle na previsão pontual, mas apenas {len(aprov)} '
                     f'passam também no teste de confiança', 'Matriz de decisão')
    A.imagem(f, figs.get('E1_ranking'), 0.035, 0.80, 0.60, 0.60)
    A.texto(f, 0.665, 0.79, 'Regra de decisão em vigor', tamanho=12, cor=COR_TITULO, negrito=True)
    A.marcadores(f, 0.665, 0.745, [
        f"Hurdle: faturamento projetado de ao menos R$ {hurdle:,.0f} mil/mês.",
        f"Confiança: probabilidade de superar o hurdle de no mínimo "
        f"{P_DECK['prob_minima_aprovacao']*100:,.0f}%.",
        f"Zona cinzenta de ±{P_DECK['zona_cinzenta_pct']*100:,.0f}% em torno do hurdle classifica a praça "
        f"como 'abrir com ressalvas'.",
        f"Payback máximo de {P_DECK['payback_maximo_meses']} meses [#REF] como segunda barreira.",
        'Praça fora do domínio de dados do treino é rebaixada um nível automaticamente.',
    ], tamanho=9.8, largura=42, passo=0.052)
    A.caixa(f, 0.035, 0.165, 0.93, 0.115, 'Por que a previsão pontual não basta',
            'Com erro típico de R$ ' + f"{m['rmse_teste']:,.1f}" + ' mil/mês, uma praça projetada a poucos '
            'reais acima do hurdle tem, na prática, quase 50% de chance de ficar abaixo dele. A coluna de '
            'probabilidade converte a previsão e o erro do modelo em uma medida direta de risco de decisão, '
            'e é ela que separa uma aprovação de uma ressalva.', largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: quadrante ---------------------
    f = A.novo_slide('Retorno e confiança andam juntos: as praças aprovadas ocupam o quadrante superior direito',
                     'Risco x retorno')
    A.imagem(f, figs.get('E6_risco_retorno'), 0.035, 0.80, 0.60, 0.62)
    A.texto(f, 0.665, 0.79, 'Como ler o quadrante', tamanho=12, cor=COR_TITULO, negrito=True)
    y_lista = A.marcadores(f, 0.665, 0.745, [
        'Eixo horizontal: probabilidade de a unidade superar o hurdle, dado o erro do modelo.',
        'Eixo vertical: faturamento médio mensal projetado.',
        'Tamanho da bolha: VPL estimado em 5 anos sob as premissas do comitê [#REF].',
        'Só o quadrante superior direito reúne retorno e confiança suficientes para aprovação direta.',
    ], tamanho=10, largura=42, passo=0.052)
    A.tabela(f, 0.665, y_lista - 0.02, 0.30, ['Praça', 'P(> hurdle)', 'Veredito'], [0.7, 1.1, 1.6],
             [[str(r[idc]), f"{r['PROB_SUPERAR_HURDLE']*100:,.0f}%", r['RECOMENDACAO']]
              for _, r in R.iterrows()],
             alt_linha=0.042, tamanho=8.8, alinhamentos=['center', 'right', 'center'],
             cores_col={2: [CORES_VEREDITO[v] for v in R['RECOMENDACAO']]})
    A.fechar_slide(f)

    # ----------------------------------------------- S: anatomia ----------------------
    f = A.novo_slide('A diferença entre as praças vem de porte, formato e densidade populacional — '
                     'não de esforço operacional', 'Anatomia da previsão')
    A.imagem(f, figs.get('E10_contribuicoes'), 0.035, 0.80, 0.64, 0.62)
    A.texto(f, 0.71, 0.79, 'Leitura', tamanho=12, cor=COR_TITULO, negrito=True)
    top_cand = R.iloc[0]
    contrib = pacote['app']['contribuicoes']
    c_top = contrib.loc[top_cand[idc]].drop('BASE_MEDIA_DA_REDE')
    c_top = c_top[c_top.abs() > 1e-6].sort_values(key=np.abs, ascending=False)
    A.marcadores(f, 0.71, 0.745, [
        f"Cada barra parte da loja média da rede (R$ {contrib.iloc[0]['BASE_MEDIA_DA_REDE']:,.0f} mil/mês) "
        f"e soma o efeito de cada característica do ponto.",
        f"Na praça prioritária ({top_cand[idc]}), os três maiores contribuintes são "
        + ', '.join(f"{_dicionario_de(i)['Rotulo']} ({v:+,.1f})" for i, v in c_top.head(3).items()) + '.',
        'Praças com barras predominantemente negativas não têm problema de gestão: têm um ponto com '
        'fundamentos de demanda mais fracos.',
    ], tamanho=10, largura=36, passo=0.062)
    A.fechar_slide(f)

    # ----------------------------------------------- S: one-pagers --------------------
    for _, r in R.iterrows():
        cor = CORES_VEREDITO[r['RECOMENDACAO']]
        f = A.novo_slide(f"{r['cidade']} ({r['regiao']}) — {r['RECOMENDACAO'].lower()}",
                         f"Praça {r[idc]}  ·  prioridade {r['RANKING']} de {len(R)}")
        f.patches.append(mpatches.Rectangle((0.035, 0.80), 0.93, 0.045, transform=f.transFigure,
                                            color=cor))
        f.text(0.05, 0.8225, f"VEREDITO: {r['RECOMENDACAO']}", fontsize=12.5, color='white',
               fontweight='bold', va='center')
        f.text(0.95, 0.8225, str(r.get('endereco_referencia', ''))[:60], fontsize=9.5, color='white',
               va='center', ha='right')
        kpis = [('Faturamento projetado', f"R$ {r['PREVISAO_FATURAMENTO']:,.0f} mil",
                 f"hurdle: R$ {hurdle:,.0f} mil"),
                (f"Intervalo de {P_DECK['nivel_confianca']*100:,.0f}%",
                 f"{r['IC_INF']:,.0f}–{r['IC_SUP']:,.0f}", 'R$ mil/mês'),
                ('Prob. de superar', f"{r['PROB_SUPERAR_HURDLE']*100:,.0f}%", 'o hurdle do comitê'),
                ('VPL em 5 anos', f"R$ {r['VPL_R$']/1e6:,.2f} mi",
                 'n/a' if not np.isfinite(r['PAYBACK_MESES']) else f"payback {r['PAYBACK_MESES']:,.0f} meses")]
        for i, (rot, val, sub) in enumerate(kpis):
            A.kpi(f, 0.035 + i * 0.233, 0.775, 0.215, 0.125, rot, val, sub)
        campos = [('area_m2', 'Área (m²)'), ('populacao_1km', 'Pop. 1 km (mil)'),
                  ('renda_media_domiciliar', 'Renda (R$)'), ('fluxo_pedestres_dia', 'Fluxo/dia'),
                  ('vagas_estacionamento', 'Vagas'), ('distancia_concorrente_m', 'Concorrente (m)'),
                  ('shopping', 'Shopping'), ('investimento_marketing_mensal', 'Marketing (R$)'),
                  ('n_funcionarios', 'Equipe')]
        campos = [(c, rot) for c, rot in campos if c in R.columns]
        A.tabela(f, 0.035, 0.625, 0.93, [rot for _, rot in campos], [1] * len(campos),
                 [[f"{r[c]:,.0f}" for c, _ in campos]], alt_linha=0.042, tamanho=9)
        A.imagem(f, figs.get(f"E3_mini_{r[idc]}"), 0.035, 0.51, 0.60, 0.33)
        A.texto(f, 0.665, 0.525, 'Direcionadores', tamanho=11, cor=COR_TITULO, negrito=True)
        c_r = contrib.loc[r[idc]].drop('BASE_MEDIA_DA_REDE')
        c_r = c_r[c_r.abs() > 1e-6].sort_values(key=np.abs, ascending=False).head(5)
        A.tabela(f, 0.665, 0.485, 0.30, ['Fator', 'R$ mil/mês'], [2.2, 1.0],
                 [[_dicionario_de(i)['Rotulo'][:22], f"{v:+,.1f}"] for i, v in c_r.items()],
                 alt_linha=0.040, tamanho=8.6, alinhamentos=['left', 'right'],
                 cores_col={1: [C['verde'] if v > 0 else C['terracota'] for v in c_r.values]})
        acao = ('Prosseguir com a due diligence imobiliária e travar o cronograma de abertura.'
                if r['RECOMENDACAO'] == 'ABRIR' else
                'Não alocar capital neste ciclo; reavaliar apenas diante de mudança material no ponto.'
                if r['RECOMENDACAO'] == 'NÃO ABRIR' else
                'Aprovar somente com plano de mitigação, metas de marketing e revisão em 90 dias.')
        A.caixa(f, 0.035, 0.165, 0.93, 0.11, 'Decisão e próximo passo',
                f"{r['JUSTIFICATIVA']}. {acao}", largura_txt=145, tamanho=9.8,
                cor_barra=cor)
        A.fechar_slide(f)

    # ----------------------------------------------- S: sensibilidade -----------------
    sens = pacote['app']['sensibilidade']
    esforco = pacote['app']['esforco_hurdle']
    f = A.novo_slide('Depois de escolhido o ponto, sobra pouca alavanca: marketing move o resultado, '
                     'mas não salva uma praça fraca', 'Sensibilidade')
    A.imagem(f, figs.get('E5_tornado'), 0.035, 0.80, 0.58, 0.44)
    A.texto(f, 0.63, 0.80, 'Esforço para atingir o hurdle', tamanho=11.5, cor=COR_TITULO, negrito=True)
    esf = esforco.drop_duplicates(subset=['Candidata'], keep='first')
    A.tabela(f, 0.63, 0.758, 0.335, ['Praça', 'Gap', 'Variação necessária'], [0.7, 0.8, 2.2],
             [[str(r['Candidata']), f"{r['Gap_R$mil']:,.0f}", str(r['Variacao_necessaria'])[:26]]
              for _, r in esf.iterrows()],
             alt_linha=0.045, tamanho=8.8, alinhamentos=['center', 'right', 'left'])
    sem_efeito = sorted(set(P_DECK['alavancas']) - set(ctx['features']))
    A.caixa(f, 0.035, 0.31, 0.93, 0.14, 'O que a simulação mostra',
            'As alavancas gerenciais entram no modelo com efeito real, porém pequeno frente às '
            'características do ponto: mesmo variações de ±20% na verba de marketing movem o faturamento '
            'projetado em poucos R$ mil por mês. ' +
            (f"As variáveis {', '.join(ROTULOS_CURTOS.get(v, v) for v in sem_efeito)} não foram "
             f"selecionadas pelo modelo e, "
             f"estatisticamente, alterá-las não muda a previsão. " if sem_efeito else '') +
            'A consequência prática é direta: o valor é criado na escolha do ponto, não na operação.',
            largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: caso econômico ----------------
    f = A.novo_slide('Sob as premissas do comitê, as praças aprovadas devolvem o capital em menos de '
                     f"{P_DECK['payback_maximo_meses']} meses [#REF]", 'Caso econômico')
    A.imagem(f, figs.get('E7_economia'), 0.035, 0.815, 0.93, 0.35)
    A.tabela(f, 0.035, 0.44, 0.93,
             ['Praça', 'Receita/mês', 'EBITDA/mês', 'CAPEX', 'Payback', 'VPL 5 anos', 'TIR a.a.', 'Veredito'],
             [0.6, 1.1, 1.1, 1.1, 0.9, 1.1, 0.9, 1.7],
             [[str(r[idc]), f"R$ {r['RECEITA_MENSAL_R$']/1e3:,.0f} mil",
               f"R$ {r['EBITDA_MENSAL_R$']/1e3:,.1f} mil", f"R$ {r['CAPEX_R$']/1e3:,.0f} mil",
               'n/a' if not np.isfinite(r['PAYBACK_MESES']) else f"{r['PAYBACK_MESES']:,.0f} m",
               f"R$ {r['VPL_R$']/1e6:,.2f} mi",
               'n/a' if not np.isfinite(r['TIR_ANUAL_%']) else f"{r['TIR_ANUAL_%']:,.0f}%",
               r['RECOMENDACAO']] for _, r in R.iterrows()],
             alt_linha=0.042, tamanho=9,
             alinhamentos=['center', 'right', 'right', 'right', 'center', 'right', 'right', 'center'],
             cores_col={7: [CORES_VEREDITO[v] for v in R['RECOMENDACAO']]})
    A.caixa(f, 0.035, 0.175, 0.93, 0.10, 'Premissas por trás destes números [#REF]',
            f"Margem de contribuição de {P_DECK['margem_contribuicao_pct']*100:,.0f}%, custo fixo mensal de "
            f"R$ {P_DECK['custo_fixo_mensal_reais']:,.0f} por unidade, CAPEX de "
            f"R$ {P_DECK['capex_fixo_reais']:,.0f} mais R$ {P_DECK['capex_por_m2_reais']:,.0f} por m², custo "
            f"de capital de {P_DECK['taxa_desconto_anual']*100:,.0f}% a.a. e horizonte de "
            f"{P_DECK['horizonte_meses']} meses. Nenhum desses valores foi extraído da base: são premissas "
            f"de trabalho a validar com a controladoria.", largura_txt=150, tamanho=9.6)
    A.fechar_slide(f)

    # ---------------------------------------------------- separador seção 04 ----------
    separador('04', 'Riscos e premissas', 'O que pode invalidar a recomendação e como o comitê deve '
                                          'ler os números')

    # ----------------------------------------------- S: aderência e riscos ------------
    f = A.novo_slide('Uma candidata opera fora do domínio de dados do modelo e teve a recomendação '
                     'rebaixada por regra' if R['FORA_DO_DOMINIO'].any() else
                     'Todas as candidatas estão dentro do domínio de dados em que o modelo foi treinado',
                     'Aderência e riscos')
    A.imagem(f, figs.get('E8_extrapolacao'), 0.035, 0.80, 0.55, 0.50)
    A.texto(f, 0.615, 0.79, 'Principais riscos', tamanho=12, cor=COR_TITULO, negrito=True)
    fora = R[R['FORA_DO_DOMINIO']]
    def _amigavel(lista_vars):
        return ', '.join(ROTULOS_CURTOS.get(v.strip(), v.strip()) for v in str(lista_vars).split(','))
    riscos = []
    if len(fora):
        riscos.append('Extrapolação: ' + ', '.join(
            f"{r[idc]} ({_amigavel(r['DETALHE_EXTRAPOLACAO'])})" for _, r in fora.iterrows()) +
            ' — perfil atípico frente à base histórica; a previsão é menos confiável e a recomendação '
            'foi rebaixada automaticamente.')
    riscos += [
        'Corte de outliers: a limpeza por IQR removeu operações excepcionais, então o modelo tende a '
        'subestimar pontos fora do padrão da rede.',
        'Efeito de praça: dummies de cidade e região carregam parte relevante do poder explicativo e são '
        'estimadas com poucas lojas por praça.',
        'Causalidade: os coeficientes medem associação entre lojas, não efeito causal de mudar uma '
        'variável em uma loja específica.',
        'Premissas econômicas: payback, VPL e TIR dependem inteiramente de hipóteses não extraídas da '
        'base [#REF] e precisam ser validadas com a controladoria.',
    ]
    imput = pacote['app']['imputacoes']
    if len(imput):
        riscos.append('Qualidade de dado: ' + '; '.join(
            f"{r['Candidata']} teve {ROTULOS_CURTOS.get(r['Variavel'], r['Variavel'])} imputada pela "
            f"mediana da base ({r['Valor_imputado']:,.0f})" for _, r in imput.iterrows()) + '.')
    A.marcadores(f, 0.615, 0.745, riscos, tamanho=9.5, largura=48, passo=0.048,
                 cor_marca=C['terracota'])
    A.fechar_slide(f)

    # ----------------------------------------------- S: premissas ---------------------
    f = A.novo_slide('Todos os parâmetros de decisão são explícitos e reprocessáveis', 'Premissas')
    A.texto(f, 0.035, 0.79, 'Parâmetros de decisão (definidos pelo comitê)', tamanho=11.5,
            cor=COR_TITULO, negrito=True)
    A.tabela(f, 0.035, 0.765, 0.44, ['Parâmetro', 'Valor', 'Natureza'], [1.8, 1.1, 1.4],
             [['Hurdle de faturamento', f"R$ {hurdle:,.0f} mil/mês", 'Comitê'],
              ['Regra de decisão', P_DECK['regra_decisao'], 'Comitê'],
              ['Probabilidade mínima', f"{P_DECK['prob_minima_aprovacao']*100:,.0f}%", 'Comitê'],
              ['Zona cinzenta', f"±{P_DECK['zona_cinzenta_pct']*100:,.0f}%", 'Comitê'],
              ['Nível de confiança', f"{P_DECK['nivel_confianca']*100:,.0f}%", 'Estatística'],
              ['Método do intervalo', P_DECK['metodo_intervalo'], 'Estatística'],
              ['Rebaixar extrapolação', str(P_DECK['rebaixar_por_extrapolacao']), 'Modelagem'],
              ['Limite de |z|', f"{P_DECK['limite_z_extrapolacao']}", 'Modelagem']],
             alt_linha=0.042, tamanho=8.8, alinhamentos=['left', 'right', 'center'])
    A.texto(f, 0.525, 0.79, 'Premissas econômicas — não derivadas dos dados [#REF]', tamanho=11.5,
            cor=COR_TITULO, negrito=True)
    A.tabela(f, 0.525, 0.765, 0.44, ['Premissa', 'Valor', 'Natureza'], [1.8, 1.1, 1.4],
             [['Margem de contribuição', f"{P_DECK['margem_contribuicao_pct']*100:,.0f}%", 'Premissa [#REF]'],
              ['Custo fixo mensal', f"R$ {P_DECK['custo_fixo_mensal_reais']:,.0f}", 'Premissa [#REF]'],
              ['CAPEX fixo', f"R$ {P_DECK['capex_fixo_reais']:,.0f}", 'Premissa [#REF]'],
              ['CAPEX por m²', f"R$ {P_DECK['capex_por_m2_reais']:,.0f}", 'Premissa [#REF]'],
              ['Custo de capital', f"{P_DECK['taxa_desconto_anual']*100:,.0f}% a.a.", 'Premissa [#REF]'],
              ['Horizonte de avaliação', f"{P_DECK['horizonte_meses']} meses", 'Convenção'],
              ['Payback máximo', f"{P_DECK['payback_maximo_meses']} meses", 'Comitê'],
              ['Pesos do score', 'retorno 45 / conf. 25 / fin. 20 / ader. 10', 'Comitê']],
             alt_linha=0.042, tamanho=8.8, alinhamentos=['left', 'right', 'center'])
    A.caixa(f, 0.035, 0.395, 0.93, 0.13, 'Como reprocessar o cenário',
            'Todos os valores acima vivem no dicionário PARAMETROS, no início da célula de aplicação. '
            'Alterar o hurdle, a regra de decisão, a margem ou o custo de capital e reexecutar as duas '
            'últimas células regenera previsões, gráficos, o relatório executivo e este deck — sem '
            'nenhuma edição manual de número. É isso que permite ao comitê testar cenários alternativos '
            'em minutos e comparar vereditos sob hipóteses diferentes.', largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: próximos passos ---------------
    separador('05', 'Próximos passos', 'O que pedimos ao comitê e em que sequência')
    f = A.novo_slide('Três ondas de trabalho até a abertura das unidades aprovadas', 'Plano de ação')
    ondas = [
        ('Onda 1 — 0 a 30 dias', C['verde_escuro'], [
            'Validar as premissas econômicas com a controladoria (margem, CAPEX, custo fixo) [#REF]',
            'Aprovar formalmente as praças recomendadas e reservar o CAPEX correspondente',
            'Coletar o dado faltante das candidatas com imputação (renda do entorno)']),
        ('Onda 2 — 30 a 90 dias', C['verde'], [
            'Due diligence imobiliária e negociação dos contratos das praças aprovadas',
            'Plano de mitigação e metas de marketing para as praças com ressalva',
            'Reavaliar as praças em observação com dados atualizados do ponto']),
        ('Onda 3 — 90 dias em diante', C['verde_medio'], [
            'Abertura das unidades e acompanhamento mensal de faturamento realizado x projetado',
            'Realimentar a base com as novas lojas e reestimar o modelo a cada ciclo',
            'Medir o erro de previsão em produção e recalibrar o hurdle se necessário']),
    ]
    x = 0.035
    for titulo_onda, cor, itens in ondas:
        f.patches.append(mpatches.Rectangle((x, 0.30), 0.30, 0.50, transform=f.transFigure,
                                            color=C['verde_palido'], zorder=1))
        f.patches.append(mpatches.Rectangle((x, 0.755), 0.30, 0.045, transform=f.transFigure,
                                            color=cor, zorder=2))
        f.text(x + 0.015, 0.7775, titulo_onda, fontsize=11, color='white', fontweight='bold',
               va='center', zorder=3)
        yy = 0.72
        for it in itens:
            f.patches.append(mpatches.Rectangle((x + 0.015, yy - 0.006), 0.006, 0.010,
                                                transform=f.transFigure, color=cor, zorder=3))
            f.text(x + 0.028, yy, textwrap.fill(it, 36), fontsize=9.3, color=COR_TEXTO, va='top',
                   zorder=3, linespacing=1.45)
            yy -= 0.115
        x += 0.325
    A.caixa(f, 0.035, 0.255, 0.93, 0.105, 'Decisão solicitada nesta reunião',
            f"Aprovação do CAPEX de R$ {aprov['CAPEX_R$'].sum()/1e6:,.2f} mi [#REF] para as praças "
            + (', '.join(str(r[idc]) for _, r in aprov.iterrows()) or '—') +
            f", com VPL projetado de R$ {aprov['VPL_R$'].sum()/1e6:,.2f} mi em cinco anos, e mandato para "
            f"reavaliar as praças em observação em 90 dias.", largura_txt=150, tamanho=10)
    A.fechar_slide(f)

    # ----------------------------------------------- S: apêndice ----------------------
    f = A.novo_slide('Apêndice: especificação completa do modelo e parâmetros de execução', 'Apêndice')
    A.texto(f, 0.035, 0.79, 'Equação estimada (unidades originais)', tamanho=11.5, cor=COR_TITULO,
            negrito=True)
    eq = (f"faturamento_medio_mensal = {ctx['intercepto_original']:,.2f} " +
          ' '.join(f"{c:+,.4f}·{v}" for v, c in zip(tc['Variavel'], tc['Coef_unidade_original'])))
    A.texto(f, 0.035, 0.755, eq, tamanho=8.6, largura=135)
    A.texto(f, 0.035, 0.63, 'Variáveis do modelo campeão', tamanho=11.5, cor=COR_TITULO, negrito=True)
    A.texto(f, 0.035, 0.60, ', '.join(ctx['features']), tamanho=9, largura=130)
    A.texto(f, 0.035, 0.50, 'Parâmetros de execução', tamanho=11.5, cor=COR_TITULO, negrito=True)
    chaves = ['hurdle_faturamento', 'regra_decisao', 'prob_minima_aprovacao', 'zona_cinzenta_pct',
              'nivel_confianca', 'metodo_intervalo', 'rebaixar_por_extrapolacao',
              'limite_z_extrapolacao', 'margem_contribuicao_pct', 'custo_fixo_mensal_reais',
              'capex_por_m2_reais', 'capex_fixo_reais', 'taxa_desconto_anual', 'horizonte_meses',
              'payback_maximo_meses', 'pesos_score']
    A.texto(f, 0.035, 0.47, '\n'.join(f"{k} = {P_DECK[k]}" for k in chaves), tamanho=8.4, largura=130)
    A.texto(f, 0.035, 0.115, 'Arquivos gerados nesta execução: ' +
            f"{P_DECK['arquivo_pdf']} (relatório executivo), {P_DECK['arquivo_csv']} (matriz de decisão), "
            f"{params['arquivo']} (este deck) e a pasta {P_DECK['pasta_figuras']}/ com os exhibits.",
            tamanho=9, italico=True, largura=140)
    A.fechar_slide(f)

    return A.finalizar()

# =====================================================================================
# 6. EXECUÇÃO
# =====================================================================================
def _titulo_console(txt, char='='):
    print('\n' + char * 100)
    print(txt.upper().center(100))
    print(char * 100)

_titulo_console('consolidação geral do trabalho', '#')
PACOTE = coletar_resultados()
print(f"Base bruta:      {PACOTE['df_bruto'].shape if PACOTE['df_bruto'] is not None else 'não localizada'}")
print(f"Base tratada:    {PACOTE['df_tratado'].shape}")
print(f"Praças candidatas: {len(PACOTE['R'])}")
print(f"Exhibits disponíveis: {len(PACOTE['figs'])}")

_titulo_console('1. perfil da rede: alta performance x performance padrão')
print(f"Corte aplicado: R$ {PACOTE['profiler']['corte']:,.0f} mil/mês  |  "
      f"{PACOTE['profiler']['n_alta']} lojas de alta performance x {PACOTE['profiler']['n_padrao']} demais")
display(PACOTE['profiler']['tabela'].round(3))

_titulo_console('2. correlação de cada variável com o faturamento')
corr_df = PACOTE['correlacoes'].to_frame('Correlacao_com_alvo')
corr_df['Rotulo'] = [ROTULOS_CURTOS.get(i, i) for i in corr_df.index]
corr_df['Disponivel_na_decisao'] = ['Não' if i in VARIAVEIS_VETADAS else 'Sim' for i in corr_df.index]
display(corr_df.head(15).round(3))

_titulo_console('3. estatísticas descritivas das variáveis de negócio')
display(PACOTE['descritivas'].round(2))

_titulo_console('4. benchmark: todos os modelos testados no trabalho')
BENCHMARK = benchmark_modelos(PACOTE['df_tratado'], PACOTE['ctx'], DECK)
display(BENCHMARK.round(4))
print('\nLeitura: modelos que usam variáveis pós-abertura têm ajuste superior e utilidade nula para '
      'decidir a abertura de um ponto novo. O modelo campeão é o único aplicável na data da decisão.')
if PACOTE['stepwise_features']:
    print(f"\nVariáveis selecionadas pelo stepwise executado na Parte 2: {PACOTE['stepwise_features']}")

_titulo_console('5. matriz de decisão consolidada')
display(PACOTE['R'][[P_DECK['coluna_id_candidata'], 'cidade', 'PREVISAO_FATURAMENTO', 'IC_INF', 'IC_SUP',
                     'PROB_SUPERAR_HURDLE', 'PAYBACK_MESES', 'VPL_R$', 'SCORE_PRIORIDADE',
                     'RECOMENDACAO']].round(2))

_titulo_console('6. montagem do deck de apresentação')
ARQUIVO_DECK = montar_deck(PACOTE, BENCHMARK, DECK)
print(f"Deck gerado: {ARQUIVO_DECK}")
if DECK['exportar_png']:
    print(f"Slides individuais em PNG: pasta {DECK['pasta_slides']}/")
print(f"Relatório executivo: {P_DECK['arquivo_pdf']}")
print(f"Matriz de decisão: {P_DECK['arquivo_csv']}")
print(f"Exhibits: pasta {P_DECK['pasta_figuras']}/")

_titulo_console('entregáveis desta execução', '#')
for arquivo in [ARQUIVO_DECK, P_DECK['arquivo_pdf'], P_DECK['arquivo_csv']]:
    if arquivo and os.path.exists(arquivo):
        print(f"  ✔ {arquivo}  ({os.path.getsize(arquivo)/1024:,.0f} KB)")
